In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.io import mmread
from scipy.sparse import csr_matrix, issparse
from typing import List
from typing import Tuple
import scvi
import torch
import scipy.sparse as sp
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from scvi.model import SCVI
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import xgboost as xgb
import shap
from scipy.stats import pearsonr
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.linear_model import ElasticNetCV
import gseapy as gp
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score
from tqdm import tqdm
import sys
import re

## Direct loading from `combined_annotated_final.h5ad`

Replaces the old pipeline (`mmread` + `rownames.txt` / `colnames.txt` / `barcodes_to_patient.tsv` / `blood_features.txt` / `celltype_annotations.tsv`).
We read the annotated `.h5ad` **once** and extract from it:
- the **raw counts** (layer `counts`, non-normalized integer counts) -> placed into `adata.X` and `adata.layers['counts']`;
- the **biomarkers / covariates** (`patient_id`, `sex`, `QC_MT.pct`, `batch_merging`, `CellType`, `urea26`, `bmi26`, ...) from `adata.obs`, plus a `blood` table ready for the biomarker step.

In [ ]:
# =====================================================================
# 1) READ THE ANNOTATED .h5ad
# =====================================================================
import scanpy as sc
import numpy as np
import pandas as pd
from scipy import sparse

H5AD_PATH = "/home/nlapique/Blood/cell_bender_processed/combined_annotated_final_with_dunedinpace_traits.h5ad"

adata = sc.read_h5ad(H5AD_PATH)
print("AnnData:", adata.shape, "(cells x genes)")
print("Layers  :", list(adata.layers.keys()))
print("obs cols:", adata.obs.columns.tolist())

In [ ]:
import scanpy as sc, numpy as np, pandas as pd


A =    adata  # backed = doesn't load the full matrix

sex_col = "sex"                            # the RAW numeric column (1/2); adjust if named differently
code = A.obs[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()

female_markers = ["XIST"]
male_markers   = ["RPS4Y1", "DDX3Y", "UTY", "EIF1AY", "KDM5D", "USP9Y"]
fm = [g for g in female_markers if g in A.var_names]
mm = [g for g in male_markers   if g in A.var_names]
print("female markers found:", fm, "| male markers found:", mm)

genes = fm + mm
sub = A[:, genes].to_memory().X           # load ONLY these few gene columns
sub = sub.toarray() if hasattr(sub, "toarray") else np.asarray(sub)

E = pd.DataFrame(sub, columns=genes)
E["code"] = code.values
means = E.groupby("code").mean()
means["XIST_like"] = means[fm].mean(axis=1) if fm else np.nan
means["Y_genes"]   = means[mm].mean(axis=1) if mm else np.nan

print("\nMean expression per code:")
print(means[["XIST_like", "Y_genes"]])

print("\nVerdict:")
for c, r in means.iterrows():
    verdict = "MALE" if r["Y_genes"] > r["XIST_like"] else "FEMALE"
    print(f"  code {c}:  XIST={r['XIST_like']:.2f}  Y-genes={r['Y_genes']:.2f}  ->  {verdict}")

In [ ]:
adata.obs.head(3)

In [ ]:
# =====================================================================
# 2) COUNTS BRUTS  (layer 'counts', entiers non normalises)
# =====================================================================
COUNTS_LAYER = "counts"

if COUNTS_LAYER in adata.layers:
    counts = adata.layers[COUNTS_LAYER]
else:
    # Fallback: look for a layer with integer values
    counts = None
    for k, m in adata.layers.items():
        s = m[:50]
        s = s.toarray() if hasattr(s, "toarray") else np.asarray(s)
        if np.allclose(s, np.round(s)):
            print(f"! layer 'counts' missing -> using '{k}'")
            counts = adata.layers[k]; break
    if counts is None:
        raise ValueError(f"Aucun layer de counts entiers. Layers={list(adata.layers.keys())}")

counts = counts.tocsr() if sparse.issparse(counts) else sparse.csr_matrix(counts)

_s = counts[:100].toarray()
print(f"counts : shape={counts.shape}, max={_s.max():.1f}, "
      f"entiers={np.allclose(_s, np.round(_s))}")

# The rest of the notebook assumes adata.X == raw counts (like the old mmread pipeline).
# Keep the original normalized/log matrix in a separate layer.
if "lognorm" not in adata.layers:
    adata.layers["lognorm"] = adata.X      # original X (normalized / log)
adata.X = counts                            # X = counts bruts
adata.layers["counts"] = counts             # used by HVG (seurat_v3) and scVI

In [ ]:
# =====================================================================
# 3) OBS: biomarkers + covariates (robust name resolution)
# =====================================================================
def _pick(cols, *cands):
    for c in cands:
        if c in cols:
            return c
    return None

cols = set(adata.obs.columns)
col_patient  = _pick(cols, "replicate", "patient_id", "orig.ident", "sample")
col_qcmt     = _pick(cols, "QC_MT.pct", "percent.mt", "pct_counts_mt", "percent_mt")
col_sex      = _pick(cols, "sex", "Sex", "gender")
col_batch    = _pick(cols, "batch_merging", "sample", "Batch")
col_celltype = _pick(cols, "CellType", "cellType", "cell_type", "celltype")

for name, col in [("patient", col_patient), ("QC_MT.pct", col_qcmt),
                  ("sex", col_sex), ("batch", col_batch), ("CellType", col_celltype)]:
    flag = "" if col is not None else "  <-- INTROUVABLE (a verifier)"
    print(f"  {name:10s} -> {col}{flag}")

# patient_id = join key with the biomarkers (== 'replicate' on the blood side)
adata.obs["patient_id"] = adata.obs[col_patient].astype(str).str.strip()

# nCount_RNA recomputed on the raw counts
adata.obs["nCount_RNA"] = np.asarray(counts.sum(axis=1)).ravel()

# QC_MT.pct (continu)
if col_qcmt is not None:
    adata.obs["QC_MT.pct"] = adata.obs[col_qcmt].astype("float32")

# batch_merging (categorical)
if col_batch is not None:
    adata.obs["batch_merging"] = adata.obs[col_batch].astype("category")

# CellType
if col_celltype is not None and col_celltype != "CellType":
    adata.obs["CellType"] = adata.obs[col_celltype].astype(str)

# sex -> F / M  (handles numeric 1/2 coding or strings already F/M)
sex_raw = adata.obs[col_sex]
uniq = set(pd.Series(sex_raw.unique()).dropna().astype(str))
if uniq <= {"1", "2", "1.0", "2.0"}:
    adata.obs["sex"] = pd.to_numeric(sex_raw, errors="coerce").map({1: "F", 2: "M"}).astype("category")
else:
    adata.obs["sex"] = sex_raw.astype(str).str.upper().str[0].map({"F": "F", "M": "M"}).astype("category")
print("sex :", adata.obs["sex"].value_counts(dropna=False).to_dict())

In [ ]:
# =====================================================================
# 4) TABLE 'blood' (biomarqueurs par cellule) reconstruite depuis obs
#    -> utilisee telle quelle par build_bio_indexed_from_blood (cell plus bas)
# =====================================================================
biomarker_cols = ["bmi26", "MeanArtPr26", "VOmax26",
                  "choles26", "trigly26", "hdl26", "urea26", "wbc26"]
present = [c for c in biomarker_cols if c in adata.obs.columns]
missing = [c for c in biomarker_cols if c not in adata.obs.columns]
if missing:
    print(f"! biomarkers missing from obs: {missing} (adjust the list in build_bio_indexed)")

blood = adata.obs.copy()
blood["replicate"] = adata.obs["patient_id"].values     # patient key
if "QC_MT.pct" not in blood.columns and col_qcmt is not None:
    blood["QC_MT.pct"] = adata.obs["QC_MT.pct"].values
print("blood:", blood.shape, "| biomarkers available:", present)

In [ ]:
# =====================================================================
# 5) VERIFICATIONS
# =====================================================================
need = ["patient_id", "nCount_RNA", "QC_MT.pct", "sex", "batch_merging", "CellType"]
print("obs prets :", [c for c in need if c in adata.obs.columns])
print("X = counts bruts ? min/max =", adata.X.min(), "/", adata.X.max())
print("layers :", list(adata.layers.keys()))
adata.obs[[c for c in need if c in adata.obs.columns]].head()

In [ ]:
adata.obs.columns

In [ ]:
cells = list(adata.obs_names)
print(len(cells), "cells")
print(cells[:10])   # affi

In [ ]:
adata.layers

In [ ]:
cell_id = "1406wC11.ACATTTCAGCGCACAA-1"
gene    = "ARHGAP15"

In [ ]:
# Check that the cell and gene exist
print(cell_id in adata.obs_names)   # should be True
print(gene in adata.var_names)      # should be True

In [ ]:
sub = adata[cell_id, gene].layers["counts"]

In [ ]:
# Si sparse → toarray()
if hasattr(sub, "toarray"):
    value = sub.toarray()[0, 0]
else:
    value = sub[0, 0]

print(f"Raw counts for {gene} in {cell_id} =", int(value))

In [ ]:
adata.obs["nCount_RNA"].iloc[1]

In [ ]:
adata.obs.columns

In [ ]:
def select_highly_variable_genes(
    adata: sc.AnnData,
    n_top_genes: int = 2000,
    flavor: str = 'seurat_v3',
    layer: str = 'counts',
    max_cells_for_hvg: int = 100_000,
    random_state: int = 0,
) -> sc.AnnData:
    """
    Standard method: selects the genes with the highest variability,
    avoiding giant AnnData copies that blow up RAM.

    Available flavors:
    - 'seurat': Classic (mean-variance)
    - 'seurat_v3': Adjusted variance (recommended for UMI)
    - 'cell_ranger': Like Cell Ranger

    Args:
        adata: full AnnData (modified in .var['highly_variable'])
        n_top_genes: Number of genes to keep
        flavor: Computation method
        layer: Layer to use ('counts' recommended)
        max_cells_for_hvg: max number of cells used to compute HVG.
                           If adata.n_obs > max_cells_for_hvg, cells are randomly
                           subsampled for the HVG computation.
        random_state: seed for the random draw.

    Returns:
        adata_reduced: AnnData reduced to the selected genes (full n_obs, n_vars = n_top_genes)
    """
    print("=" * 80)
    print("📊 METHOD 1: HIGHLY VARIABLE GENES (HVG)")
    print("=" * 80)

    n_cells = adata.n_obs
    print(f"📦 Input AnnData: n_obs={adata.n_obs:,}, n_vars={adata.n_vars:,}")

    # 1) Choose the subset for HVG computation (if too many cells)
    if (max_cells_for_hvg is not None) and (n_cells > max_cells_for_hvg):
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n_cells, size=max_cells_for_hvg, replace=False)
        adata_hvg = adata[idx].copy()   # copy only the selected cells
        print(f"⏳ Computing HVG on a subset of {max_cells_for_hvg:,} cells "
              f"(out of {n_cells:,} total) to save RAM.")
    else:
        adata_hvg = adata   # no giant copy
        print("⏳ Computing HVG on ALL cells (no subsampling).")

    # 2) Compute HVG on the chosen object (subset or full)
    print(f"⏳ Computing HVG (flavor: {flavor}, layer: {layer}, n_top_genes={n_top_genes})...")
    sc.pp.highly_variable_genes(
        adata_hvg,
        n_top_genes=n_top_genes,
        flavor=flavor,
        layer=layer,
        subset=False  # don't subset here, only fill .var['highly_variable']
    )
    # 3) Propagate the HVG mask to the full AnnData
    #    (var_names are the same: same genes, same order)
    hvg_mask = adata_hvg.var["highly_variable"].to_numpy()
    n_hvg = int(hvg_mask.sum())
    print(f"✓ {n_hvg} highly variable genes identified in the HVG object")

    # Make sure adata.var has the right column
    adata.var["highly_variable"] = False
    adata.var.loc[adata_hvg.var.index[hvg_mask], "highly_variable"] = True

    # 4) Subset the FULL AnnData on these HVG
    adata_reduced = adata[:, adata.var["highly_variable"]].copy()
    print(f"✓ Dimensions finales de adata_reduced: {adata_reduced.shape}")

    # 5) Statistics (if you already have this function)
    try:
        print_selection_stats(adata, 'highly_variable')
    except NameError:
        pass  # if the function does not exist in this context

    return adata_reduced
    

In [ ]:
adata_reduced = select_highly_variable_genes(adata)

In [ ]:
adata_reduced.write_h5ad("/home/nlapique/Blood/adata_reduced_HVG_2000_with_sex_qcmt.h5ad")

In [ ]:
counts = adata_reduced.obs["batch_merging"].value_counts()
# Keep only categories with ≥ 3 cells
valid = counts[counts >= 3].index
adata_reduced = adata_reduced[adata_reduced.obs["batch_merging"].isin(valid)].copy()

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_reduced,
    layer="counts",
    batch_key="batch_merging",          # ⬅️ batch technique
    categorical_covariate_keys=["sex"], # ⬅️ covariable
    continuous_covariate_keys=["QC_MT.pct"], # ⬅️ covariable
    # no gene_likelihood here!
)

In [ ]:
# 2) IMPORTANT: new model (DO NOT load an old one here)
#model_scvi = scvi.model.SCVI(adata_reduced)
model_scvi = scvi.model.SCVI(
    adata_reduced,
    n_latent=30,
    gene_likelihood="nb",   # ⬅️ THIS is correct
    # especially NOT: categorical_covariate_keys=...
)

In [ ]:
# 3) check registry
print("cat covs:", model_scvi.adata_manager.get_state_registry("extra_categorical_covs"))
print("cont covs:", model_scvi.adata_manager.get_state_registry("extra_continuous_covs"))

In [ ]:
scvi.settings.dl_num_workers = 6
model_scvi.train(max_epochs=300,  early_stopping=True, enable_progress_bar=True)

In [ ]:
model_scvi.save("/home/nlapique/Blood/model_scvi_adata_reduced_epochs_300_with_sex.h5ad", overwrite=True)

In [ ]:
import scvi
import scanpy as sc
import scipy.sparse as sp
import pandas as pd
from scvi.external import MRVI
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from scvi.external import MRVI


# load your AnnData object



In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
import os
import torch

# Check GPU status
print("Checking GPU availability...")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        print(f"  Memory reserved: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")

# Solution 1: Clear GPU memory
print("\nClearing GPU cache...")
torch.cuda.empty_cache()

In [ ]:
import scvi
import lightning
import torch

print("scvi-tools:", scvi.__version__)
print("lightning:", lightning.__version__)
print("torch:", torch.__version__)


In [ ]:
test_adata= sc.read_h5ad("/home/nlapique/Blood/adata_reduced_HVG_2000_with_sex_qcmt.h5ad")

In [ ]:
import scvi
import scanpy as sc
import pandas as pd
from scvi.external import MRVI

# Force PyTorch backend
import os
os.environ["SCVI_USE_TORCH"] = "1"

# Or use TorchMRVI directly
from scvi.external.mrvi_torch import TorchMRVI


test_adata.X = test_adata.X.tocsr()
sample_key = "sample"

# Setup with PyTorch
TorchMRVI.setup_anndata(test_adata, sample_key=sample_key)
model = TorchMRVI(test_adata)

# Train on GPU with PyTorch
model.train(
    max_epochs=2500,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    batch_size=1024,
    train_size=0.9,
    datasplitter_kwargs={
        "num_workers": 14,
        "pin_memory": True,
        "persistent_workers": True,
    },
    plan_kwargs={
        "n_epochs_kl_warmup": 50,
    },
    check_val_every_n_epoch=5,
)
# Save to specific directory
model.save("/home/nlapique/Blood/xgboost_mrvi_blood_model_3000_hvg_donnors_500epochs", overwrite=True)
model2=model


In [ ]:

import scvi
import scanpy as sc
import pandas as pd
import numpy as np
from scvi.external.mrvi_torch import TorchMRVI
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Force PyTorch backend
os.environ["SCVI_USE_TORCH"] = "1"

# Load your data
print("Loading data...")
test_adata = sc.read_h5ad('/home/nlapique/Blood/adata_men_removed_donors2.h5ad')
test_adata.X = test_adata.X.tocsr()

sample_key = "VOmax26_bin_male"

# Configuration
N_MODELS = 6  # Number of models to train
SEEDS = [42, 123, 456, 789, 2024, 3141] 
BASE_DIR = "/home/nlapique/Blood/mrvi_ensemble"
os.makedirs(BASE_DIR, exist_ok=True)

### Second act

In [ ]:
adata = sc.read_h5ad("/home/nlapique/Blood/adata_reduced_HVG_2000_with_sex_qcmt.h5ad")

In [ ]:
#[{"id":"a82311d1-e871-477f-8c47-e9d64f99e38c","cell_type":"code","source":"","metadata":{"trusted":true},"outputs":[],"execution_count":null}]
model = SCVI.load("/home/nlapique/Blood/model_scvi_adata_reduced_epochs_300_with_sex.h5ad", adata)

In [ ]:
adata.obsm["X_scvi_corrected"] = model.get_latent_representation()

In [ ]:
def build_bio_indexed_from_blood(
    blood: pd.DataFrame,
    biomarker_cols: list = ["sex", "qt_mct"],
) -> pd.DataFrame:
    """
    Builds bio_indexed (patient_id × biomarkers) from blood (cells × variables),
    using the 'replicate' column as the patient identifier.
    """

    if "replicate" not in blood.columns:
        raise ValueError("The blood DataFrame must contain a 'replicate' column.")

    missing = [c for c in biomarker_cols if c not in blood.columns]
    if missing:
        raise ValueError(f"Missing biomarker columns in blood: {missing}")

    # 1) Keep only replicate + biomarkers
    tmp = blood[["replicate", *biomarker_cols]].copy()

    # 2) Drop rows where biomarkers are NaN (if desired)
    tmp = tmp.dropna(subset=biomarker_cols, how="any")

    # 3) Check there are no contradictions between cells of the same patient
    n_unique = tmp.groupby("replicate")[biomarker_cols].nunique()
    bad = n_unique[(n_unique > 1).any(axis=1)]
    if len(bad) > 0:
        print("⚠️ Patients with non-unique biomarkers in blood:")
        print(bad.head())
        print("→ We will still take the first value per patient.")
        
    # 4) Move to patient level: 1 row per patient
    bio_indexed = (
        tmp.drop_duplicates(subset="replicate")
           .set_index("replicate")
           .sort_index()
    )

    # 5) Rename the index to 'patient_id' to match df_latent_by_patient

    return bio_indexed


## ⚙️ CONFIG — a single parameter to change

Change **`BIOMARKER`** below, then re-run from this cell onward: `bio_indexed` -> `df_pb_latent` -> distribution -> PDF report -> Excel export -> dotplots all use this variable. The name must exactly match a column of `blood`/`adata.obs`.

In [ ]:
# =====================================================================
# GLOBAL CONFIG  —  THE ONLY PLACE TO EDIT
# =====================================================================
BIOMARKER     = "PaceOfAgingP45"   # <<< THE ONLY PARAMETER TO CHANGE
TARGET_GROUP  = "M"                 # analyzed group (sex): "M" or "F"
N_ITER_PDF    = 50                  # iterations for the PDF report
N_ITER_EXCEL  = 50    
quantile_low=0.6
quantile_high=0.9
# iterations for the Excel gene export

# --- Automatic derived values (do not touch) ---
# Always-required covariates + the tested biomarker (deduplicated, order preserved)
EXTRA_GROUP_COLS = ["CurrentSmk26"]   # <<< extra covariates to carry through for alternative
                                      #     stratifications (e.g. splitting by smoking instead of sex)
BIO_COLS   = list(dict.fromkeys(["sex", "QC_MT.pct", BIOMARKER, *EXTRA_GROUP_COLS]))
EXCEL_FILE = f"RESULTATS_GENES_COMPLETS_{BIOMARKER}.xlsx"

# --- Check that the biomarker exists ---
try:
    if BIOMARKER not in blood.columns:
        cand = [c for c in blood.columns if BIOMARKER.lower() in c.lower()]
        raise KeyError(f"'{BIOMARKER}' not in blood.columns. Close matches: {cand}")
    print(f"✅ BIOMARKER = {BIOMARKER}")
    print(f"   BIO_COLS   = {BIO_COLS}")
    print(f"   EXCEL_FILE = {EXCEL_FILE}")
    print(f"   Groupe = {TARGET_GROUP} | n_iter PDF = {N_ITER_PDF} | n_iter Excel = {N_ITER_EXCEL}")
except NameError:
    print("⚠️ 'blood' not defined yet — run the .h5ad loading cell first.")


In [ ]:
print(TARGET_GROUP)

In [ ]:
bio_indexed = build_bio_indexed_from_blood(
    blood,
    biomarker_cols=BIO_COLS
)

In [ ]:
bio_indexed

In [ ]:
def make_pseudobulk_latent_celltype(
    adata,
    latent_key="X_scvi_corrected",
    patient_col="patient_id",
    celltype_col="CellType",
    min_cells_per_group=50,
):
    """
    Pseudobulk of the corrected scVI latent.
    Returns a DataFrame:
      index   = MultiIndex (patient_id, CellType)
      columns = latent_0 ... latent_{n_latent-1}
    """
    latent = adata.obsm[latent_key]   # (n_cells × n_latent)
    n_cells, n_latent = latent.shape
    print(f"Latent shape: {latent.shape}")

    obs = adata.obs[[patient_col, celltype_col]].copy()
    obs[patient_col] = obs[patient_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)

    latent_cols = [f"latent_{i}" for i in range(n_latent)]
    df_cells = pd.DataFrame(latent, columns=latent_cols)
    df_cells[patient_col] = obs[patient_col].values
    df_cells[celltype_col] = obs[celltype_col].values
    
    # Aggregation by (patient, cell type)
    grouped = df_cells.groupby([patient_col, celltype_col])
    # Filter and mean
    sizes = grouped.size()
    # Keep only (patient, celltype) combinations with ≥50 cells
    valid_groups = sizes[sizes >= min_cells_per_group].index

    df_pb = (
        grouped.mean()
               .loc[valid_groups]
               .sort_index()
    )

    print(
        f"Pseudobulk latent: {df_pb.shape[0]} groups (patient × CellType), "
        f"{df_pb.shape[1]} latent features"
    )
    return df_pb

In [ ]:
latent_key  = "X_scvi_corrected"
patient_col = "patient_id"
celltype_col = "CellType"   # adjust if needed

df_latent_by_patient = make_pseudobulk_latent_celltype(
    adata,
    latent_key=latent_key,
    patient_col=patient_col,
    celltype_col=celltype_col,
    min_cells_per_group=50,    # tu peux ajuster (ex 30)
)

In [ ]:
df_latent_by_patient.head(1)

In [ ]:
df_pb_latent = df_latent_by_patient.join(
    bio_indexed,
    on="patient_id",   # join key
    how="inner"        # keep only common patients
)

In [ ]:
#df_pb_latent["sex"] = "M" #Change all donors to M

In [ ]:
df_pb_latent.head(1)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. PARAMETERS ---
target_col = BIOMARKER   # Your continuous target
SEX_GROUP  = TARGET_GROUP       # "ALL" = males + females together | "M" / "F" for a single sex
                         #  -> should match target_group used in the XGBoost modeling

print(f"DISTRIBUTION ANALYSIS: {target_col}  (sex group: {SEX_GROUP})")

# --- 2. DATA PREPARATION ---
df_viz = df_pb_latent.copy()

# Clean up Index/IDs
if isinstance(df_viz.index, pd.MultiIndex): df_viz = df_viz.reset_index()
if 'patient_id' not in df_viz.columns:
    if 'index' in df_viz.columns: df_viz = df_viz.rename(columns={'index': 'patient_id'})
    else: df_viz['patient_id'] = df_viz.index.astype(str)

# Sex selection driven by SEX_GROUP (same mapping as the modeling functions)
sex_col = 'sex' if 'sex' in df_viz.columns else 'sex_patient'
df_viz[sex_col] = df_viz[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()
sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
df_viz['sex_mapped'] = df_viz[sex_col].map(sex_map).fillna('Unknown')
if str(SEX_GROUP).upper() in ('ALL', 'BOTH'):
    print(f"   -> No sex filter (both sexes): {df_viz['patient_id'].nunique()} patients.")
else:
    df_viz = df_viz[df_viz['sex_mapped'] == SEX_GROUP]
    print(f"   -> Sex filter '{SEX_GROUP}': {df_viz['patient_id'].nunique()} patients remaining.")

# Remove duplicates (one patient = one value)
df_viz = df_viz.drop_duplicates(subset=['patient_id'])

# --- 3. THRESHOLD COMPUTATION ---
# Ignore missing values for the computation
values = df_viz[target_col].dropna()
low_thresh = values.quantile(quantile_low)
high_thresh = values.quantile(quantile_high)
print(f"   -> LOW threshold  : <= {low_thresh:.2f}  (q0.50)")
print(f"   -> HIGH threshold : >= {high_thresh:.2f}  (q0.90)")

# Build the labels for counting
def label_patient(val):
    if pd.isna(val): return "Missing"
    if val >= high_thresh: return "High (Sick)"
    elif val <= low_thresh: return "Low (Healthy)"
    else: return "Middle (Excluded)"

df_viz['Category'] = df_viz[target_col].map(label_patient)

# --- 4. VISUALIZATION ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PLOT 1: Continuous histogram (the shape of the data)
sns.histplot(df_viz[target_col], kde=True, ax=axes[0], color='grey', bins=30)
axes[0].axvline(low_thresh, color='blue', linestyle='--', linewidth=2, label=f'Low ({low_thresh:.1f})')
axes[0].axvline(high_thresh, color='red', linestyle='--', linewidth=2, label=f'High ({high_thresh:.1f})')

# Shade the zones
y_min, y_max = axes[0].get_ylim()
axes[0].fill_betweenx([y_min, y_max], df_viz[target_col].min(), low_thresh, color='blue', alpha=0.1)
axes[0].fill_betweenx([y_min, y_max], high_thresh, df_viz[target_col].max(), color='red', alpha=0.1)

axes[0].set_title(f"Continuous Distribution of {target_col}", fontsize=14)
axes[0].legend()
axes[0].set_xlabel("Biomarker Value")

# PLOT 2: Count bars (class balance)
order = ["Low (Healthy)", "Middle (Excluded)", "High (Sick)"]
palette = {"Low (Healthy)": "navy", "Middle (Excluded)": "lightgrey", "High (Sick)": "firebrick"}

sns.countplot(data=df_viz, x='Category', order=order, palette=palette, ax=axes[1])
axes[1].set_title("Number of Patients per Class", fontsize=14)

for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=12, padding=3)

plt.tight_layout()
plt.show()

# --- 5. TEXT SUMMARY ---
print("\nHEADCOUNT SUMMARY:")
print(df_viz['Category'].value_counts().loc[order])


In [ ]:
def draw_table(df, title, pdf_object):
    fig, ax = plt.subplots(figsize=(8.5, 11)) # Standard A4 format
    ax.axis('off')
    
    # Page title
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Data preparation (rounding)
    table_data = df.copy()
    if 'Stability (%)' in table_data.columns:
        table_data['Stability (%)'] = table_data['Stability (%)'].apply(lambda x: f"{x:.1f}%")
    if 'Final Importance' in table_data.columns:
        table_data['Final Importance'] = table_data['Final Importance'].apply(lambda x: f"{x:.4f}")
    
    # Build the table
    cols = table_data.columns
    cell_text = table_data.values
    
    the_table = ax.table(
        cellText=cell_text,
        colLabels=cols,
        loc='center',
        cellLoc='center',
        colColours=['#eeeeee'] * len(cols) # Light grey header
    )
    
    # Styling
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(10)
    the_table.scale(1.2, 2) # More airy
    
    # Bold the header
    for (row, col), cell in the_table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold')
    
    pdf_object.savefig(fig, bbox_inches='tight')
    plt.close(fig)

# --- MAIN FUNCTION ---
def run_discovery_pdf_report(
    model, 
    adata, 
    df_clinical, 
    biomarker="urea26", 
    cell_type="Classical_monocytes_1",
    n_iterations=50, 
    stability_thresh=60, 
    n_top_genes=1000,
    target_group="M"
):
    # Output file name
    filename = f"Rapport_{biomarker}_{cell_type}_{target_group}.pdf"
    filename_csv = f"Tableau_{biomarker}_{cell_type}_{target_group}.csv"
    file_tree   = f"Arbre_{biomarker}_{cell_type}.pdf" 
    print(f"⚡️ ROBUST ANALYSIS & PDF REPORT: {filename}")

    # --- 1. PREPARATION ---
    pd.options.mode.chained_assignment = None 
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex): df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns: df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()
    sex_col = 'sex' if 'sex' in df_clin.columns else 'sex_patient'
    df_clin[sex_col] = df_clin[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()

    # Sex mapping
    sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
    df_clin['sex_mapped'] = df_clin[sex_col].map(sex_map).fillna('Unknown')

    # Filter
    df_target = df_clin[df_clin['sex_mapped'] == target_group].drop_duplicates(subset=['patient_id'])
    
    #target_sex = "2"
    #df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])
    #if len(df_male) < 10: target_sex = "M"; df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])

    #high = df_male[biomarker].quantile(0.70)
    #low = df_male[biomarker].quantile(0.30)
    #df_male['Label'] = np.nan
    #df_male.loc[df_male[biomarker] >= high, 'Label'] = 1
    #df_male.loc[df_male[biomarker] <= low, 'Label'] = 0
    #df_ml = df_male.dropna(subset=['Label'])
    
    high =  df_target[biomarker].quantile(quantile_high)
    low =  df_target[biomarker].quantile(quantile_low)
    df_target['Label'] = np.nan
    df_target.loc[df_target[biomarker] >= high, 'Label'] = 1
    df_target.loc[df_target[biomarker] <= low, 'Label'] = 0
    df_ml = df_target.dropna(subset=['Label'])
    
    print(f"   -> Task: {len(df_ml[df_ml['Label']==1])} High vs {len(df_ml[df_ml['Label']==0])} Low")

    if cell_type not in adata.obs['CellType'].values: return None
    adata_subset = adata[adata.obs['CellType'] == cell_type].copy()
    obs_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_subset = adata_subset[obs_pats.isin(df_ml['patient_id'])].copy()
    
    if hasattr(adata_subset.X, "toarray"): means = np.array(adata_subset.X.mean(axis=0)).flatten()
    else: means = np.mean(adata_subset.X, axis=0)
    hvg = pd.Series(means, index=adata_subset.var_names).sort_values(ascending=False).head(n_top_genes).index.tolist()
    
    # Discrete denoising
    X_denoised = model.get_normalized_expression(adata_subset, gene_list=hvg, library_size=1e4, return_numpy=True)
    df_expr = pd.DataFrame(X_denoised, columns=hvg)
    df_expr['Patient'] = adata_subset.obs['patient_id'].astype(str).str.strip().values
    X_full = df_expr.groupby('Patient').mean()
    
    common = X_full.index.intersection(df_ml['patient_id'])
    X_full = X_full.loc[common]
    y_full = df_ml.set_index('patient_id').loc[common]['Label']

    if len(X_full) < 20: return None

    # --- 2. ROBUSTNESS (50 ITERATIONS) ---
    print(f"   -> Computing robustness ({n_iterations} runs)", end="")
    gene_counts = pd.Series(0, index=hvg)
    
    for i in range(n_iterations):
        if i % 10 == 0: print(".", end="")
        seed = 42 + i
        X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.3, random_state=seed, stratify=y_full)
        clf = XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=3,
            subsample=0.8, colsample_bytree=0.6,
            device="cuda", tree_method="hist", eval_metric='logloss',
            random_state=seed
        )
        clf.fit(X_train, y_train)
        imps = pd.Series(clf.feature_importances_, index=X_full.columns)
        gene_counts[imps[imps > 0].index] += 1
    print(" Done.")

    stability_pct = (gene_counts / n_iterations) * 100
    stable_genes = stability_pct[stability_pct >= stability_thresh].index.tolist()
    if len(stable_genes) < 2: stable_genes = stability_pct.sort_values(ascending=False).head(10).index.tolist()
    
    # --- 3. FINAL MODEL ---
    X_stable = X_full[stable_genes]
    X_train, X_test, y_train, y_test = train_test_split(X_stable, y_full, test_size=0.3, random_state=42, stratify=y_full)
    final_clf = XGBClassifier(
        n_estimators=500, learning_rate=0.02, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        device="cuda", tree_method="hist", eval_metric='logloss',
        random_state=42
    )
    final_clf.fit(X_train, y_train)
   
    y_proba = final_clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    print(f"🎯 FINAL AUC: {roc_auc:.2f}")
    if roc_auc < 0.59:
        print("❌ UNSTABLE RESULT: this was a fluke.")
        return None

    # --- 4. TABLE PREPARATION ---
    df_res = pd.DataFrame({
        "Gene": stable_genes,
        "Stability (%)": stability_pct[stable_genes].values,
        "Final Importance": final_clf.feature_importances_
    }).sort_values("Stability (%)", ascending=False).head(25) # Top 25 for the PDF

    # --- CSV EXPORT (NEW) ---
    # Save the whole table (not just the top 25) in French Excel format (; and ,)
    df_res.to_csv(filename_csv, index=False, sep=';', decimal=',')
    print(f"   -> 💾 CSV saved: {filename_csv}")

    # --- 5. PDF GENERATION ---
    print(f"   -> Creating PDF file: {filename} ...")
    
    with PdfPages(filename) as pdf:
        
        # PAGE 1 : ROC CURVE
        fig_roc = plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'ROC (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
        plt.title(f"Diagnostic Performance: {biomarker} ({cell_type})", fontsize=14)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        pdf.savefig(fig_roc) # Save into the PDF
        plt.show() # Show in the notebook
        plt.close(fig_roc)
        
        # PAGE 2 : SHAP SUMMARY
        explainer = shap.TreeExplainer(final_clf)
        shap_values = explainer.shap_values(X_stable)
        
        fig_shap = plt.figure()
        shap.summary_plot(shap_values, X_stable, show=False, max_display=20)
        plt.title(f"Biological Signature (SHAP) - {cell_type}", fontsize=14)
        plt.tight_layout()
        pdf.savefig(fig_shap, bbox_inches='tight')
        plt.show()
        plt.close(fig_shap)
        
        # PAGE 3: THE NICE TABLE
        draw_table(df_res, f"Top Robust Genes: {cell_type}", pdf)
        
        # PAGE 4: DECISION TREE (NEW)
        # Create a very large figure so the tree is legible
        #fig_tree, ax_tree = plt.subplots(figsize=(20, 12))
        #xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='LR')
        #plt.title(f"Simplified Decision Tree - {cell_type}", fontsize=18)
        #pdf.savefig(fig_tree, bbox_inches='tight')
        #plt.close(fig_tree)

        # --- 4. SAVE VERTICAL TREE (SEPARATE PDF) ---
        try:
            # "Vertical poster" format (30x40 inches)
            fig_tree, ax_tree = plt.subplots(figsize=(30, 40)) 
            # rankdir='TB' = Top to Bottom (Vertical)
            xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='TB') 
            
            plt.title(f"Vertical Decision Tree - {cell_type}", fontsize=30, pad=30)
            plt.savefig(file_tree, bbox_inches='tight') # Save the file directly
            plt.close(fig_tree)
            print(f"   -> 🌳 Vertical tree saved: {file_tree}")
        except Exception as e:
            print(f"   ⚠️ Tree error: {e}")
        
    print(f"✅ Done! File saved: {filename}")
    display(df_res.head(10).style.bar(subset=['Stability (%)'], color='#d65f5f'))
    
    return final_clf, df_res

In [ ]:
# All cell types present in the data (instead of a fixed list).
# Source: the df_pb_latent pseudobulk (types actually available after aggregation),
# with fallback to adata.obs["CellType"].
try:
    my_valid_types = sorted(
        pd.Series(df_pb_latent.index.get_level_values("CellType")).astype(str).unique().tolist()
    )
    print("Source: df_pb_latent (CellType)")
except Exception:
    my_valid_types = sorted(adata.obs["CellType"].astype(str).unique().tolist())
    print("Source: adata.obs['CellType']")

print(f"{len(my_valid_types)} cell types:")
for t in my_valid_types:
    print("  -", t)


In [ ]:
my_biomarkers = ["urea26", "VOmax26", "MeanArtPr26", "bmi26", "choles26", "trigly26","hdl26"]

In [ ]:
def draw_table(df, title, pdf_object):
    fig, ax = plt.subplots(figsize=(8.5, 11)) # Standard A4 format
    ax.axis('off')
    
    # Page title
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Data preparation (rounding)
    table_data = df.copy()
    if 'Stability (%)' in table_data.columns:
        table_data['Stability (%)'] = table_data['Stability (%)'].apply(lambda x: f"{x:.1f}%")
    if 'Final Importance' in table_data.columns:
        table_data['Final Importance'] = table_data['Final Importance'].apply(lambda x: f"{x:.4f}")
    
    # Build the table
    cols = table_data.columns
    cell_text = table_data.values
    
    the_table = ax.table(
        cellText=cell_text,
        colLabels=cols,
        loc='center',
        cellLoc='center',
        colColours=['#eeeeee'] * len(cols) # Light grey header
    )
    
    # Styling
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(10)
    the_table.scale(1.2, 2) # More airy
    
    # Bold the header
    for (row, col), cell in the_table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold')
    
    pdf_object.savefig(fig, bbox_inches='tight')
    plt.close(fig)


In [ ]:
# ⚠️ SUPERSEDED — this definition is immediately overwritten by the patched version
# a few cells below (train/test split BEFORE gene selection). Left here only for history.
# --- MAIN FUNCTION ---
def run_discovery_pdf_report(
    model, 
    adata, 
    df_clinical, 
    biomarker="PaceOfAgingP45", 
    cell_type="Classical_monocyte",
    n_iterations=50, 
    stability_thresh=60, 
    n_top_genes=1000,
    target_group="ALL"
):
    # Output file name
    filename = f"Rapport_{biomarker}_{cell_type}_{target_group}.pdf"
    filename_csv = f"Tableau_{biomarker}_{cell_type}_{target_group}.csv"
    file_tree   = f"Arbre_{biomarker}_{cell_type}.pdf" 
    print(f"⚡️ ROBUST ANALYSIS & PDF REPORT: {filename}")

    # --- 1. PREPARATION ---
    pd.options.mode.chained_assignment = None 
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex): df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns: df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()
    sex_col = 'sex' if 'sex' in df_clin.columns else 'sex_patient'
    df_clin[sex_col] = df_clin[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()

    # Sex mapping
    sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
    df_clin['sex_mapped'] = df_clin[sex_col].map(sex_map).fillna('Unknown')

    # Filter
    df_target = df_clin[df_clin['sex_mapped'] == target_group].drop_duplicates(subset=['patient_id'])
    
    #target_sex = "2"
    #df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])
    #if len(df_male) < 10: target_sex = "M"; df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])

    #high = df_male[biomarker].quantile(0.70)
    #low = df_male[biomarker].quantile(0.30)
    #df_male['Label'] = np.nan
    #df_male.loc[df_male[biomarker] >= high, 'Label'] = 1
    #df_male.loc[df_male[biomarker] <= low, 'Label'] = 0
    #df_ml = df_male.dropna(subset=['Label'])
    
    high =  df_target[biomarker].quantile(quantile_high)
    low =  df_target[biomarker].quantile(quantile_low)
    df_target['Label'] = np.nan
    df_target.loc[df_target[biomarker] >= high, 'Label'] = 1
    df_target.loc[df_target[biomarker] <= low, 'Label'] = 0
    df_ml = df_target.dropna(subset=['Label'])
    
    print(f"   -> Task: {len(df_ml[df_ml['Label']==1])} High vs {len(df_ml[df_ml['Label']==0])} Low")

    if cell_type not in adata.obs['CellType'].values: return None
    adata_subset = adata[adata.obs['CellType'] == cell_type].copy()
    obs_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_subset = adata_subset[obs_pats.isin(df_ml['patient_id'])].copy()
    
    if hasattr(adata_subset.X, "toarray"): means = np.array(adata_subset.X.mean(axis=0)).flatten()
    else: means = np.mean(adata_subset.X, axis=0)
    hvg = pd.Series(means, index=adata_subset.var_names).sort_values(ascending=False).head(n_top_genes).index.tolist()
    
    # Discrete denoising
    X_denoised = model.get_normalized_expression(adata_subset, gene_list=hvg, library_size=1e4, return_numpy=True)
    df_expr = pd.DataFrame(X_denoised, columns=hvg)
    df_expr['Patient'] = adata_subset.obs['patient_id'].astype(str).str.strip().values
    X_full = df_expr.groupby('Patient').mean()
    
    common = X_full.index.intersection(df_ml['patient_id'])
    X_full = X_full.loc[common]
    y_full = df_ml.set_index('patient_id').loc[common]['Label']

    if len(X_full) < 20: return None

    # --- 2. ROBUSTNESS (50 ITERATIONS) ---
    print(f"   -> Computing robustness ({n_iterations} runs)", end="")
    gene_counts = pd.Series(0, index=hvg)
    
    for i in range(n_iterations):
        if i % 10 == 0: print(".", end="")
        seed = 42 + i
        X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.3, random_state=seed, stratify=y_full)
        clf = XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=3,
            subsample=0.8, colsample_bytree=0.6,
            device="cuda", tree_method="hist", eval_metric='logloss',
            random_state=seed
        )
        clf.fit(X_train, y_train)
        imps = pd.Series(clf.feature_importances_, index=X_full.columns)
        gene_counts[imps[imps > 0].index] += 1
    print(" Done.")

    stability_pct = (gene_counts / n_iterations) * 100
    stable_genes = stability_pct[stability_pct >= stability_thresh].index.tolist()
    if len(stable_genes) < 2: stable_genes = stability_pct.sort_values(ascending=False).head(10).index.tolist()
    
    # --- 3. FINAL MODEL ---
    X_stable = X_full[stable_genes]
    X_train, X_test, y_train, y_test = train_test_split(X_stable, y_full, test_size=0.3, random_state=42, stratify=y_full)
    final_clf = XGBClassifier(
        n_estimators=500, learning_rate=0.02, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        device="cuda", tree_method="hist", eval_metric='logloss',
        random_state=42
    )
    final_clf.fit(X_train, y_train)
   
    y_proba = final_clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    print(f"🎯 FINAL AUC: {roc_auc:.2f}")
    if roc_auc < 0.59:
        print("❌ UNSTABLE RESULT: this was a fluke.")
        return None

    # --- 4. TABLE PREPARATION ---
        # --- 4. TABLE PREPARATION ---
    # FULL gene list for the CSV: every gene with stability > 50%.
    # (final-model importance only exists for the >=stability_thresh genes; others -> NaN)
    imp_map = dict(zip(stable_genes, final_clf.feature_importances_))
    df_all = pd.DataFrame({"Gene": stability_pct.index, "Stability (%)": stability_pct.values})
    df_all = df_all[df_all["Stability (%)"] > 50].sort_values("Stability (%)", ascending=False)
    df_all["Final Importance"] = df_all["Gene"].map(imp_map)

    # Top 25 of the stricter (>=stability_thresh) set, for the PDF table + plots
    df_res = pd.DataFrame({
        "Gene": stable_genes,
        "Stability (%)": stability_pct[stable_genes].values,
        "Final Importance": final_clf.feature_importances_
    }).sort_values("Stability (%)", ascending=False).head(25)

    # --- CSV EXPORT: FULL list (all genes with stability > 50%) ---
    df_all.to_csv(filename_csv, index=False, sep=';', decimal=',')
    print(f"   -> 💾 CSV saved ({len(df_all)} genes >50%): {filename_csv}")

    # --- 5. PDF GENERATION ---
    print(f"   -> Creating PDF file: {filename} ...")
    
    with PdfPages(filename) as pdf:
        
        # PAGE 1 : ROC CURVE
        fig_roc = plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'ROC (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
        plt.title(f"Diagnostic Performance: {biomarker} ({cell_type})", fontsize=14)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        pdf.savefig(fig_roc) # Save into the PDF
        plt.show() # Show in the notebook
        plt.close(fig_roc)
        
        # PAGE 2 : SHAP SUMMARY
        explainer = shap.TreeExplainer(final_clf)
        shap_values = explainer.shap_values(X_stable)
        
        fig_shap = plt.figure()
        shap.summary_plot(shap_values, X_stable, show=False, max_display=20)
        plt.title(f"Biological Signature (SHAP) - {cell_type}", fontsize=14)
        plt.tight_layout()
        pdf.savefig(fig_shap, bbox_inches='tight')
        plt.show()
        plt.close(fig_shap)
        
        # PAGE 3: THE NICE TABLE
        draw_table(df_res, f"Top Robust Genes: {cell_type}", pdf)
        
        # PAGE 4: DECISION TREE (NEW)
        # Create a very large figure so the tree is legible
        #fig_tree, ax_tree = plt.subplots(figsize=(20, 12))
        #xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='LR')
        #plt.title(f"Simplified Decision Tree - {cell_type}", fontsize=18)
        #pdf.savefig(fig_tree, bbox_inches='tight')
        #plt.close(fig_tree)

        # --- 4. SAVE VERTICAL TREE (SEPARATE PDF) ---
        try:
            # "Vertical poster" format (30x40 inches)
            fig_tree, ax_tree = plt.subplots(figsize=(30, 40)) 
            # rankdir='TB' = Top to Bottom (Vertical)
            xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='TB') 
            
            plt.title(f"Vertical Decision Tree - {cell_type}", fontsize=30, pad=30)
            plt.savefig(file_tree, bbox_inches='tight') # Save the file directly
            plt.close(fig_tree)
            print(f"   -> 🌳 Vertical tree saved: {file_tree}")
        except Exception as e:
            print(f"   ⚠️ Tree error: {e}")
        
    print(f"✅ Done! File saved: {filename}")
    display(df_res.head(10).style.bar(subset=['Stability (%)'], color='#d65f5f'))
    
    return final_clf, df_res

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

for cell_type in my_valid_types: # or my_valid_types
    try:
        # 1. Capture the result in a temporary variable
        result = run_discovery_pdf_report(
            model=model, 
            adata=adata, 
            df_clinical=df_pb_latent, 
            biomarker=BIOMARKER, 
            cell_type=cell_type, 
            n_iterations=N_ITER_PDF,
            target_group=TARGET_GROUP
        )
        
        # 2. If the result is empty (None), it means "Not enough patients" or "Type absent"
        if result is None:
            continue # Silently move on to the next type
            
        # 3. If OK, we can unpack the variables
        clf, table = result

    except Exception as e:
        # On an unexpected bug, print it but do not block the whole script
        print(f"⚠️ CRASH on {cell_type}: {e}")
        continue

print("\n🏁 ALL DONE! CHECK YOUR PDF FILES.")


In [ ]:
# --- MAIN FUNCTION (v2: train/test separation before gene selection + train-vs-test overfit gap) ---
def run_discovery_pdf_report(
    model, adata, df_clinical,
    biomarker="PaceOfAgingP45", cell_type="Classical_monocyte",
    n_iterations=50, stability_thresh=60, n_top_genes=1000, target_group="ALL",
    exclude_patients=None,        # <<< donors to remove this round (list of patient_id)
    make_pdf=True,                # <<< False -> skip PDF/SHAP/tree (fast, for the jackknife loop)
    min_auc=0.0,                  # <<< keep every round; set >0 only to drop weak runs
    outer_test_size=0.3,          # <<< NEW: held-out fraction, set aside BEFORE gene selection ever runs
    outer_random_state=42,        # <<< NEW: fixed split so repeated calls stay comparable
    min_train=15, min_test=5,     # <<< NEW: minimum group sizes on each side of the split
    n_perm=0,                     # <<< NEW: >0 runs a label-permutation null to calibrate stability_thresh
                                   #     (expensive: multiplies runtime by n_perm -> use standalone, NOT inside
                                   #      the jackknife loop in the next cell)
    n_iterations_perm=20,         # <<< NEW: iterations per permutation round (kept low to bound cost)
    inner_val_size=0.3,           # <<< NEW: carve this fraction OUT OF the train donors as a held-out
                                   #     inner-validation slice, scored with a fresh model using the
                                   #     already-selected stable_genes (outer test_pat stays untouched)
    n_estimators=500,             # <<< NEW: tree count for the FINAL and inner-val classifiers
                                   #     (the stability-selection XGB stays fixed at 300 trees / depth 3 --
                                   #      only the final model's capacity is exposed here for tuning)
    max_depth=4,                  # <<< NEW: max tree depth for the FINAL and inner-val classifiers
    group_col=None,               # <<< NEW: column to stratify on INSTEAD OF sex (e.g. "CurrentSmk26").
                                   #     None -> old behavior (stratify on sex_mapped, target_group="M"/"F"/"ALL").
                                   #     Set -> target_group is compared against THIS column's raw values
                                   #     (e.g. group_col="CurrentSmk26", target_group=1.0 for smokers).
    label_cutoffs="group",        # <<< NEW: where the High/Low biomarker cutoffs come from.
                                   #     "group"  -> quantiles of the target_group's donors only (old behavior:
                                   #                 each sex/tertile gets its OWN definition of High/Low)
                                   #     "global" -> quantiles of ALL donors in df_clinical, computed BEFORE the
                                   #                 target_group filter, so every group shares one definition
                                   #     (low, high) -> use these raw biomarker values directly as the cutoffs
):
    filename = f"Rapport_{biomarker}_{cell_type}_{target_group}.pdf"
    filename_csv = f"Tableau_{biomarker}_{cell_type}_{target_group}.csv"
    file_tree   = f"Arbre_{biomarker}_{cell_type}.pdf"
    if make_pdf: print(f"⚡️ ROBUST ANALYSIS & PDF REPORT: {filename}")

    # --- 1. LABELS (unchanged) ---
    pd.options.mode.chained_assignment = None
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex): df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns: df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()
    sex_col = 'sex' if 'sex' in df_clin.columns else 'sex_patient'
    df_clin[sex_col] = df_clin[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()

    sex_map = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}
    df_clin['sex_mapped'] = df_clin[sex_col].map(sex_map).fillna('Unknown')

    filter_col = group_col if group_col is not None else 'sex_mapped'
    if str(target_group).upper() in ('ALL', 'BOTH'):
        df_target = df_clin.drop_duplicates(subset=['patient_id'])
    else:
        df_target = df_clin[df_clin[filter_col] == target_group].drop_duplicates(subset=['patient_id'])

    if isinstance(label_cutoffs, (tuple, list)):
        low, high = label_cutoffs
    elif label_cutoffs == "global":
        df_all_donors = df_clin.drop_duplicates(subset=['patient_id'])
        high = df_all_donors[biomarker].quantile(quantile_high)
        low  = df_all_donors[biomarker].quantile(quantile_low)
    elif label_cutoffs == "group":
        high = df_target[biomarker].quantile(quantile_high)
        low  = df_target[biomarker].quantile(quantile_low)
    else:
        raise ValueError(f"label_cutoffs must be 'group', 'global' or a (low, high) tuple, got {label_cutoffs!r}")
    if low > high:
        raise ValueError(f"label cutoffs overlap: low={low} > high={high}")
    df_target['Label'] = np.nan
    df_target.loc[df_target[biomarker] >= high, 'Label'] = 1
    df_target.loc[df_target[biomarker] <= low, 'Label'] = 0
    df_ml = df_target.dropna(subset=['Label'])

    if exclude_patients is not None:
        excl = {str(x).strip() for x in exclude_patients}
        df_ml = df_ml[~df_ml['patient_id'].isin(excl)]

    if cell_type not in adata.obs['CellType'].values: return None
    if df_ml['Label'].nunique() < 2 or len(df_ml) < (min_train + min_test): return None

    if make_pdf:
        print(f"   -> Task: {len(df_ml[df_ml['Label']==1])} High vs {len(df_ml[df_ml['Label']==0])} Low "
              f"(High >= {high:.3f}, Low <= {low:.3f}, label_cutoffs={label_cutoffs!r})")

    # --- 2. OUTER SPLIT FIRST -- nothing below this line may look at the test patients ---
    #     until the single "evaluate the final model" step at the very end.
    train_pat, test_pat = train_test_split(
        df_ml['patient_id'], test_size=outer_test_size,
        random_state=outer_random_state, stratify=df_ml['Label']
    )
    train_pat, test_pat = set(train_pat), set(test_pat)
    labels_by_patient = df_ml.set_index('patient_id')['Label']

    adata_subset = adata[adata.obs['CellType'] == cell_type].copy()
    obs_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_subset = adata_subset[obs_pats.isin(df_ml['patient_id'])].copy()
    subset_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_train = adata_subset[subset_pats.isin(train_pat)].copy()
    adata_test  = adata_subset[subset_pats.isin(test_pat)].copy()

    def _pseudobulk(adata_grp, gene_list):
        X_den = model.get_normalized_expression(adata_grp, gene_list=gene_list, library_size=1e4, return_numpy=True)
        df_expr = pd.DataFrame(X_den, columns=gene_list)
        df_expr['Patient'] = adata_grp.obs['patient_id'].astype(str).str.strip().values
        return df_expr.groupby('Patient').mean()

    # --- 3. CANDIDATE GENES from TRAIN cells only (unsupervised, but still test-blind) ---
    if hasattr(adata_train.X, "toarray"): means = np.array(adata_train.X.mean(axis=0)).flatten()
    else: means = np.mean(adata_train.X, axis=0)
    hvg = pd.Series(means, index=adata_train.var_names).sort_values(ascending=False).head(n_top_genes).index.tolist()

    X_train_full = _pseudobulk(adata_train, hvg)
    X_test_full  = _pseudobulk(adata_test, hvg)
    X_train_full = X_train_full.loc[X_train_full.index.intersection(train_pat)]
    X_test_full  = X_test_full.loc[X_test_full.index.intersection(test_pat)]
    y_train_full = labels_by_patient.loc[X_train_full.index]
    y_test_full  = labels_by_patient.loc[X_test_full.index]

    if len(X_train_full) < min_train or len(X_test_full) < min_test: return None
    if y_train_full.nunique() < 2 or y_test_full.nunique() < 2: return None

    def _stability_run(X, y, seed_offset):
        counts = pd.Series(0, index=hvg)
        for i in range(n_iterations):
            seed = seed_offset + i
            Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
            clf = XGBClassifier(
                n_estimators=300, learning_rate=0.03, max_depth=3,
                subsample=0.8, colsample_bytree=0.6,
                device="cuda", tree_method="hist", eval_metric='logloss',   # <-- set device="cpu" if CUDA errors
                random_state=seed
            )
            clf.fit(Xtr, ytr)
            imps = pd.Series(clf.feature_importances_, index=X.columns)
            counts[imps[imps > 0].index] += 1
        return counts

    # --- 4. ROBUSTNESS / STABILITY -- resampled ONLY inside the TRAIN patients ---
    if make_pdf: print(f"   -> Computing robustness on TRAIN only ({n_iterations} runs)", end="")
    gene_counts = _stability_run(X_train_full, y_train_full, seed_offset=42)
    if make_pdf: print(" Done.")
    stability_pct = (gene_counts / n_iterations) * 100

    # --- 5. OPTIONAL: label-permutation null, also computed on TRAIN only ---
    effective_thresh = stability_thresh
    empirical_thresh = None
    if n_perm > 0:
        if make_pdf: print(f"   -> Calibrating threshold with {n_perm} label permutations", end="")
        rng = np.random.default_rng(0)
        null_max = []
        for p in range(n_perm):
            y_perm = pd.Series(rng.permutation(y_train_full.values), index=y_train_full.index)
            perm_counts = pd.Series(0, index=hvg)
            for i in range(n_iterations_perm):
                seed = 900_000 + p * 1000 + i
                Xtr, _, ytr, _ = train_test_split(X_train_full, y_perm, test_size=0.3, random_state=seed, stratify=y_perm)
                clf = XGBClassifier(
                    n_estimators=300, learning_rate=0.03, max_depth=3,
                    subsample=0.8, colsample_bytree=0.6,
                    device="cuda", tree_method="hist", eval_metric='logloss',
                    random_state=seed
                )
                clf.fit(Xtr, ytr)
                imps = pd.Series(clf.feature_importances_, index=X_train_full.columns)
                perm_counts[imps[imps > 0].index] += 1
            null_max.append((perm_counts / n_iterations_perm * 100).max())
            if make_pdf and p % 5 == 0: print(".", end="")
        if make_pdf: print(" Done.")
        empirical_thresh = float(np.percentile(null_max, 95))
        effective_thresh = max(stability_thresh, empirical_thresh)
        if make_pdf:
            print(f"   -> Null 95th percentile stability: {empirical_thresh:.1f}% "
                  f"(using max({stability_thresh}, {empirical_thresh:.1f}) = {effective_thresh:.1f}% as cutoff)")

    stable_genes = stability_pct[stability_pct >= effective_thresh].index.tolist()
    if len(stable_genes) < 2:
        stable_genes = stability_pct.sort_values(ascending=False).head(10).index.tolist()

    X_train_stable = X_train_full[stable_genes]
    X_test_stable  = X_test_full[stable_genes]

    # --- 5b. INNER VALIDATION -- split the TRAIN donors again (sub-train / sub-val), fit a FRESH
    #     model on the sub-train slice using the stable_genes already chosen above, and score it on
    #     the sub-val slice. This never touches test_pat; it's a second, independent check of
    #     generalization using only donors the outer split already called "train".
    inner_val_auc = np.nan
    n_subtrain = n_subval = 0
    if y_train_full.nunique() == 2 and len(X_train_stable) >= (min_train + min_test):
        try:
            sub_train_pat, sub_val_pat = train_test_split(
                X_train_stable.index, test_size=inner_val_size,
                random_state=outer_random_state, stratify=y_train_full.loc[X_train_stable.index]
            )
            X_subtrain, X_subval = X_train_stable.loc[sub_train_pat], X_train_stable.loc[sub_val_pat]
            y_subtrain, y_subval = y_train_full.loc[sub_train_pat], y_train_full.loc[sub_val_pat]
            n_subtrain, n_subval = len(X_subtrain), len(X_subval)
            if y_subtrain.nunique() == 2 and y_subval.nunique() == 2:
                inner_clf = XGBClassifier(
                    n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth,
                    subsample=0.8, colsample_bytree=0.8,
                    device="cuda", tree_method="hist", eval_metric='logloss',   # <-- set device="cpu" if CUDA errors
                    random_state=42
                )
                inner_clf.fit(X_subtrain, y_subtrain)
                y_proba_subval = inner_clf.predict_proba(X_subval)[:, 1]
                fpr_iv, tpr_iv, _ = roc_curve(y_subval, y_proba_subval)
                inner_val_auc = auc(fpr_iv, tpr_iv)
        except ValueError:
            pass   # too few donors of one class to stratify the inner split -> leave as NaN

    # --- 6. FINAL MODEL -- fit on ALL train patients, evaluate on the never-touched test patients ---
    final_clf = XGBClassifier(
        n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth,
        subsample=0.8, colsample_bytree=0.8,
        device="cuda", tree_method="hist", eval_metric='logloss',       # <-- set device="cpu" if CUDA errors
        random_state=42
    )
    final_clf.fit(X_train_stable, y_train_full)

    y_proba = final_clf.predict_proba(X_test_stable)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_full, y_proba)
    roc_auc = auc(fpr, tpr)

    # --- overfitting metric: same model, scored on the TRAIN patients it was fit on ---
    y_proba_train = final_clf.predict_proba(X_train_stable)[:, 1]
    fpr_tr, tpr_tr, _ = roc_curve(y_train_full, y_proba_train)
    train_auc = auc(fpr_tr, tpr_tr)
    overfit_gap = train_auc - roc_auc   # large positive gap -> memorizing train, not generalizing

    # gap between in-sample train score and the FRESH inner-val model's score, both computed without
    # ever touching test_pat -- cheaper/noisier than overfit_gap (which uses the real held-out test
    # set) but usable for hyperparameter tuning without peeking at test.
    overfit_gap_innerval = (train_auc - inner_val_auc) if not np.isnan(inner_val_auc) else np.nan

    if make_pdf:
        print(f"🎯 TEST (held-out) AUC: {roc_auc:.2f} | TRAIN AUC: {train_auc:.2f} | "
              f"OVERFIT GAP: {overfit_gap:+.2f}")
        print(f"   -> Inner-val AUC (fresh model, sub-train {n_subtrain} / sub-val {n_subval}, "
              f"carved out of the 70% train donors, real test set untouched): {inner_val_auc:.2f} "
              f"| INNER OVERFIT GAP: {overfit_gap_innerval:+.2f}")
    if roc_auc < min_auc:
        return None

    # per-patient scores: TEST scores are honest (model never saw these patients before this line);
    # TRAIN scores are in-sample/optimistic and flagged as such.
    all_ids = list(X_train_stable.index) + list(X_test_stable.index)
    all_X = pd.concat([X_train_stable, X_test_stable])
    scores_all = final_clf.predict_proba(all_X)[:, 1]
    lbl = labels_by_patient.loc[all_ids].astype(int)
    df_scores = pd.DataFrame({
        "patient_id": all_ids,
        "label": lbl.values,
        "category": np.where(lbl.values == 1, "High", "Low"),
        "score_high": scores_all,
        "predicted": np.where(scores_all >= 0.5, "High", "Low"),
        "in_train": [pid in X_train_stable.index for pid in all_ids],  # True = optimistic (seen in training)
    })

    # --- 7. TABLE PREPARATION ---
    imp_map = dict(zip(stable_genes, final_clf.feature_importances_))
    df_all = pd.DataFrame({"Gene": stability_pct.index, "Stability (%)": stability_pct.values})
    df_all = df_all[df_all["Stability (%)"] > 50].sort_values("Stability (%)", ascending=False)
    df_all["Final Importance"] = df_all["Gene"].map(imp_map)

    df_res = pd.DataFrame({
        "Gene": stable_genes,
        "Stability (%)": stability_pct[stable_genes].values,
        "Final Importance": final_clf.feature_importances_
    }).sort_values("Stability (%)", ascending=False).head(25)

    # --- 8. PDF (only on demand) ---
    if make_pdf:
        df_all.to_csv(filename_csv, index=False, sep=';', decimal=',')
        print(f"   -> 💾 CSV saved ({len(df_all)} genes >50%): {filename_csv}")
        print(f"   -> Creating PDF file: {filename} ...")
        with PdfPages(filename) as pdf:
            fig_roc = plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'Test ROC (AUC = {roc_auc:.2f}, held-out)')
            plt.plot(fpr_tr, tpr_tr, color='#1f77b4', lw=2, linestyle=':',
                     label=f'Train ROC (AUC = {train_auc:.2f}, in-sample)')
            plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
            plt.title(f"Diagnostic Performance: {biomarker} ({cell_type})\nOverfit gap = {overfit_gap:+.2f}", fontsize=14)
            plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
            plt.legend(loc="lower right"); plt.grid(alpha=0.3)
            pdf.savefig(fig_roc); plt.show(); plt.close(fig_roc)

            explainer = shap.TreeExplainer(final_clf)
            shap_values = explainer.shap_values(X_train_stable)  # SHAP on train is fine, it's just explaining the model
            fig_shap = plt.figure()
            shap.summary_plot(shap_values, X_train_stable, show=False, max_display=20)
            plt.title(f"Biological Signature (SHAP) - {cell_type}", fontsize=14)
            plt.tight_layout(); pdf.savefig(fig_shap, bbox_inches='tight'); plt.show(); plt.close(fig_shap)

            draw_table(df_res, f"Top Robust Genes: {cell_type}", pdf)
            try:
                fig_tree, ax_tree = plt.subplots(figsize=(30, 40))
                xgb.plot_tree(final_clf, num_trees=0, ax=ax_tree, rankdir='TB')
                plt.title(f"Vertical Decision Tree - {cell_type}", fontsize=30, pad=30)
                plt.savefig(file_tree, bbox_inches='tight'); plt.close(fig_tree)
                print(f"   -> 🌳 Vertical tree saved: {file_tree}")
            except Exception as e:
                print(f"   ⚠️ Tree error: {e}")
        print(f"✅ Done! File saved: {filename}")

    return {"roc_auc": roc_auc, "train_auc": train_auc, "overfit_gap": overfit_gap,
            "inner_val_auc": inner_val_auc, "overfit_gap_innerval": overfit_gap_innerval,
            "n_subtrain": n_subtrain, "n_subval": n_subval,
            "df_all": df_all, "df_res": df_res,
            "final_clf": final_clf, "donors": list(train_pat | test_pat),
            "df_scores": df_scores,
            "empirical_stability_threshold": empirical_thresh,
            "stability_threshold_used": effective_thresh,
            "outer_random_state": outer_random_state, "n_train": len(X_train_full), "n_test": len(X_test_full),
            "label_cutoff_low": low, "label_cutoff_high": high}


In [ ]:
# --- MULTI-SPLIT WRAPPER: repeats the outer train/test split to get a DISTRIBUTION of held-out AUCs ---
# A single 70/30 split on ~dozens of patients is a high-variance estimate. This reruns
# run_discovery_pdf_report() with several independent outer_random_state seeds and reports
# mean/std of test AUC, train AUC and the overfit gap, plus how often each gene is picked
# as "stable" across splits (a gene that's only stable in 1 of 10 splits is not robust).
def run_discovery_multi_split(
    model, adata, df_clinical,
    biomarker="PaceOfAgingP45", cell_type="Classical_monocyte",
    n_iterations=50, stability_thresh=60, n_top_genes=1000, target_group="ALL",
    exclude_patients=None,
    n_outer_splits=10,             # number of independent outer train/test splits
    outer_split_seeds=None,        # optional explicit seed list; default = range(n_outer_splits)
    outer_test_size=0.3,
    min_train=15, min_test=5,
    n_perm=0, n_iterations_perm=20,
    inner_val_size=0.3,            # passed through to run_discovery_pdf_report's inner validation split
    n_estimators=500,              # passed through -- tree count for the final/inner-val classifiers
    max_depth=4,                   # passed through -- max tree depth for the final/inner-val classifiers
    group_col=None,                # passed through -- stratify on this column instead of sex (see
                                    # run_discovery_pdf_report's docstring-comment for group_col)
    label_cutoffs="group",         # passed through -- "group" (old), "global", or a (low, high) tuple
                                    # (see run_discovery_pdf_report's comment for label_cutoffs)
    make_pdf_for_first=False,      # generate the full PDF/SHAP/tree report only for the first split
):
    seeds = list(outer_split_seeds) if outer_split_seeds is not None else list(range(n_outer_splits))
    rows = []
    gene_hits = {}   # gene -> number of splits where it landed in the top-25 "stable" table

    for k, seed in enumerate(seeds):
        res = run_discovery_pdf_report(
            model=model, adata=adata, df_clinical=df_clinical,
            biomarker=biomarker, cell_type=cell_type,
            n_iterations=n_iterations, stability_thresh=stability_thresh,
            n_top_genes=n_top_genes, target_group=target_group,
            exclude_patients=exclude_patients,
            make_pdf=(make_pdf_for_first and k == 0),
            min_auc=0.0,
            outer_test_size=outer_test_size,
            outer_random_state=seed,
            min_train=min_train, min_test=min_test,
            n_perm=n_perm, n_iterations_perm=n_iterations_perm,
            inner_val_size=inner_val_size,
            n_estimators=n_estimators, max_depth=max_depth,
            group_col=group_col,
            label_cutoffs=label_cutoffs,
        )
        if res is None:
            rows.append({"split": k, "seed": seed, "test_auc": np.nan, "train_auc": np.nan,
                         "inner_val_auc": np.nan, "overfit_gap": np.nan, "overfit_gap_innerval": np.nan,
                         "n_train": np.nan, "n_test": np.nan, "n_stable_genes": np.nan})
            continue
        rows.append({
            "split": k, "seed": seed,
            "test_auc": res["roc_auc"], "train_auc": res["train_auc"],
            "inner_val_auc": res["inner_val_auc"], "overfit_gap": res["overfit_gap"],
            "overfit_gap_innerval": res["overfit_gap_innerval"],
            "n_train": res["n_train"], "n_test": res["n_test"], "n_stable_genes": len(res["df_res"]),
        })
        for g in res["df_res"]["Gene"]:
            gene_hits[g] = gene_hits.get(g, 0) + 1

    df_splits = pd.DataFrame(rows)
    n_valid = int(df_splits["test_auc"].notna().sum())

    summary = {
        "cell_type": cell_type, "biomarker": biomarker, "target_group": target_group,
        "label_cutoffs": label_cutoffs,
        "n_splits_requested": len(seeds), "n_splits_valid": n_valid,
        "test_auc_mean": df_splits["test_auc"].mean(), "test_auc_std": df_splits["test_auc"].std(),
        "train_auc_mean": df_splits["train_auc"].mean(),
        "inner_val_auc_mean": df_splits["inner_val_auc"].mean(), "inner_val_auc_std": df_splits["inner_val_auc"].std(),
        "overfit_gap_mean": df_splits["overfit_gap"].mean(), "overfit_gap_std": df_splits["overfit_gap"].std(),
        "overfit_gap_innerval_mean": df_splits["overfit_gap_innerval"].mean(),
        "overfit_gap_innerval_std": df_splits["overfit_gap_innerval"].std(),
    }

    print(f"📊 {cell_type} | {biomarker} ({target_group}) over {n_valid}/{len(seeds)} valid splits:")
    print(f"   Test AUC       : {summary['test_auc_mean']:.3f} ± {summary['test_auc_std']:.3f}")
    print(f"   Inner-val AUC  : {summary['inner_val_auc_mean']:.3f} ± {summary['inner_val_auc_std']:.3f}  "
          f"(fresh model, held out of the 70% train donors)")
    print(f"   Train AUC      : {summary['train_auc_mean']:.3f}")
    print(f"   Overfit gap    : {summary['overfit_gap_mean']:+.3f} ± {summary['overfit_gap_std']:.3f}  "
          f"(train AUC minus TEST AUC; large positive = memorizing, not generalizing)")
    print(f"   Inner ovft gap : {summary['overfit_gap_innerval_mean']:+.3f} ± {summary['overfit_gap_innerval_std']:.3f}  "
          f"(train AUC minus INNER-VAL AUC)")

    genes_across_splits = (
        pd.Series(gene_hits, dtype=float)
          .sort_values(ascending=False)
          .rename("times_in_top25")
          .to_frame()
    )
    if len(seeds) > 0:
        genes_across_splits["pct_of_splits"] = genes_across_splits["times_in_top25"] / len(seeds) * 100

    return {"df_splits": df_splits, "summary": summary, "genes_across_splits": genes_across_splits}


In [ ]:
import os
# --- USAGE: ALL cell types, male vs female, several independent outer splits ---
# Cost note: this is n_outer_splits x len(my_valid_types) x 2 sexes full runs of
# run_discovery_pdf_report (each of which does its own n_iterations-round stability
# loop). Kept at 5 splits here (vs 10 for the single-cell-type MAIT check above) to
# bound total runtime -- bump it up once you know which cell types are worth a closer look.
N_OUTER_SPLITS_SWEEP = 5
OUTDIR_MULTI = f"MULTISPLIT_{BIOMARKER}"
os.makedirs(OUTDIR_MULTI, exist_ok=True)

all_rows = []
per_celltype_results = {}   # (cell_type, sex) -> full dict returned by run_discovery_multi_split

for cell_type in my_valid_types:
    for sex in ["M", "F"]:
        print(f"\n===== {cell_type} | sex={sex} =====")
        try:
            res = run_discovery_multi_split(
                model=model, adata=adata, df_clinical=df_pb_latent,
                biomarker=BIOMARKER, cell_type=cell_type,
                n_iterations=N_ITER_PDF, target_group=sex,
                n_outer_splits=N_OUTER_SPLITS_SWEEP,
            )
        except Exception as e:
            print(f"⚠️ CRASH on {cell_type} ({sex}): {e}")
            continue

        per_celltype_results[(cell_type, sex)] = res
        s = res["summary"]
        all_rows.append({
            "cell_type": cell_type, "sex": sex,
            "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
            "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
            "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
            "train_auc_mean": s["train_auc_mean"],
            "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
            "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
            "overfit_gap_innerval_std": s["overfit_gap_innerval_std"],
        })

df_all_celltypes = pd.DataFrame(all_rows).sort_values("test_auc_mean", ascending=False)
df_all_celltypes.to_csv(f"{OUTDIR_MULTI}/summary_all_celltypes_by_sex.csv", index=False, sep=';', decimal=',')
df_all_celltypes


In [ ]:
# --- HYPERPARAMETER SWEEP: n_estimators x max_depth for the FINAL / inner-val classifier ---
# Selection metric is INNER-VAL AUC, never test_auc -- tuning on the real held-out test set would
# quietly turn it into a second training set. test_auc_mean is carried along for reference only;
# once you've picked a hyperparameter region from inner_val_auc_mean, do one confirmatory
# run_discovery_multi_split() call at that setting and read ITS test_auc as the honest number.
#
# Cost note: the stability-selection step (n_iterations rounds) does NOT depend on n_estimators/
# max_depth and is redone at every single grid point anyway -- so this is
# len(N_ESTIMATORS_GRID) x len(MAX_DEPTH_GRID) x len(CELL_TYPES_FOR_TUNING) x len(SEXES_FOR_TUNING)
# x N_OUTER_SPLITS_TUNE full pipeline runs. Keep the grid / cell-type list / split count small here;
# widen only once a promising region shows up.

N_ESTIMATORS_GRID     = [100, 200, 300, 500, 800]
MAX_DEPTH_GRID         = [2, 3, 4, 6]
CELL_TYPES_FOR_TUNING  = ["B Naive"]     # <<< EDIT: which cell type(s) to tune on
SEXES_FOR_TUNING       = ["M"]              # <<< EDIT: which sex(es) to tune on
N_OUTER_SPLITS_TUNE    = 3                  # <<< fewer than the full sweep's 5, to bound runtime

hp_rows = []
hp_results = {}   # (cell_type, sex, n_estimators, max_depth) -> full run_discovery_multi_split() output

for cell_type in CELL_TYPES_FOR_TUNING:
    for sex in SEXES_FOR_TUNING:
        for n_est in N_ESTIMATORS_GRID:
            for depth in MAX_DEPTH_GRID:
                print(f"\n===== {cell_type} | sex={sex} | n_estimators={n_est} | max_depth={depth} =====")
                try:
                    res = run_discovery_multi_split(
                        model=model, adata=adata, df_clinical=df_pb_latent,
                        biomarker=BIOMARKER, cell_type=cell_type,
                        n_iterations=N_ITER_PDF, target_group=sex,
                        n_outer_splits=N_OUTER_SPLITS_TUNE,
                        n_estimators=n_est, max_depth=depth,
                    )
                except Exception as e:
                    print(f"⚠️ CRASH on {cell_type} ({sex}, n_est={n_est}, depth={depth}): {e}")
                    continue

                hp_results[(cell_type, sex, n_est, depth)] = res
                s = res["summary"]
                hp_rows.append({
                    "cell_type": cell_type, "sex": sex,
                    "n_estimators": n_est, "max_depth": depth,
                    "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
                    "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
                    "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
                    "train_auc_mean": s["train_auc_mean"],
                    "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],  # reference only, don't select on this
                })

df_hp_sweep = pd.DataFrame(hp_rows).sort_values("inner_val_auc_mean", ascending=False)
OUTDIR_HP = f"HPARAM_SWEEP_{BIOMARKER}"
os.makedirs(OUTDIR_HP, exist_ok=True)
hp_tag = f"{'_'.join(CELL_TYPES_FOR_TUNING)}_{'_'.join(SEXES_FOR_TUNING)}".replace(" ", "")
df_hp_sweep.to_csv(f"{OUTDIR_HP}/hparam_sweep_{hp_tag}.csv", index=False, sep=';', decimal=',')
df_hp_sweep


In [ ]:
# --- USAGE: ALL cell types, split by SMOKING STATUS (CurrentSmk26) instead of sex ---
# Same structure as the sex sweep above, but stratifying on group_col="CurrentSmk26" -- donors with
# CurrentSmk26==0.0 ("non-smoker at 26") vs ==1.0 ("smoker at 26"), each analyzed across ALL sexes
# pooled together (target_group is now being matched against CurrentSmk26's own values, not sex_mapped).
# Requires CurrentSmk26 to be in df_pb_latent -- i.e. you must have re-run the
# bio_indexed / df_pb_latent cells AFTER CurrentSmk26 was added to BIO_COLS in the GLOBAL CONFIG cell.
assert "CurrentSmk26" in df_pb_latent.columns, (
    "CurrentSmk26 missing from df_pb_latent -- re-run the GLOBAL CONFIG cell, then the "
    "bio_indexed = build_bio_indexed_from_blood(...) cell, then the df_pb_latent = ...join(...) cell."
)

SMOKING_LABELS = {0.0: "NonSmoker26", 1.0: "Smoker26"}
N_OUTER_SPLITS_SMOKING = 5
OUTDIR_SMOKING = f"MULTISPLIT_{BIOMARKER}_by_smoking"
os.makedirs(OUTDIR_SMOKING, exist_ok=True)

all_rows_smoking = []
per_celltype_results_smoking = {}   # (cell_type, smoking_value) -> full dict returned by run_discovery_multi_split

for cell_type in my_valid_types:
    for smk_value, smk_label in SMOKING_LABELS.items():
        print(f"\n===== {cell_type} | {smk_label} (CurrentSmk26={smk_value}) =====")
        try:
            res = run_discovery_multi_split(
                model=model, adata=adata, df_clinical=df_pb_latent,
                biomarker=BIOMARKER, cell_type=cell_type,
                n_iterations=N_ITER_PDF, target_group=smk_value,
                group_col="CurrentSmk26",
                n_outer_splits=N_OUTER_SPLITS_SMOKING,
            )
        except Exception as e:
            print(f"⚠️ CRASH on {cell_type} ({smk_label}): {e}")
            continue

        per_celltype_results_smoking[(cell_type, smk_label)] = res
        s = res["summary"]
        all_rows_smoking.append({
            "cell_type": cell_type, "smoking_group": smk_label,
            "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
            "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
            "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
            "train_auc_mean": s["train_auc_mean"],
            "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
            "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
            "overfit_gap_innerval_std": s["overfit_gap_innerval_std"],
        })

df_all_celltypes_smoking = pd.DataFrame(all_rows_smoking).sort_values("test_auc_mean", ascending=False)
df_all_celltypes_smoking.to_csv(f"{OUTDIR_SMOKING}/summary_all_celltypes_by_smoking.csv",
                                 index=False, sep=';', decimal=',')
df_all_celltypes_smoking


In [ ]:
import os, re
OUTDIR = f"FULL3_{BIOMARKER}_{TARGET_GROUP}"
os.makedirs(OUTDIR, exist_ok=True)

rows = []
for cell_type in my_valid_types:
    try:
        r = run_discovery_pdf_report(
            model=model, adata=adata, df_clinical=df_pb_latent,
            biomarker=BIOMARKER, cell_type=cell_type,
            n_iterations=N_ITER_PDF, target_group=TARGET_GROUP,
            make_pdf=True, min_auc=0.0)          # make_pdf=True -> writes the PDF/CSV report too
        if r is None:
            print(f"skip {cell_type}: not enough donors"); continue

        rows.append({
            "cell_type": cell_type, "n_donors": len(r["donors"]),
            "test_auc": r["roc_auc"], "train_auc": r["train_auc"],
            "overfit_gap": r["overfit_gap"], "n_stable_genes": len(r["df_res"]),
        })
    except Exception as e:
        print(f"⚠️ CRASH on {cell_type}: {e}"); continue

df_summary = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
df_summary.to_csv(f"{OUTDIR}/summary_all_celltypes.csv", index=False, sep=';', decimal=',')
df_summary

In [ ]:
import os, re
OUTDIR = f"JK_LOO2_{BIOMARKER}_{TARGET_GROUP}"
os.makedirs(OUTDIR, exist_ok=True)
SKIP={}   # already processed -> skip (case-insensitive)

for cell_type in my_valid_types:
    safe = re.sub(r'[\\/*?:\[\]]', '_', cell_type)
    auc_path    = f"{OUTDIR}/{safe}_round_AUCs.csv"
    gene_path   = f"{OUTDIR}/{safe}_round_genes.csv"
    scores_path = f"{OUTDIR}/{safe}_round_donor_scores.csv"   # <<< new output

    if cell_type.strip().lower() in SKIP:
        print(f"⏭️  skip {cell_type} (in SKIP list)"); continue
    if os.path.exists(auc_path):
        print(f"⏭️  skip {cell_type} (output already exists: {auc_path})"); continue

    try:
        # (a) one full run (no donor removed) to get the tested-donor list
        full = run_discovery_pdf_report(
            model=model, adata=adata, df_clinical=df_pb_latent,
            biomarker=BIOMARKER, cell_type=cell_type,
            n_iterations=N_ITER_PDF, target_group=TARGET_GROUP,
            make_pdf=False, min_auc=0.0)
        if full is None:
            print(f"skip {cell_type}: not enough donors"); continue
        donors = full["donors"]
        print(f"⚡️ {cell_type}: {len(donors)} donors -> {len(donors)} rounds")

        # (b) one round per donor: remove that donor, rerun, append results
        for k, d in enumerate(donors, start=1):
            r = run_discovery_pdf_report(
                model=model, adata=adata, df_clinical=df_pb_latent,
                biomarker=BIOMARKER, cell_type=cell_type,
                n_iterations=N_ITER_PDF, target_group=TARGET_GROUP,
                exclude_patients=[d], make_pdf=False, min_auc=0.0)
            if r is None:
                continue

            # AUC row
            pd.DataFrame([{"round": k, "removed_donor": d, "auc": r["roc_auc"],
                           "n_genes": len(r["df_all"])}]).to_csv(
                auc_path, mode="a", header=not os.path.exists(auc_path),
                index=False, sep=';', decimal=',')

            # full gene table
            g = r["df_all"].copy(); g.insert(0, "removed_donor", d); g.insert(0, "round", k)
            g.to_csv(gene_path, mode="a", header=not os.path.exists(gene_path),
                     index=False, sep=';', decimal=',')

            # NEW: per-donor category + classification score for this round
            s = r["df_scores"].copy(); s.insert(0, "removed_donor", d); s.insert(0, "round", k)
            s.to_csv(scores_path, mode="a", header=not os.path.exists(scores_path),
                     index=False, sep=';', decimal=',')

            if k % 10 == 0 or k == 1:
                print(f"   round {k}/{len(donors)} (drop {d}): AUC={r['roc_auc']:.3f}")

        print(f"   -> done {cell_type}: {len(donors)} rounds appended")

    except Exception as e:
        print(f"⚠️ CRASH on {cell_type}: {e}"); continue

print("\n🏁 ALL DONE.")

In [ ]:
 # --- 2. ANALYSIS FUNCTION (Silent & Fast mode) ---
def get_top_genes_for_type(model, adata, df_clinical, biomarker, cell_type, n_iterations=30):
    # Slightly fewer iterations (30) for speed, or keep 50 for max precision
    
    # --- PREPARATION ---
    pd.options.mode.chained_assignment = None 
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex): df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns: df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()
    sex_col = 'sex' if 'sex' in df_clin.columns else 'sex_patient'
    df_clin[sex_col] = df_clin[sex_col].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    target_sex = "2"
    df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])
    if len(df_male) < 10: target_sex = "M"; df_male = df_clin[df_clin[sex_col] == target_sex].drop_duplicates(subset=['patient_id'])

    high = df_male[biomarker].quantile(0.70); low = df_male[biomarker].quantile(0.30)
    df_male['Label'] = np.nan; df_male.loc[df_male[biomarker] >= high, 'Label'] = 1; df_male.loc[df_male[biomarker] <= low, 'Label'] = 0
    df_ml = df_male.dropna(subset=['Label'])
    
    if cell_type not in adata.obs['CellType'].values: return None
    adata_subset = adata[adata.obs['CellType'] == cell_type].copy()
    obs_pats = adata_subset.obs['patient_id'].astype(str).str.strip()
    adata_subset = adata_subset[obs_pats.isin(df_ml['patient_id'])].copy()
    
    if hasattr(adata_subset.X, "toarray"): means = np.array(adata_subset.X.mean(axis=0)).flatten()
    else: means = np.mean(adata_subset.X, axis=0)
    hvg = pd.Series(means, index=adata_subset.var_names).sort_values(ascending=False).head(1000).index.tolist()
    
    # Denoising
    X_denoised = model.get_normalized_expression(adata_subset, gene_list=hvg, library_size=1e4, return_numpy=True)
    df_expr = pd.DataFrame(X_denoised, columns=hvg)
    df_expr['Patient'] = adata_subset.obs['patient_id'].astype(str).str.strip().values
    X_full = df_expr.groupby('Patient').mean()
    common = X_full.index.intersection(df_ml['patient_id'])
    X_full = X_full.loc[common]; y_full = df_ml.set_index('patient_id').loc[common]['Label']
    if len(X_full) < 20: return None

    # --- ROBUSTNESS ---
    gene_counts = pd.Series(0, index=hvg)
    for i in range(n_iterations):
        seed = 42 + i
        X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.3, random_state=seed, stratify=y_full)
        clf = XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.6, device="cuda", tree_method="hist", eval_metric='logloss', random_state=seed)
        clf.fit(X_train, y_train)
        imps = pd.Series(clf.feature_importances_, index=X_full.columns)
        gene_counts[imps[imps > 0].index] += 1
    
    stability_pct = (gene_counts / n_iterations) * 100
    stable_genes = stability_pct[stability_pct >= 50].index.tolist() # Keep everything > 50%
    if len(stable_genes) < 2: stable_genes = stability_pct.sort_values(ascending=False).head(10).index.tolist()
    
    # --- FINAL IMPORTANCE ---
    X_stable = X_full[stable_genes]
    X_train, X_test, y_train, y_test = train_test_split(X_stable, y_full, test_size=0.3, random_state=42, stratify=y_full)
    final_clf = XGBClassifier(n_estimators=500, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, device="cuda", tree_method="hist", eval_metric='logloss', random_state=42)
    final_clf.fit(X_train, y_train)
    
    # Build the clean DataFrame
    df_res = pd.DataFrame({
        "Gene": stable_genes,
        "Stability (%)": stability_pct[stable_genes].values,
        "Importance_XGB": final_clf.feature_importances_
    }).sort_values("Stability (%)", ascending=False)
    
    return df_res

In [ ]:
!pip install openpyxl

In [ ]:
# --- 3. RUN & SAVE TO EXCEL ---
filename_excel = EXCEL_FILE
print(f"🚀 Starting compilation to {filename_excel}...")

with pd.ExcelWriter(filename_excel, engine='openpyxl') as writer:
    processed_count = 0
    
    for cell_type in tqdm(my_valid_types):
        try:
            df_genes = get_top_genes_for_type(
                model=model, 
                adata=adata, 
                df_clinical=df_pb_latent, 
                biomarker=BIOMARKER, 
                cell_type=cell_type,
                n_iterations=N_ITER_EXCEL
            )
            
            if df_genes is not None:
                # 1. CLEAN CHARACTERS FORBIDDEN BY EXCEL
                # Replace ?, /, *, [, ], : with an underscore "_"
                clean_name = re.sub(r'[\\/*?:\[\]]', '_', cell_type)
                
                # 2. 31-CHARACTER LIMIT (Excel rejects longer names)
                sheet_name = clean_name[:31]
                
                # Save
                df_genes.to_excel(writer, sheet_name=sheet_name, index=False)
                processed_count += 1
                
        except Exception as e:
            print(f"⚠️ Error on {cell_type}: {e}")

print(f"\n✅ Done! {processed_count} cell types saved to '{filename_excel}'.")
print("You can download this file from the left-hand menu.")

In [ ]:
# --- 1. PARAMETERS ---
file_path = EXCEL_FILE
target_keyword = "MAIT" 

print(f"🔍 MERGE DIAGNOSTIC FOR: {target_keyword}")

try:
    # --- 2. FETCH GENES ---
    xls = pd.ExcelFile(file_path)
    found_sheet = None
    for sheet in xls.sheet_names:
        if sheet.startswith(target_keyword):
            found_sheet = sheet; break
            
    if found_sheet is None: raise ValueError("Sheet not found.")
    
    df_genes = pd.read_excel(file_path, sheet_name=found_sheet)
    top_genes_list = df_genes["Gene"].head(5).tolist()
    print(f"🧬 Genes: {top_genes_list}")

    # --- 3. RIGOROUS ID CLEANUP (DUPLICATE FIX) ---
    print("\n🧹 Cleaning patient identifiers...")
    
    # A. CLINICAL-side cleanup
    df_clin = df_pb_latent.copy()
    
    # Handle index/column
    if df_clin.index.name != 'patient_id' and 'patient_id' not in df_clin.columns:
        df_clin.index.name = 'patient_id'
        df_clin = df_clin.reset_index()
    elif 'patient_id' not in df_clin.columns:
        df_clin['patient_id'] = df_clin.index.astype(str)
        
    # Build clean ID
    df_clin['patient_id_clean'] = df_clin['patient_id'].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    # --- THE FIX IS HERE: DROP DUPLICATES ---
    # Keep only one row per patient to avoid the "Ambiguous Series" error
    df_clin = df_clin.drop_duplicates(subset=['patient_id_clean'])
    
    df_clin = df_clin.set_index('patient_id_clean') 
    
    # B. GENOMIC-side cleanup (adata)
    full_name_scanpy = "B_Cells_memory_MS4A1_TNFRSF13B"
    adata_subset = adata[adata.obs['CellType'] == full_name_scanpy].copy()
    print(f"   -> Raw cells found: {len(adata_subset)}")
    
    adata_subset.obs['patient_id_clean'] = adata_subset.obs['patient_id'].astype(str).str.replace(".0", "", regex=False).str.strip()

    # --- 4. COMPUTE GROUPS ---
    # Recompute thresholds on the cleaned df_clin
    high = df_clin[BIOMARKER].quantile(0.70)
    low = df_clin[BIOMARKER].quantile(0.30)
    
    def get_status(pat_id):
        # If the ID is not in the clinical file, exit
        if pat_id not in df_clin.index: return "Unknown"
        
        # Here, thanks to drop_duplicates, 'val' is guaranteed to be a single number
        val = df_clin.loc[pat_id, BIOMARKER]
        
        if val >= high: return "High"
        elif val <= low: return "Low"
        return "Middle"

    adata_subset.obs['Status'] = adata_subset.obs['patient_id_clean'].map(get_status)
    
    # --- 5. CHECK BEFORE PLOTTING ---
    counts = adata_subset.obs['Status'].value_counts()
    print("\n📊 Cell distribution after mapping:")
    print(counts)
    
    # Filtre final
    adata_final = adata_subset[adata_subset.obs['Status'].isin(["High", "Low"])]
    
    if len(adata_final) == 0:
        print("\n❌ ERROR: 0 cells remaining.")
        print("   Check whether you have many 'Unknown' above.")
    else:
        print(f"\n✅ Success! {len(adata_final)} cells ready.")
        sc.pl.dotplot(
            adata_final, 
            var_names=top_genes_list, 
            groupby="Status", 
            standard_scale='var',
            title=f"Signature: {full_name_scanpy}",
            cmap="Reds",
            swap_axes=True
        )

except Exception as e:
    print(f"\n🐛 BUG: {e}")

In [ ]:
# --- 1. PARAMETERS ---
file_path = EXCEL_FILE
target_keyword = "B_Cells_memory"

print(f"🔍 HARDENED MERGE DIAGNOSTIC FOR: {target_keyword}")

try:
    # --- 2. FETCH GENES ---
    xls = pd.ExcelFile(file_path)
    found_sheet = None
    for sheet in xls.sheet_names:
        if sheet.startswith(target_keyword):
            found_sheet = sheet; break
            
    if found_sheet is None: raise ValueError("Sheet not found.")
    
    df_genes = pd.read_excel(file_path, sheet_name=found_sheet)
    top_genes_list = df_genes["Gene"].head(5).tolist()
    print(f"🧬 Genes: {top_genes_list}")

    # --- 3. BUILD THE MAPPING DICTIONARY (THE SOLUTION) ---
    print("\n📦 Building the Patient -> biomarker dictionary...")
    
    # Prepare the clinical DataFrame
    df_clin = df_pb_latent.copy()
    
    # Make sure we have a 'patient_id' column
    if df_clin.index.name != 'patient_id' and 'patient_id' not in df_clin.columns:
        df_clin = df_clin.reset_index()
        if 'index' in df_clin.columns: df_clin = df_clin.rename(columns={'index': 'patient_id'})
    elif 'patient_id' not in df_clin.columns:
        df_clin['patient_id'] = df_clin.index.astype(str)
        
    # Clean the IDs
    df_clin['patient_id_clean'] = df_clin['patient_id'].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    # ⚡️ MAGIC HERE: convert to a dictionary ⚡️
    # This forces uniqueness (on duplicate, last wins) and simplifies everything
    urea_map = df_clin.set_index('patient_id_clean')[BIOMARKER].to_dict()
    
    print(f"   -> Dictionary ready with {len(urea_map)} unique patients.")

    # --- 4. SCANPY PREPARATION ---
    full_name_scanpy = "B_Cells_memory_MS4A1_TNFRSF13B"
    adata_subset = adata[adata.obs['CellType'] == full_name_scanpy].copy()
    
    # Clean Scanpy IDs
    adata_subset.obs['patient_id_clean'] = adata_subset.obs['patient_id'].astype(str).str.replace(".0", "", regex=False).str.strip()

    # --- 5. COMPUTE GROUPS VIA DICTIONARY ---
    # Compute thresholds on the dictionary values
    all_values = pd.Series(list(urea_map.values()))
    high = all_values.quantile(0.70)
    low = all_values.quantile(0.30)
    
    def get_status_from_dict(pat_id):
        # Ultra-fast and safe lookup in the dictionary
        if pat_id not in urea_map: return "Unknown"
        
        val = urea_map[pat_id] # Guaranteed to be a single number (float)
        
        if val >= high: return "High"
        elif val <= low: return "Low"
        return "Middle"

    adata_subset.obs['Status'] = adata_subset.obs['patient_id_clean'].map(get_status_from_dict)
    
    # --- 6. CHECK & PLOT ---
    counts = adata_subset.obs['Status'].value_counts()
    print("\n📊 Distribution:")
    print(counts)
    
    adata_final = adata_subset[adata_subset.obs['Status'].isin(["High", "Low"])]
    
    if len(adata_final) == 0:
        print("\n❌ ERROR: 0 cells found. Check your IDs.")
    else:
        print(f"\n✅ Success! {len(adata_final)} cells ready.")
        sc.pl.dotplot(
            adata_final, 
            var_names=top_genes_list, 
            groupby="Status", 
            standard_scale='var',
            title=f"Signature: {full_name_scanpy}",
            cmap="Reds",
            swap_axes=True
        )

except Exception as e:
    print(f"\n🐛 BUG AGAIN?: {e}")


In [ ]:
# --- 1. PARAMETERS ---
target_keyword = "CD4_mature_th2"          # To search in Excel
full_name_scanpy = "CD4_mature_th2?_CCR4" # To search in Scanpy

print(f"🔍 CD4 Th2 DIAGNOSTIC FOR: {full_name_scanpy}")

try:
    # --- 2. FETCH GENES (EXCEL) ---
    file_path = EXCEL_FILE
    xls = pd.ExcelFile(file_path)
    found_sheet = None
    
    # Flexible sheet search (handles special characters)
    for sheet in xls.sheet_names:
        if sheet.startswith(target_keyword) or sheet.replace("_", "") == target_keyword.replace("_", ""):
            found_sheet = sheet; break
            
    if found_sheet is None: 
        if "CD4_mature_th2_CCR4" in xls.sheet_names: found_sheet = "CD4_mature_th2_CCR4"
    
    if found_sheet is None: raise ValueError("Sheet not found in Excel.")
    
    print(f"   -> Excel sheet found: '{found_sheet}'")
    df_genes = pd.read_excel(file_path, sheet_name=found_sheet)
    top_genes_list = df_genes["Gene"].head(5).tolist()
    print(f"🧬 Top 5 Genes: {top_genes_list}")

    # --- 3. DATA PREPARATION (ROBUST INDEX-FREE METHOD) ---
    df_clin = df_pb_latent.copy()
    
    # Reset the index to avoid any conflict
    df_clin = df_clin.reset_index()
    
    # Identify the ID column
    col_id = 'patient_id'
    if col_id not in df_clin.columns:
        if 'index' in df_clin.columns: col_id = 'index'
        else: col_id = df_clin.columns[0]
            
    # Clean the IDs
    df_clin['temp_id_clean'] = df_clin[col_id].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    # BUILD THE DICTIONARY (the foolproof method)
    keys = df_clin['temp_id_clean'].values
    values = df_clin[BIOMARKER].values
    urea_map = dict(zip(keys, values))
    
    # Seuils
    vals_series = pd.Series(list(urea_map.values()))
    high = vals_series.quantile(0.70)
    low = vals_series.quantile(0.30)
    
    # --- 4. SCANPY MAPPING ---
    adata_subset = adata[adata.obs['CellType'] == full_name_scanpy].copy()
    adata_subset.obs['patient_id_clean'] = adata_subset.obs['patient_id'].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    def get_status(pat):
        if pat not in urea_map: return "Unknown"
        val = urea_map[pat]
        if val >= high: return "High"
        elif val <= low: return "Low"
        return "Middle"

    # BUG FIX HERE: added (get_status)
    adata_subset.obs['Status'] = adata_subset.obs['patient_id_clean'].map(get_status)
    
    adata_final = adata_subset[adata_subset.obs['Status'].isin(["High", "Low"])]
    
    print(f"✅ {len(adata_final)} cells ready.")

    # --- 5. PLOT ---
    sc.pl.dotplot(
        adata_final, 
        var_names=top_genes_list, 
        groupby="Status", 
        standard_scale='var',
        title=f"CD4 Th2 Signature (Massive: {len(adata_final)} cells)",
        cmap="Reds",
         swap_axes=True
    )

except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# --- Low/Medium/High tertile categories for age-26 continuous traits ---
# Pulled straight from `blood` (not through BIO_COLS) so each trait keeps its own ~89-96%
# completeness instead of all 9 traits being required non-null simultaneously (that would
# drop the cohort from 807 to 698 donors -- checked before doing this).
TRAITS_FOR_TERTILE = ["bmi26", "choles26", "trigly26", "hdl26", "urea26",
                      "VOmax26", "MeanArtPr26", "LnCRP26", "crp26ns","cortis26np"]
TRAITS_FOR_TERTILE = ["hdl26"]

trait_indexed = (
    blood[["replicate", *TRAITS_FOR_TERTILE]]
    .drop_duplicates(subset="replicate")
    .set_index("replicate")
)

for t in TRAITS_FOR_TERTILE:
    cat_col = f"{t}_cat"
    try:
        trait_indexed[cat_col] = pd.qcut(
            trait_indexed[t], q=3, labels=["Low", "Medium", "High"], duplicates="drop"
        )
    except ValueError as e:
        print(f"⚠️ Could not tertile-split {t}: {e}")
        continue
    vc = trait_indexed[cat_col].value_counts(dropna=False)
    n_non_null = trait_indexed[t].notna().sum()
    print(f"{t}: {n_non_null} donors with a value -> {dict(vc)}")

cat_cols = [f"{t}_cat" for t in TRAITS_FOR_TERTILE if f"{t}_cat" in trait_indexed.columns]
df_pb_latent = df_pb_latent.join(trait_indexed[cat_cols], on="patient_id", how="left")
df_pb_latent[cat_cols].notna().sum()


In [ ]:
import os
# --- USAGE: ALL cell types, split by Low/Medium/High tertile of each age-26 trait ---
# Same run_discovery_multi_split(group_col=...) mechanism as the smoking sweep above, but looped
# over 9 traits x 3 tertile levels x cell types. This is BOTH sexes pooled within each tertile
# (group_col points at the trait's *_cat column, not sex_mapped) -- restrict SEXES manually via
# group_col2 if you want sex-specific tertiles (not wired up here, ask if you want it added).
#
# Cost note: this is len(TRAITS_FOR_TERTILE) x 3 levels x len(CELL_TYPES_FOR_TERTILE_SWEEP)
# x N_OUTER_SPLITS_TERTILE full pipeline runs -- with all 9 traits x all cell types that's 9 x 3 x 10
# x N_OUTER_SPLITS_TERTILE = a lot. Defaulted to a short cell-type list and 3 splits to keep a first
# pass tractable; widen CELL_TYPES_FOR_TERTILE_SWEEP / N_OUTER_SPLITS_TERTILE once you see which
# traits look promising.
assert all(f"{t}_cat" in df_pb_latent.columns for t in TRAITS_FOR_TERTILE), (
    "Tertile category columns missing from df_pb_latent -- run the tertile-category cell above first."
)

CELL_TYPES_FOR_TERTILE_SWEEP = my_valid_types   # <<< EDIT: shrink this list to bound runtime, e.g.
                                                 #     ["CD4 Active", "B Naive", "B Memory"]
N_OUTER_SPLITS_TERTILE = 3                      # <<< fewer than the main sweep's 5, to bound runtime
TERTILE_LEVELS = ["Low", "Medium", "High"]

OUTDIR_TERTILE = f"MULTISPLIT_{BIOMARKER}_by_tertile"
os.makedirs(OUTDIR_TERTILE, exist_ok=True)

all_rows_tertile = []
per_celltype_results_tertile = {}   # (cell_type, trait, level) -> full dict returned by run_discovery_multi_split

for trait in TRAITS_FOR_TERTILE:
    cat_col = f"{trait}_cat"
    if cat_col not in df_pb_latent.columns:
        continue
    for cell_type in CELL_TYPES_FOR_TERTILE_SWEEP:
        for level in TERTILE_LEVELS:
            print(f"\n===== {cell_type} | {trait} = {level} =====")
            try:
                res = run_discovery_multi_split(
                    model=model, adata=adata, df_clinical=df_pb_latent,
                    biomarker=BIOMARKER, cell_type=cell_type,
                    n_iterations=N_ITER_PDF, target_group=level,
                    group_col=cat_col,
                    n_outer_splits=N_OUTER_SPLITS_TERTILE,
                )
            except Exception as e:
                print(f"⚠️ CRASH on {cell_type} ({trait}={level}): {e}")
                continue

            per_celltype_results_tertile[(cell_type, trait, level)] = res
            s = res["summary"]
            all_rows_tertile.append({
                "cell_type": cell_type, "trait": trait, "tertile": level,
                "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
                "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
                "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
                "train_auc_mean": s["train_auc_mean"],
                "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
                "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
                "overfit_gap_innerval_std": s["overfit_gap_innerval_std"],
            })

df_all_celltypes_tertile = pd.DataFrame(all_rows_tertile).sort_values("test_auc_mean", ascending=False)
df_all_celltypes_tertile.to_csv(f"{OUTDIR_TERTILE}/summary_all_celltypes_by_tertile.csv",
                                 index=False, sep=';', decimal=',')
df_all_celltypes_tertile


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

CSV_PATH = "/home/nlapique/Blood/script/MULTISPLIT_POA45_nohdl_by_tertile/summary_all_celltypes_by_tertile.csv"
OUT_PATH = "/home/nlapique/Blood/script/MULTISPLIT_POA45_nohdl_by_tertile/diagnostic_overview_sorted.png"

TERTILE_ORDER = ["Low", "Medium", "High"]
TERTILE_COLORS = {"Low": "#9ecae1", "Medium": "#4292c6", "High": "#08519c"}

# --- ordering anchors: rows sorted by AUC in the CD8 column, columns sorted by AUC in the hdl row ---
SORT_CELL_TYPE = "CD8 Active"   # rows (traits) ordered by this column's AUC, best on top
SORT_TRAIT     = "hdl26"        # columns (cell types) ordered by this row's AUC, best on the left

# --- color scale: everything below RANDOM_MAX is treated as chance (white),
#     above it goes light red -> dark red up to AUC_MAX ---
RANDOM_MAX = 0.57   # the permutation null for CD8 Active/hdl26/Low had 95th pct = 0.617
AUC_MAX    = 0.75

# the sweep CSV on disk is semicolon-separated with decimal commas
df = pd.read_csv(CSV_PATH, sep=";", decimal=",")

cmap = mcolors.LinearSegmentedColormap.from_list("auc_reds", ["#fde0dd", "#fb6a4a", "#67000d"])
cmap.set_under("white")   # AUC < RANDOM_MAX -> white
norm = mcolors.Normalize(vmin=RANDOM_MAX, vmax=AUC_MAX)

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 3, height_ratios=[1.3, 1], hspace=0.6, wspace=0.15)

# one shared row order for all panels: hdl on top, then the other traits by their
# CD8 Active AUC in the Low panel (where the hdl26 hit is)
low = df[df["tertile"] == "Low"].pivot(index="trait", columns="cell_type", values="test_auc_mean")
rest = low[SORT_CELL_TYPE].drop(SORT_TRAIT).sort_values(ascending=False, na_position="last").index
ROW_ORDER = [SORT_TRAIT, *rest]

heat_axes = []
for i, tertile in enumerate(TERTILE_ORDER):
    ax = fig.add_subplot(gs[0, i])
    heat_axes.append(ax)
    mat = df[df["tertile"] == tertile].pivot(index="trait", columns="cell_type", values="test_auc_mean")

    # columns: each panel sorted by its own hdl26 AUC; rows: shared ROW_ORDER
    col_order = mat.loc[SORT_TRAIT].sort_values(ascending=False, na_position="last").index
    mat = mat.reindex(index=ROW_ORDER, columns=col_order)

    im = ax.imshow(mat.values, cmap=cmap, norm=norm, aspect="auto")

    # AUC value in each cell
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            v = mat.values[r, c]
            if np.isnan(v):
                continue
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if v > 0.72 else ("#999999" if v < RANDOM_MAX else "black"))

    # outline the anchor column and row
    c_idx = list(mat.columns).index(SORT_CELL_TYPE)
    r_idx = list(mat.index).index(SORT_TRAIT)
    ax.add_patch(plt.Rectangle((c_idx - 0.5, -0.5), 1, mat.shape[0], fill=False, ec="black", lw=1.5))
    ax.add_patch(plt.Rectangle((-0.5, r_idx - 0.5), mat.shape[1], 1, fill=False, ec="black", lw=1.5))

    ax.set_title(f"Donors in {tertile} tertile of each trait", fontsize=11,
                 color=TERTILE_COLORS[tertile], fontweight="bold")
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index if i == 0 else [], fontsize=8)   # same rows in every panel
    ax.set_xticks(np.arange(-0.5, mat.shape[1]), minor=True)   # light grid so white cells stay visible
    ax.set_yticks(np.arange(-0.5, mat.shape[0]), minor=True)
    ax.grid(which="minor", color="#dddddd", linewidth=0.5)
    ax.tick_params(which="minor", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

cbar = fig.colorbar(im, ax=heat_axes, shrink=0.8, pad=0.01, extend="min",
                    label=f"test AUC (white = below {RANDOM_MAX:.2f}, treated as chance)")

# --- bottom panel unchanged: inner-val vs test AUC ---
ax2 = fig.add_subplot(gs[1, :])
lims = (0.25, 1.0)
ax2.plot(lims, lims, color="#999999", linewidth=1, linestyle="--", zorder=1)
ax2.text(0.97, 0.96, "y = x (inner-val predicts test)", transform=ax2.transAxes, ha="right", fontsize=8,
         color="#999999")

for tertile in TERTILE_ORDER:
    sub = df[df["tertile"] == tertile]
    ax2.scatter(sub["inner_val_auc_mean"], sub["test_auc_mean"], color=TERTILE_COLORS[tertile],
                label=tertile, s=40, alpha=0.85, edgecolors="white", linewidths=0.5, zorder=2)

ax2.axhline(0.5, color="#b2182b", linewidth=1, linestyle=":", zorder=1)
ax2.text(0.255, 0.51, "chance (test AUC = 0.5)", fontsize=8, color="#b2182b")

corr = df["inner_val_auc_mean"].corr(df["test_auc_mean"])
best = df.loc[df["test_auc_mean"].idxmax()]
worst = df.loc[df["test_auc_mean"].idxmin()]
for row in (best, worst):
    label = f"{row['trait']} / {row['cell_type']} / {row['tertile']}"
    ax2.annotate(label, (row["inner_val_auc_mean"], row["test_auc_mean"]), textcoords="offset points",
                 xytext=(6, 6), fontsize=7)

ax2.set_xlim(*lims); ax2.set_ylim(*lims)
ax2.set_xlabel("inner-CV validation AUC (used to pick/tune the model)")
ax2.set_ylabel("held-out test AUC")
ax2.set_title(f"Inner-val AUC barely predicts test AUC (r = {corr:.2f})", fontsize=11)
ax2.legend(title="tertile", frameon=False, loc="lower right")
for spine in ("top", "right"):
    ax2.spines[spine].set_visible(False)

n_ct, n_tr = df["cell_type"].nunique(), df["trait"].nunique()
fig.suptitle(f"MULTISPLIT_PaceOfAgingP45_by_tertile: {n_ct} cell types x {n_tr} traits x 3 tertiles, 3 splits each\n"
             f"rows: {SORT_TRAIT} on top, then by {SORT_CELL_TYPE} AUC in Low panel (shared) | columns: by {SORT_TRAIT} AUC (per panel)",
             fontsize=13, fontweight="bold", y=0.99)

fig.savefig(OUT_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_PATH}")
print(f"corr(inner_val_auc_mean, test_auc_mean) = {corr:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

CSV_PATH = "/home/nlapique/Blood/script/MULTISPLIT_PaceOfAgingP45_by_tertile/summary_all_celltypes_by_tertile.csv"
OUT_PATH = "/home/nlapique/Blood/script/MULTISPLIT_PaceOfAgingP45_by_tertile/diagnostic_overview.png"

TERTILE_ORDER = ["Low", "Medium", "High"]
TERTILE_COLORS = {"Low": "#9ecae1", "Medium": "#4292c6", "High": "#08519c"}

df = pd.read_csv(CSV_PATH, sep=";", decimal=",")

cell_types = sorted(df["cell_type"].unique())
traits = sorted(df["trait"].unique())

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 3, height_ratios=[1.3, 1], hspace=0.55, wspace=0.15)

cmap = mcolors.LinearSegmentedColormap.from_list("auc_diverging", ["#2166ac", "#f0f0f0", "#b2182b"])
norm = mcolors.TwoSlopeNorm(vmin=0.3, vcenter=0.5, vmax=0.85)

heat_axes = []
for i, tertile in enumerate(TERTILE_ORDER):
    ax = fig.add_subplot(gs[0, i])
    heat_axes.append(ax)
    sub = df[df["tertile"] == tertile]
    mat = sub.pivot(index="trait", columns="cell_type", values="test_auc_mean")
    mat = mat.reindex(index=traits, columns=cell_types)
    im = ax.imshow(mat.values, cmap=cmap, norm=norm, aspect="auto")
    ax.set_title(f"{tertile} PaceOfAgingP45 tertile", fontsize=11, color=TERTILE_COLORS[tertile], fontweight="bold")
    ax.set_xticks(range(len(cell_types)))
    ax.set_xticklabels(cell_types, rotation=90, fontsize=7)
    if i == 0:
        ax.set_yticks(range(len(traits)))
        ax.set_yticklabels(traits, fontsize=8)
    else:
        ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

cbar = fig.colorbar(im, ax=heat_axes, shrink=0.8, pad=0.01, label="test AUC (0.5 = chance)")
cbar.ax.axhline(0.5, color="black", linewidth=1)

ax2 = fig.add_subplot(gs[1, :])
lims = (0.25, 1.0)
ax2.plot(lims, lims, color="#999999", linewidth=1, linestyle="--", zorder=1)
ax2.text(0.97, 0.96, "y = x (inner-val predicts test)", transform=ax2.transAxes, ha="right", fontsize=8, color="#999999")

for tertile in TERTILE_ORDER:
    sub = df[df["tertile"] == tertile]
    ax2.scatter(sub["inner_val_auc_mean"], sub["test_auc_mean"], color=TERTILE_COLORS[tertile],
                label=tertile, s=40, alpha=0.85, edgecolors="white", linewidths=0.5, zorder=2)

ax2.axhline(0.5, color="#b2182b", linewidth=1, linestyle=":", zorder=1)
ax2.text(0.255, 0.51, "chance (test AUC = 0.5)", fontsize=8, color="#b2182b")

corr = df["inner_val_auc_mean"].corr(df["test_auc_mean"])
best = df.loc[df["test_auc_mean"].idxmax()]
worst = df.loc[df["test_auc_mean"].idxmin()]
for row in (best, worst):
    label = f"{row['trait']} / {row['cell_type']} / {row['tertile']}"
    ax2.annotate(label, (row["inner_val_auc_mean"], row["test_auc_mean"]), textcoords="offset points", xytext=(6, 6), fontsize=7)

ax2.set_xlim(*lims); ax2.set_ylim(*lims)
ax2.set_xlabel("inner-CV validation AUC (used to pick/tune the model)")
ax2.set_ylabel("held-out test AUC")
ax2.set_title(f"Inner-val AUC barely predicts test AUC (r = {corr:.2f})", fontsize=11)
ax2.legend(title="tertile", frameon=False, loc="lower right")
for spine in ("top", "right"):
    ax2.spines[spine].set_visible(False)

fig.suptitle("MULTISPLIT_PaceOfAgingP45_by_tertile: 9 cell types x 10 traits x 3 tertiles, 3 splits each",
             fontsize=13, fontweight="bold", y=0.98)

fig.savefig(OUT_PATH, dpi=200, bbox_inches="tight")
print(f"Saved: {OUT_PATH}")
print(f"corr(inner_val_auc_mean, test_auc_mean) = {corr:.3f}")

## Focused validation: CD8 Active x hdl26 x Low (top hit from the 243-combo tertile sweep)

`MULTISPLIT_PaceOfAgingP45_by_tertile/summary_all_celltypes_by_tertile.csv` tested 243
(cell_type x trait x tertile) combinations at only 3 outer splits each; the best one
(CD8 Active / hdl26 / Low, test_auc_mean=0.848) sits right where the *maximum* of 243 noisy
draws is expected to land even with zero real signal (empirical mean/median across all 243
combos is ~0.54-0.55, essentially chance). The cells below do two checks before trusting it:

1. Rerun just this one combo with more outer splits (10) and actually look at n_train/n_test
   per split -- the summary CSV never recorded sample size, so there's currently no way to tell
   from the file whether 0.848 came from a reasonably-sized group or from something near the
   `min_train=15` floor.
2. A label-permutation test: shuffle PaceOfAgingP45 among the eligible (CD8 Active, HDL-Low)
   donors, rerun the *exact same* pipeline (gene-stability selection + final XGB, held-out test
   AUC), repeat many times, and see how often that null process reaches 0.848 on its own.

Caveat: this only tests whether this one combo beats chance on its own. It does **not** correct
for having tried 243 combinations -- for that, compare the resulting p-value against a stricter
threshold (~0.05/243 = 0.0002), not the usual 0.05, or treat this as a first-pass filter before a
full family-wise permutation (max-of-243-under-permutation), which would cost ~243x more compute.

In [ ]:
# --- STEP 1: rerun CD8 Active x hdl26 x Low with more outer splits, and inspect n_train/n_test ---
# Costs 10 full run_discovery_pdf_report() calls (not the full 9-trait x 3-tertile x N-celltype sweep).
assert "hdl26_cat" in df_pb_latent.columns, "Run the tertile-category cell (TRAITS_FOR_TERTILE) first."

FOCUS_CELL_TYPE = "CD8 Active"
FOCUS_TRAIT     = "hdl26"
FOCUS_LEVEL     = "Low"
FOCUS_CAT_COL   = f"{FOCUS_TRAIT}_cat"
N_OUTER_SPLITS_FOCUS = 10   # vs. 3 in the original tertile sweep -> tighter mean/std estimate

focus_res = run_discovery_multi_split(
    model=model, adata=adata, df_clinical=df_pb_latent,
    biomarker=BIOMARKER, cell_type=FOCUS_CELL_TYPE,
    n_iterations=N_ITER_PDF, target_group=FOCUS_LEVEL,
    group_col=FOCUS_CAT_COL,
    n_outer_splits=N_OUTER_SPLITS_FOCUS,
)

print(focus_res["summary"])
display(focus_res["df_splits"][["split", "seed", "n_train", "n_test", "test_auc", "train_auc", "overfit_gap"]])


In [ ]:
# --- STEP 2: label-permutation null for one (cell_type, group_col, target_group) combo ---
# Shuffles the CONTINUOUS PaceOfAgingP45 value among the eligible donors (same group_col/
# target_group filter run_discovery_pdf_report applies internally -- i.e. CD8 Active donors in the
# HDL-Low tertile), then lets run_discovery_pdf_report re-derive High/Low labels from the SAME
# quantile_low/quantile_high cutoffs and rerun the exact same pipeline (gene-stability selection +
# final XGB, scored on a held-out outer split). This kills any true biomarker<->expression
# relationship while preserving sample size, class balance, and every other pipeline detail, so the
# resulting test AUCs are a fair empirical null for "what test AUC do you get from this much data
# and this pipeline with NO real signal."
#
# Cost: n_perm x n_outer_splits_per_perm full run_discovery_pdf_report() calls (each does its own
# n_iterations-round stability loop). Start small (n_perm=50-100) to gauge runtime before scaling up.
import time


def run_label_permutation_test(
    model, adata, df_clinical, biomarker, cell_type,
    group_col, target_group,
    observed_stat,                  # the real test_auc_mean being tested, e.g. 0.8478
    n_perm=100,
    n_outer_splits_per_perm=3,      # match N_OUTER_SPLITS_TERTILE so the null statistic (mean AUC
                                     # over k splits) is comparable to observed_stat
    n_iterations=20,                # fewer than N_ITER_PDF (50) to bound runtime for the null loop
    n_estimators=500, max_depth=4,  # keep these at the real pipeline's defaults
    rng_seed=0,
    verbose_every=10,
):
    rng = np.random.default_rng(rng_seed)

    # prep ONE dataframe (index/patient_id normalized exactly like run_discovery_pdf_report does),
    # and reuse it for both the eligibility mask and every permuted copy -- avoids index mismatches
    # between a freshly-reset frame and the raw df_clinical.
    df_clin = df_clinical.copy()
    if isinstance(df_clin.index, pd.MultiIndex):
        df_clin = df_clin.reset_index()
    if 'patient_id' not in df_clin.columns:
        df_clin['patient_id'] = df_clin.index.astype(str)
    df_clin['patient_id'] = df_clin['patient_id'].astype(str).str.strip()

    eligible_mask = df_clin[group_col] == target_group
    eligible_idx = df_clin.index[eligible_mask]
    print(f"-> {len(eligible_idx)} donors eligible for {cell_type} | {group_col}={target_group}")

    null_stats = []
    t0 = time.time()
    for p in range(n_perm):
        df_perm = df_clin.copy()
        shuffled_vals = df_perm.loc[eligible_idx, biomarker].sample(
            frac=1.0, random_state=int(rng.integers(0, 2**31 - 1))
        ).values
        df_perm.loc[eligible_idx, biomarker] = shuffled_vals

        split_aucs = []
        for k in range(n_outer_splits_per_perm):
            seed = 10_000 + p * 100 + k
            r = run_discovery_pdf_report(
                model=model, adata=adata, df_clinical=df_perm,
                biomarker=biomarker, cell_type=cell_type,
                n_iterations=n_iterations, target_group=target_group,
                group_col=group_col, make_pdf=False, min_auc=0.0,
                outer_random_state=seed,
                n_estimators=n_estimators, max_depth=max_depth,
            )
            if r is not None:
                split_aucs.append(r["roc_auc"])
        if split_aucs:
            null_stats.append(float(np.mean(split_aucs)))

        if (p + 1) % verbose_every == 0:
            elapsed = time.time() - t0
            remaining = elapsed / (p + 1) * (n_perm - p - 1)
            print(f"   permutation {p+1}/{n_perm} done ({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")

    null_stats = np.array(null_stats)
    n_valid = len(null_stats)
    n_exceed = int(np.sum(null_stats >= observed_stat))
    p_value = (1 + n_exceed) / (1 + n_valid)

    print(f"\n📊 Null distribution ({n_valid}/{n_perm} valid permutations):")
    print(f"   mean={null_stats.mean():.3f}  std={null_stats.std():.3f}  "
          f"max={null_stats.max():.3f}  95th pct={np.percentile(null_stats, 95):.3f}")
    print(f"   Observed statistic: {observed_stat:.3f}")
    print(f"   Empirical p-value (P[null >= observed]): {p_value:.4f}  "
          f"({n_exceed}/{n_valid} null draws >= observed)")

    return {"null_stats": null_stats, "observed_stat": observed_stat, "p_value": p_value,
            "n_exceed": n_exceed, "n_valid": n_valid}


In [ ]:
# --- STEP 3: run the permutation test ---
# 0.8478260869565218 = test_auc_mean for CD8 Active / hdl26 / Low from the original 3-split sweep
# (summary_all_celltypes_by_tertile.csv). Start with n_perm=100 to gauge runtime, then scale up
# (500-1000) if this looks worth pinning down more precisely -- remember the effective bar given
# 243 combos were tried is closer to p < 0.05/243 = 0.0002, not p < 0.05.
PERM_OBSERVED_STAT = 0.752760524499655
N_PERM = 100

FOCUS_CELL_TYPE = "NK cells"
FOCUS_TRAIT     = "hdl26"
FOCUS_LEVEL     = "Low"
FOCUS_CAT_COL   = f"{FOCUS_TRAIT}_cat"



perm_result = run_label_permutation_test(
    model=model, adata=adata, df_clinical=df_pb_latent,
    biomarker=BIOMARKER, cell_type=FOCUS_CELL_TYPE,
    group_col=FOCUS_CAT_COL, target_group=FOCUS_LEVEL,
    observed_stat=PERM_OBSERVED_STAT,
    n_perm=N_PERM,
    n_outer_splits_per_perm=3,   # matches N_OUTER_SPLITS_TERTILE used for the observed stat
    n_iterations=20,
)


In [ ]:
# --- STEP 4: plot the null distribution against the observed statistic ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(perm_result["null_stats"], bins=20, color="#888888", alpha=0.8, label="Permutation null")
ax.axvline(perm_result["observed_stat"], color="#d62728", lw=2,
           label=f"Observed (mean over 3 real splits) = {perm_result['observed_stat']:.3f}")
ax.set_xlabel("Mean test AUC over outer splits")
ax.set_ylabel("Count")
ax.set_title(f"Permutation null: {FOCUS_CELL_TYPE} x {FOCUS_TRAIT}={FOCUS_LEVEL}\n"
             f"p = {perm_result['p_value']:.4f}  (n_perm={perm_result['n_valid']})")
ax.legend()
plt.tight_layout()
plt.savefig(f"permtest_{FOCUS_CELL_TYPE}_{FOCUS_TRAIT}_{FOCUS_LEVEL}.png".replace(" ", "_"), dpi=150)
plt.show()


## Donors with consistently low HDL (Low tertile at 26, 32, 38 AND 45)
Restricts the multi-split AUC analysis to donors whose HDL was in the Low tertile at every age, instead of at 26 only.

In [ ]:
# --- Donors CONSISTENTLY in the Low HDL tertile at ages 26, 32, 38 AND 45 ---
# Tertiles are computed separately at each age, on the donors in `blood` (same convention as the
# TRAITS_FOR_TERTILE cell above). A donor is "ConsistentLow" only if they are in the Low tertile at
# ALL four ages. Donors with HDL at all four ages but not always Low are "Other"; donors missing HDL
# at any age stay NaN, so they are never selected by run_discovery_pdf_report's group filter.
HDL_COLS = ["hdl26", "HDL32", "HDL38", "HDL45"]

hdl_indexed = (
    blood[["replicate", *HDL_COLS]]
    .drop_duplicates(subset="replicate")
    .set_index("replicate")
)
hdl_tertiles = hdl_indexed.apply(lambda s: pd.qcut(s, q=3, labels=["Low", "Medium", "High"]))
has_all_ages = hdl_indexed[HDL_COLS].notna().all(axis=1)
all_low = (hdl_tertiles == "Low").all(axis=1)

hdl_indexed["hdl_consistent"] = pd.Series(np.nan, index=hdl_indexed.index, dtype=object)
hdl_indexed.loc[has_all_ages, "hdl_consistent"] = "Other"
hdl_indexed.loc[all_low, "hdl_consistent"] = "ConsistentLow"

print(f"{has_all_ages.sum()} donors with HDL at all 4 ages -> {all_low.sum()} consistently in the Low tertile")
for c in HDL_COLS:
    print(f"   {c}: Low tertile = HDL <= {hdl_indexed.loc[hdl_tertiles[c] == 'Low', c].max():.2f}")

# how many High / Low pace donors each label_cutoffs mode would give (before per-cell-type filtering)
pace = (blood.drop_duplicates(subset="replicate").set_index("replicate")[BIOMARKER]
        .reindex(hdl_indexed.index))
pace_sel = pace[all_low].dropna()
for mode, ref in [("global", pace.dropna()), ("group", pace_sel)]:
    lo, hi = ref.quantile(quantile_low), ref.quantile(quantile_high)
    print(f"   label_cutoffs={mode!r}: High >= {hi:.3f} -> {(pace_sel >= hi).sum()} donors, "
          f"Low <= {lo:.3f} -> {(pace_sel <= lo).sum()} donors")

df_pb_latent = (
    df_pb_latent.drop(columns=["hdl_consistent"], errors="ignore")
    .join(hdl_indexed[["hdl_consistent"]], on="patient_id", how="left")
)
df_pb_latent["hdl_consistent"].value_counts(dropna=False)


In [ ]:
import os
# --- USAGE: ALL cell types, donors with consistently Low HDL at 26/32/38/45 ---
# Same run_discovery_multi_split(group_col=...) mechanism as the tertile sweep, restricted to the
# "ConsistentLow" donors (both sexes pooled). Only ~120 donors qualify, so two settings differ from
# the tertile sweep:
#  - label_cutoffs="global": High/Low pace is defined on the WHOLE cohort (>= 90th / <= 70th pct), so
#    "High" means the same as for everyone else. With "group" (old behavior) the cutoffs would be
#    re-derived inside these ~120 donors -> only ~12 High donors in total, ~4 per test set.
#  - 10 outer splits instead of 3: each test set holds only a handful of High donors, so a single
#    split's AUC is very noisy.
# Cost: len(my_valid_types) x N_OUTER_SPLITS_HDL_CONSISTENT full pipeline runs.
assert "hdl_consistent" in df_pb_latent.columns, (
    "hdl_consistent missing from df_pb_latent -- run the consistently-low-HDL selection cell above first."
)

N_OUTER_SPLITS_HDL_CONSISTENT = 10
HDL_LABEL_CUTOFFS = "global"      # <<< "global" (recommended here) or "group" (old behavior)

OUTDIR_HDL_CONSISTENT = f"MULTISPLIT_{BIOMARKER}_hdl_consistent_low_{HDL_LABEL_CUTOFFS}"
os.makedirs(OUTDIR_HDL_CONSISTENT, exist_ok=True)

all_rows_hdl_consistent = []
per_celltype_results_hdl_consistent = {}   # cell_type -> full dict returned by run_discovery_multi_split

for cell_type in my_valid_types:
    print(f"\n===== {cell_type} | HDL consistently Low (26/32/38/45) =====")
    try:
        res = run_discovery_multi_split(
            model=model, adata=adata, df_clinical=df_pb_latent,
            biomarker=BIOMARKER, cell_type=cell_type,
            n_iterations=N_ITER_PDF, target_group="ConsistentLow",
            group_col="hdl_consistent",
            n_outer_splits=N_OUTER_SPLITS_HDL_CONSISTENT,
            label_cutoffs=HDL_LABEL_CUTOFFS,
        )
    except Exception as e:
        print(f"⚠️ CRASH on {cell_type}: {e}")
        continue

    per_celltype_results_hdl_consistent[cell_type] = res
    s, splits = res["summary"], res["df_splits"]
    all_rows_hdl_consistent.append({
        "cell_type": cell_type, "label_cutoffs": HDL_LABEL_CUTOFFS,
        "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
        "n_train": splits["n_train"].median(), "n_test": splits["n_test"].median(),
        "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
        "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
        "train_auc_mean": s["train_auc_mean"],
        "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
    })

df_hdl_consistent = pd.DataFrame(all_rows_hdl_consistent).sort_values("test_auc_mean", ascending=False)
df_hdl_consistent.to_csv(f"{OUTDIR_HDL_CONSISTENT}/summary_all_celltypes_hdl_consistent_low.csv",
                         index=False, sep=';', decimal=',')
df_hdl_consistent


## Three-model comparison: does snRNA-seq add to age-26 clinical traits?

The cells above train one classifier per cell type (or per tertile), each on a High/Low cut of pace.
These cells instead fit **one model per donor set** on the **continuous** pace value, with every
cell type in the same row, and compare four feature sets on the **same donors and the same folds**:

| model | features |
|---|---|
| `0_base` | sex + donor-mean `QC_MT.pct` (the floor) |
| `1_clinical` | base + age-26 clinical traits |
| `2_expression` | base + snRNA-seq features (composition + scVI latent) |
| `3_clinical+expression` | base + both |

The key number is **3 minus 1**: what the snRNA-seq adds beyond a routine age-26 clinical panel.

- Model: elastic net (standardized features; alpha and l1_ratio tuned by an inner 5-fold CV on the training donors only).
- Evaluation: repeated 5-fold CV over **donors** (one row per donor), so no donor is ever in both train and test.
- Metrics: test R², Pearson/Spearman r, and an AUC for High vs Low pace using the `quantile_low`/`quantile_high`
  cutoffs from the GLOBAL CONFIG cell (computed on each training fold), comparable with the earlier AUCs.
- Model differences are tested with the Nadeau–Bengio corrected t-test, because CV folds overlap and a plain t-test is too optimistic.

**Caveat:** several clinical predictors (HDL, BMI, cholesterol, triglycerides, CRP, VO2max, blood pressure, urea, WBC, HbA1c, cortisol)
are pace *components* measured at the start of the 26→45 slope. Measurement noise alone couples a baseline value to its slope
(regression to the mean), so part of `1_clinical`'s score is not biology. `POA45_nohdl` (pace without HDL) is run alongside as a check;
add other `POA45_no*` columns to `TARGETS_3M` to check other components.

In [ ]:
# --- THREE-MODEL COMPARISON, STEP 1: one row per donor, three feature blocks ---
# Needs: blood (cell 8), adata with obsm["X_scvi_corrected"] (cells 36-38), df_pb_latent (cell 48),
# BIOMARKER / quantile_low / quantile_high (GLOBAL CONFIG).
import re
import numpy as np
import pandas as pd

# continuous targets; POA45_nohdl = pace without its HDL component. dict.fromkeys drops duplicates
# (e.g. when BIOMARKER is itself "POA45_nohdl"), which would otherwise make Y_3M[target] two columns.
TARGETS_3M = list(dict.fromkeys([BIOMARKER, "PaceOfAgingP45"]))

# Age-26 predictors only: blood was drawn at 26 and pace is the 26->45 slope, so anything measured
# at 32/38/45 (or fit over 26-45) is future information and is refused below.
CLINICAL_COLS_3M = ["bmi26", "choles26", "trigly26", "hdl26", "urea26", "VOmax26", "MeanArtPr26",
                    "LnCRP26", "wbc26", "glyhaem26", "cortis26np", "CurrentSmk26", "PackYrsBy26"]

# snRNA-seq blocks (use any subset):
#   "composition"  -> CLR of cell-type proportions, from all of the donor's cells
#   "latent_all"   -> donor mean of X_scvi_corrected over all cells
#   "latent_by_ct" -> per-cell-type mean latent from df_pb_latent, for cell types present (>= 50 cells)
#                     in at least MIN_CT_COVERAGE of donors; absent cell types are imputed + flagged
EXPR_BLOCKS_3M = ["composition", "latent_all", "latent_by_ct"]
MIN_CT_COVERAGE = 0.5

SEX_MAP = {'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}

_future = [c for c in CLINICAL_COLS_3M if re.search(r"(32|38|45)$|_iz_|POA|Pace", c)]
assert not _future, f"Not age-26 predictors, remove them: {_future}"
for _lst in (TARGETS_3M, CLINICAL_COLS_3M):
    _miss = [c for c in _lst if c not in blood.columns]
    if _miss:
        print(f"⚠️ not in blood, dropped: {_miss}")
        _lst[:] = [c for c in _lst if c in blood.columns]

# --- donor-level clinical table (traits are constant within a donor; QC_MT.pct is per cell -> mean) ---
_rep = blood["replicate"].astype(str).str.strip()
donor_blood = blood.assign(replicate=_rep).drop_duplicates(subset="replicate").set_index("replicate")
qc_mt_mean = blood["QC_MT.pct"].groupby(_rep).mean()

_sex = (donor_blood["sex"].astype(str).str.replace(".0", "", regex=False).str.strip().map(SEX_MAP))
X_base = pd.DataFrame({"sex_M": (_sex == "M").astype(float).where(_sex.notna()),
                       "QC_MT_mean": qc_mt_mean.reindex(donor_blood.index)})
X_clinical = donor_blood[CLINICAL_COLS_3M].apply(pd.to_numeric, errors="coerce")
Y_3M = donor_blood[TARGETS_3M].apply(pd.to_numeric, errors="coerce")

# --- snRNA-seq blocks ---
_obs = pd.DataFrame({"patient_id": adata.obs["patient_id"].astype(str).str.strip().values,
                     "CellType": adata.obs["CellType"].astype(str).values})
expr_parts = []
if "composition" in EXPR_BLOCKS_3M:
    _counts = pd.crosstab(_obs["patient_id"], _obs["CellType"])
    _logp = np.log(_counts + 0.5)                       # pseudocount for cell types a donor lacks
    expr_parts.append(_logp.sub(_logp.mean(axis=1), axis=0).add_prefix("clr|"))
if "latent_all" in EXPR_BLOCKS_3M:
    _lat = pd.DataFrame(np.asarray(adata.obsm["X_scvi_corrected"]), index=_obs["patient_id"].values)
    expr_parts.append(_lat.groupby(level=0).mean().add_prefix("latall|"))
if "latent_by_ct" in EXPR_BLOCKS_3M:
    _pb = df_pb_latent
    if not isinstance(_pb.index, pd.MultiIndex):
        _pb = _pb.set_index(["patient_id", "CellType"])
    _lat_cols = [c for c in _pb.columns if c.startswith("latent_")]
    _wide = _pb[_lat_cols].unstack("CellType")          # columns: (latent_i, CellType)
    _wide.index = _wide.index.astype(str).str.strip()
    _present = _wide.xs(_lat_cols[0], axis=1, level=0).notna()
    _coverage = _present.reindex(_counts.index if "composition" in EXPR_BLOCKS_3M else _wide.index,
                                 fill_value=False).mean().sort_values(ascending=False)
    _keep_ct = list(_coverage[_coverage >= MIN_CT_COVERAGE].index)
    print("cell-type coverage (fraction of donors with >= 50 cells):")
    print(_coverage.round(2).to_string())
    print(f"-> latent_by_ct uses {len(_keep_ct)} cell types: {_keep_ct}")
    if _keep_ct:
        _w = _wide.loc[:, _wide.columns.get_level_values(1).isin(_keep_ct)]
        _w.columns = [f"{ct}|{lat}" for lat, ct in _w.columns]
        _has = _present[_keep_ct].astype(float).add_prefix("has|")
        expr_parts += [_w, _has]
X_expression = pd.concat(expr_parts, axis=1)

# donors with snRNA-seq data (the composition/latent_all index covers every donor in adata)
donors_sn = X_expression.index[X_expression.notna().any(axis=1)]
X_expression = X_expression.loc[donors_sn]
if "has|" in "".join(X_expression.columns):
    _has_cols = [c for c in X_expression.columns if c.startswith("has|")]
    X_expression[_has_cols] = X_expression[_has_cols].fillna(0.0)

BLOCKS_3M = {"base": X_base, "clinical": X_clinical, "expression": X_expression}
print(f"\n{len(donors_sn)} donors with snRNA-seq | features: base={X_base.shape[1]}, "
      f"clinical={X_clinical.shape[1]}, expression={X_expression.shape[1]}")
for t in TARGETS_3M:
    n = Y_3M[t].reindex(donors_sn).notna().sum()
    print(f"   {t}: {n} donors with snRNA-seq and a value")

In [ ]:
# --- THREE-MODEL COMPARISON, STEP 2: repeated donor-level CV, same folds for every model ---
import os
from scipy import stats
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score, roc_auc_score

N_FOLDS_3M, N_REPEATS_3M = 5, 5   # 25 test folds per model and target
N_JOBS_3M = 4                     # parallel jobs inside ElasticNetCV's inner CV
MODELS_3M = {
    "0_base":                ["base"],
    "1_clinical":            ["base", "clinical"],
    "2_expression":          ["base", "expression"],
    "3_clinical+expression": ["base", "clinical", "expression"],
}
COMPARISONS_3M = [("1_clinical", "0_base"), ("2_expression", "0_base"),
                  ("3_clinical+expression", "1_clinical"), ("3_clinical+expression", "2_expression")]
OUTDIR_3M = f"THREE_MODEL_{BIOMARKER}"
os.makedirs(OUTDIR_3M, exist_ok=True)


def _make_model_3m():
    # imputation and scaling are fit on the training fold only (they sit inside the pipeline)
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], cv=5, max_iter=20000, n_jobs=N_JOBS_3M),
    )


def corrected_ttest(d, n_train, n_test):
    """Nadeau & Bengio (2003) corrected repeated-CV t-test on per-fold differences d."""
    d = np.asarray(d, dtype=float)
    J = len(d)
    var = np.var(d, ddof=1) * (1.0 / J + n_test / n_train)
    t = d.mean() / np.sqrt(var) if var > 0 else np.nan
    return t, 2 * stats.t.sf(abs(t), J - 1)


fold_rows, coef_rows, oof_rows = [], [], []
for target in TARGETS_3M:
    y_all = Y_3M[target].reindex(donors_sn)
    donors = y_all.index[y_all.notna()]
    y = y_all.loc[donors].values
    X_by_model = {m: pd.concat([BLOCKS_3M[b].reindex(donors) for b in blocks], axis=1)
                  for m, blocks in MODELS_3M.items()}
    print(f"\n===== {target}: {len(donors)} donors =====")

    rkf = RepeatedKFold(n_splits=N_FOLDS_3M, n_repeats=N_REPEATS_3M, random_state=0)
    for fold, (tr, te) in enumerate(rkf.split(donors)):
        lo, hi = np.quantile(y[tr], quantile_low), np.quantile(y[tr], quantile_high)
        ext = (y[te] <= lo) | (y[te] >= hi)          # test donors that get a High/Low label
        for m, X in X_by_model.items():
            pipe = _make_model_3m().fit(X.iloc[tr], y[tr])
            pred = pipe.predict(X.iloc[te])
            auc = (roc_auc_score(y[te][ext] >= hi, pred[ext])
                   if len(np.unique(y[te][ext] >= hi)) == 2 else np.nan)
            fold_rows.append({
                "target": target, "model": m, "fold": fold, "repeat": fold // N_FOLDS_3M,
                "n_train": len(tr), "n_test": len(te), "n_features": X.shape[1],
                "test_r2": r2_score(y[te], pred),
                "test_pearson": stats.pearsonr(y[te], pred)[0] if np.std(pred) > 0 else np.nan,
                "test_spearman": stats.spearmanr(y[te], pred)[0] if np.std(pred) > 0 else np.nan,
                "test_auc_high_vs_low": auc,
                "train_r2": r2_score(y[tr], pipe.predict(X.iloc[tr])),
                "n_nonzero_coef": int((pipe[-1].coef_ != 0).sum()),
            })
            if fold < N_FOLDS_3M:                    # first repeat = one out-of-fold prediction per donor
                oof_rows += [{"target": target, "model": m, "patient_id": p, "observed": o, "predicted": q}
                             for p, o, q in zip(donors[te], y[te], pred)]
            if m == "3_clinical+expression":
                coef_rows += [{"target": target, "fold": fold, "feature": f, "coef": c}
                              for f, c in zip(X.columns, pipe[-1].coef_)]
        print(f"   fold {fold + 1}/{N_FOLDS_3M * N_REPEATS_3M} done", end="\r")
    print()

df_folds_3m = pd.DataFrame(fold_rows)
metrics = ["test_r2", "test_pearson", "test_spearman", "test_auc_high_vs_low", "train_r2", "n_nonzero_coef"]
df_summary_3m = (df_folds_3m.groupby(["target", "model"])[metrics].agg(["mean", "std"]).round(3))
df_summary_3m.columns = [f"{a}_{b}" for a, b in df_summary_3m.columns]
df_summary_3m = df_summary_3m.reset_index()

comp_rows = []
for target in TARGETS_3M:
    sub = df_folds_3m[df_folds_3m["target"] == target]
    n_tr, n_te = sub["n_train"].mean(), sub["n_test"].mean()
    for a, b in COMPARISONS_3M:
        for metric in ("test_r2", "test_spearman"):
            d = (sub[sub["model"] == a].set_index("fold")[metric]
                 - sub[sub["model"] == b].set_index("fold")[metric]).dropna()
            t, p = corrected_ttest(d, n_tr, n_te)
            comp_rows.append({"target": target, "comparison": f"{a} - {b}", "metric": metric,
                              "mean_diff": d.mean(), "sd_diff": d.std(), "frac_folds_better": (d > 0).mean(),
                              "corrected_t": t, "p_corrected": p})
df_comp_3m = pd.DataFrame(comp_rows)

df_coef_3m = (pd.DataFrame(coef_rows).groupby(["target", "feature"])["coef"]
              .agg(mean_coef="mean", frac_nonzero=lambda c: (c != 0).mean()).reset_index())
df_coef_3m["abs_mean_coef"] = df_coef_3m["mean_coef"].abs()
df_coef_3m = df_coef_3m.sort_values(["target", "abs_mean_coef"], ascending=[True, False])
df_oof_3m = pd.DataFrame(oof_rows)

for name, d in [("per_fold", df_folds_3m), ("summary", df_summary_3m), ("comparisons", df_comp_3m),
                ("coefficients_clinical+expression", df_coef_3m), ("oof_predictions_repeat0", df_oof_3m)]:
    d.to_csv(f"{OUTDIR_3M}/{name}.csv", index=False, sep=';', decimal=',')

print(df_summary_3m[["target", "model", "test_r2_mean", "test_r2_std", "test_spearman_mean",
                     "test_auc_high_vs_low_mean", "train_r2_mean"]].to_string(index=False))
print()
print(df_comp_3m.round(4).to_string(index=False))
print("\nTop features of 3_clinical+expression (standardized coef, mean over folds):")
for target in TARGETS_3M:
    print(f"--- {target}")
    print(df_coef_3m[df_coef_3m["target"] == target].head(15)[["feature", "mean_coef", "frac_nonzero"]]
          .round(3).to_string(index=False))

In [ ]:
# --- THREE-MODEL COMPARISON, STEP 3: plot test R² per fold for each model ---
import matplotlib.pyplot as plt

model_order = list(MODELS_3M)
model_labels = ["base\n(sex, QC)", "clinical\n(age 26)", "snRNA-seq", "clinical +\nsnRNA-seq"]
fig, axes = plt.subplots(1, len(TARGETS_3M), figsize=(5.5 * len(TARGETS_3M), 4.8), sharey=True, squeeze=False)
rng_jit = np.random.default_rng(0)

for ax, target in zip(axes[0], TARGETS_3M):
    sub = df_folds_3m[df_folds_3m["target"] == target]
    wide = sub.pivot(index="fold", columns="model", values="test_r2")[model_order]
    for fold_vals in wide.values:                     # thin grey line per fold: same donors across models
        ax.plot(range(len(model_order)), fold_vals, color="#d0d0d0", linewidth=0.6, zorder=1)
    for k, m in enumerate(model_order):
        v = wide[m].values
        ax.scatter(k + rng_jit.uniform(-0.08, 0.08, len(v)), v, s=14, color="#6f8fb3", alpha=0.8,
                   edgecolors="white", linewidths=0.4, zorder=2)
        ax.errorbar(k, v.mean(), yerr=v.std(), fmt="o", color="#1f3b63", markersize=8,
                    capsize=4, linewidth=2, zorder=3)
        ax.text(k + 0.18, v.mean(), f"{v.mean():.3f}", va="center", fontsize=9, color="#333333")
    ax.axhline(0, color="#999999", linestyle="--", linewidth=1, zorder=0)
    c = df_comp_3m[(df_comp_3m["target"] == target) & (df_comp_3m["metric"] == "test_r2")
                   & (df_comp_3m["comparison"] == "3_clinical+expression - 1_clinical")].iloc[0]
    ax.set_title(f"{target}\nadding snRNA-seq to clinical: ΔR² = {c['mean_diff']:+.3f} "
                 f"(p = {c['p_corrected']:.2g})", fontsize=10)
    ax.set_xticks(range(len(model_order)))
    ax.set_xticklabels(model_labels, fontsize=9)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0][0].set_ylabel("held-out test R² (one dot per CV fold)")
axes[0][0].text(0.01, 0.01, "dashed line: R² = 0, no better than predicting the mean",
                transform=axes[0][0].transAxes, fontsize=7.5, color="#777777")
fig.suptitle(f"Predicting pace of aging from age-26 data: {N_REPEATS_3M}x{N_FOLDS_3M}-fold donor-level CV, "
             f"elastic net", fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTDIR_3M}/three_model_r2.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{OUTDIR_3M}/three_model_r2.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUTDIR_3M}/three_model_r2.png (+ .pdf, and the CSVs above)")

## Two-stage cascade: clinical classifier first, snRNA-seq XGBoost for the donors it is unsure about

Pace is split into High (≥ `quantile_high`) vs Low (≤ `quantile_low`), with cutoffs from each training fold and the middle donors left out, as in the earlier classifiers.

1. **Stage 1:** a logistic regression on the age-26 clinical traits (+ sex, QC), with balanced class weights so that 0.5 means "no idea".
2. **Uncertain band:** a donor is uncertain when the stage-1 probability is close to 0.5. The width of the band is set on the **training** donors' out-of-fold probabilities: the `UNCERTAIN_FRAC` least confident training donors. The same width is applied to test donors. The true label is never used to pick who goes to stage 2; a new donor wouldn't have one.
3. **Stage 2:** XGBoost on the snRNA-seq block from Step 1 (composition + scVI latent), trained on the training donors in the band (`STAGE2_TRAIN_ON="band"`) or on all training donors (`"all"`, more data but not specialized). It scores only the test donors in the band.

**The honest comparison is inside the band:** on the same uncertain test donors, is the stage-2 AUC higher than the stage-1 AUC? A pooled "cascade" AUC over all donors is also reported. It mixes scores from two different models, though, so use it as a secondary number.

Needs Steps 1–2 above to have run (`BLOCKS_3M`, `Y_3M`, `donors_sn`, `TARGETS_3M`, `corrected_ttest`).

In [ ]:
# --- CASCADE, STEP 4: clinical logistic regression -> snRNA-seq XGBoost on the uncertain band ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier

UNCERTAIN_FRAC = 0.4        # fraction of (training) donors, least confident at stage 1, sent to stage 2
STAGE2_TRAIN_ON = "band"    # "band" = train XGBoost only on uncertain training donors; "all" = on all of them
STAGE2_BLOCK = "expression"  # snRNA-seq features for stage 2: "expression" (Step 1 latent) or "pb_genes" (Step 6)
N_FOLDS_CAS, N_REPEATS_CAS = 5, 5


def _make_stage1():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000))


def _make_stage2(y_train):
    # shallow, heavily regularized trees: only a few hundred training donors in the band
    n_pos = max(int(y_train.sum()), 1)
    return XGBClassifier(n_estimators=300, max_depth=2, learning_rate=0.03, subsample=0.8,
                         colsample_bytree=0.5, min_child_weight=5, reg_lambda=5.0,
                         scale_pos_weight=(len(y_train) - n_pos) / n_pos,
                         eval_metric="logloss", n_jobs=N_JOBS_3M, random_state=0)


def _auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan


X_s1_all = pd.concat([BLOCKS_3M["base"], BLOCKS_3M["clinical"]], axis=1)
X_s2_all = BLOCKS_3M[STAGE2_BLOCK]            # XGBoost handles the NaNs of absent cell types itself

cas_rows = []
for target in TARGETS_3M:
    y_cont = Y_3M[target].reindex(donors_sn).dropna()
    donors = y_cont.index
    print(f"\n===== {target}: {len(donors)} donors =====")
    rkf = RepeatedKFold(n_splits=N_FOLDS_CAS, n_repeats=N_REPEATS_CAS, random_state=0)
    for fold, (tr, te) in enumerate(rkf.split(donors)):
        # High/Low labels from TRAINING-fold cutoffs; middle donors are dropped from both sides
        lo, hi = y_cont.iloc[tr].quantile(quantile_low), y_cont.iloc[tr].quantile(quantile_high)
        def _lab(idx):
            v = y_cont.iloc[idx]
            keep = (v <= lo) | (v >= hi)
            return donors[idx][keep.values], (v[keep] >= hi).astype(int).values
        d_tr, y_tr = _lab(tr)
        d_te, y_te = _lab(te)

        # stage 1 + band width from out-of-fold training probabilities (no test labels involved)
        s1 = _make_stage1()
        p_oof = cross_val_predict(s1, X_s1_all.loc[d_tr], y_tr, method="predict_proba",
                                  cv=StratifiedKFold(5, shuffle=True, random_state=fold))[:, 1]
        width = np.quantile(np.abs(p_oof - 0.5), UNCERTAIN_FRAC)
        s1.fit(X_s1_all.loc[d_tr], y_tr)
        p1_te = s1.predict_proba(X_s1_all.loc[d_te])[:, 1]

        # stage 2
        tr2 = np.abs(p_oof - 0.5) <= width if STAGE2_TRAIN_ON == "band" else np.ones(len(d_tr), bool)
        if len(np.unique(y_tr[tr2])) < 2:
            continue
        s2 = _make_stage2(y_tr[tr2]).fit(X_s2_all.loc[d_tr[tr2]], y_tr[tr2])
        band_te = np.abs(p1_te - 0.5) <= width
        p2_band = s2.predict_proba(X_s2_all.loc[d_te[band_te]])[:, 1] if band_te.any() else np.array([])

        p_cascade = p1_te.copy()
        p_cascade[band_te] = p2_band
        cas_rows.append({
            "target": target, "fold": fold, "n_test": len(d_te), "n_test_high": int(y_te.sum()),
            "n_train_stage2": int(tr2.sum()), "n_train_stage2_high": int(y_tr[tr2].sum()),
            "n_band_test": int(band_te.sum()), "n_band_test_high": int(y_te[band_te].sum()),
            "auc_clinical_all": _auc(y_te, p1_te),
            "auc_cascade_all": _auc(y_te, p_cascade),
            "auc_clinical_band": _auc(y_te[band_te], p1_te[band_te]),
            "auc_snrna_band": _auc(y_te[band_te], p2_band),
            "auc_clinical_confident": _auc(y_te[~band_te], p1_te[~band_te]),
        })
        print(f"   fold {fold + 1}/{N_FOLDS_CAS * N_REPEATS_CAS} done", end="\r")
    print()

df_cas = pd.DataFrame(cas_rows)
auc_cols = ["auc_clinical_all", "auc_cascade_all", "auc_clinical_band", "auc_snrna_band", "auc_clinical_confident"]
n_cols = ["n_test", "n_test_high", "n_train_stage2", "n_train_stage2_high", "n_band_test", "n_band_test_high"]
df_cas_summary = df_cas.groupby("target")[auc_cols + n_cols].agg(["mean", "std"]).round(3)
df_cas_summary.columns = [f"{a}_{b}" for a, b in df_cas_summary.columns]

cas_comp = []
for target, sub in df_cas.groupby("target"):
    n_tr = (sub["n_test"] * (N_FOLDS_CAS - 1)).mean()
    for label, a, b in [("band: snRNA XGB - clinical", "auc_snrna_band", "auc_clinical_band"),
                        ("band: snRNA XGB - chance (0.5)", "auc_snrna_band", None),
                        ("all: cascade - clinical", "auc_cascade_all", "auc_clinical_all")]:
        d = (sub[a] - (sub[b] if b else 0.5)).dropna()
        t, p = corrected_ttest(d, n_tr, sub["n_test"].mean())
        cas_comp.append({"target": target, "comparison": label, "mean_diff": d.mean(),
                         "frac_folds_better": (d > 0).mean(), "corrected_t": t, "p_corrected": p})
df_cas_comp = pd.DataFrame(cas_comp)

tag = f"cascade_band{UNCERTAIN_FRAC}_{STAGE2_TRAIN_ON}" + ("" if STAGE2_BLOCK == "expression" else f"_{STAGE2_BLOCK}")
df_cas.to_csv(f"{OUTDIR_3M}/{tag}_per_fold.csv", index=False, sep=';', decimal=',')
df_cas_summary.to_csv(f"{OUTDIR_3M}/{tag}_summary.csv", sep=';', decimal=',')
df_cas_comp.to_csv(f"{OUTDIR_3M}/{tag}_comparisons.csv", index=False, sep=';', decimal=',')

print(df_cas_summary.T.to_string())
print()
print(df_cas_comp.round(4).to_string(index=False))

In [ ]:
# --- CASCADE, STEP 5: plot per-fold AUCs (left: all test donors, right: uncertain band only) ---
fig, axes = plt.subplots(len(TARGETS_3M), 2, figsize=(10, 4.2 * len(TARGETS_3M)), squeeze=False, sharey=True)
panels = [("auc_clinical_all", "auc_cascade_all", ["clinical\nonly", "cascade"], "all test donors (High vs Low)"),
          ("auc_clinical_band", "auc_snrna_band", ["clinical\n(stage 1)", "snRNA-seq XGB\n(stage 2)"],
           f"uncertain band only ({UNCERTAIN_FRAC:.0%} least confident)")]
for r, target in enumerate(TARGETS_3M):
    sub = df_cas[df_cas["target"] == target]
    for c, (a, b, labels, title) in enumerate(panels):
        ax = axes[r][c]
        va, vb = sub[a].values, sub[b].values
        for x0, x1 in zip(va, vb):                   # one grey line per fold (same test donors)
            ax.plot([0, 1], [x0, x1], color="#d0d0d0", linewidth=0.6, zorder=1)
        for k, v in enumerate((va, vb)):
            ax.scatter(k + rng_jit.uniform(-0.05, 0.05, len(v)), v, s=14, color="#6f8fb3", alpha=0.8,
                       edgecolors="white", linewidths=0.4, zorder=2)
            ax.errorbar(k, np.nanmean(v), yerr=np.nanstd(v), fmt="o", color="#1f3b63", markersize=8,
                        capsize=4, linewidth=2, zorder=3)
            ax.text(k + 0.1, np.nanmean(v), f"{np.nanmean(v):.3f}", va="center", fontsize=9, color="#333333")
        ax.axhline(0.5, color="#999999", linestyle="--", linewidth=1, zorder=0)
        n_band = sub["n_band_test"].mean()
        ax.set_title(f"{target}: {title}" + (f"\n~{n_band:.0f} test donors per fold" if c == 1 else ""),
                     fontsize=10)
        ax.set_xticks([0, 1]); ax.set_xticklabels(labels, fontsize=9); ax.set_xlim(-0.4, 1.5)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    axes[r][0].set_ylabel("held-out test AUC (one dot per CV fold)")
fig.suptitle(f"Clinical -> snRNA-seq cascade (stage 2 trained on: {STAGE2_TRAIN_ON}); dashed line = chance",
             fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTDIR_3M}/{tag}.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{OUTDIR_3M}/{tag}.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUTDIR_3M}/{tag}.png (+ .pdf and CSVs)")

## Pseudobulk gene-expression features instead of the scVI latent

The 30 scVI latent dimensions were trained to separate cell types, not to track aging, and they added nothing above.
These cells build gene-level features instead:

- **Pseudobulk:** sum the raw counts of each donor's cells per cell type, over the 2000 HVGs, then log(1 + CPM).
  A cell type is used when it has ≥ `PB_MIN_CELLS` cells in ≥ `PB_MIN_COVERAGE` of donors. One extra pseudobulk over all
  of the donor's cells is added (every donor has it). Genes are filtered by mean expression only (no labels involved, so no leakage).
- **Models** (same donors, same 5×5 folds as Step 2): `0_base` and `1_clinical` (the Step 2 elastic net), `4_pb_genes`
  (ridge on base + genes), and `5_clinical+pb_genes` (the clinical elastic net, then a ridge on the genes fit to what it misses).
  Ridge is used for the genes because it is fast with thousands of features and suits many small effects.
- **Key number:** 5 minus 1, i.e. whether gene expression adds to the clinical model.

The cascade cell above can use these features too: set `STAGE2_BLOCK = "pb_genes"` there and re-run it.

In [ ]:
# --- PSEUDOBULK GENES, STEP 6: log-CPM per donor x cell type from the raw counts ---
from scipy import sparse

PB_MIN_CELLS = 20            # a donor's cell type is used when it has >= this many cells
PB_MIN_COVERAGE = 0.5        # keep cell types that pass PB_MIN_CELLS in >= this fraction of donors
PB_MIN_MEAN_CPM = 20         # per cell type, keep genes with mean CPM >= this across donors
PB_INCLUDE_ALL_CELLS = True  # also add one pseudobulk over all of the donor's cells
PB_CHUNK = 50_000            # cells per chunk when summing counts (bounds memory)

_pb_counts = adata.layers["counts"] if "counts" in adata.layers else adata.X   # must be raw counts
_keys = pd.MultiIndex.from_arrays([adata.obs["patient_id"].astype(str).str.strip().values,
                                   adata.obs["CellType"].astype(str).values], names=["patient_id", "CellType"])
_codes, _groups = pd.factorize(_keys)
_n = len(_codes)
_G = sparse.csc_matrix((np.ones(_n, dtype=np.float32), (_codes, np.arange(_n))), shape=(len(_groups), _n))

_sums = np.zeros((len(_groups), _pb_counts.shape[1]))
for _s in range(0, _n, PB_CHUNK):
    _sums += (_G[:, _s:_s + PB_CHUNK] @ _pb_counts[_s:_s + PB_CHUNK]).toarray()
pb_sum = pd.DataFrame(_sums, index=pd.MultiIndex.from_tuples(list(_groups), names=_keys.names),
                      columns=list(adata.var_names))
pb_ncells = pd.Series(np.bincount(_codes), index=pb_sum.index)


def _log_cpm_block(sums, label):
    cpm = sums.div(sums.sum(axis=1), axis=0) * 1e6
    genes = cpm.columns[cpm.mean() >= PB_MIN_MEAN_CPM]
    print(f"   {label}: {len(cpm)} donors, {len(genes)} genes with mean CPM >= {PB_MIN_MEAN_CPM}")
    return np.log1p(cpm[genes]).add_prefix(f"pb|{label}|")


_ok = pb_ncells >= PB_MIN_CELLS
_cov = (_ok[_ok].groupby(level="CellType").size() / len(donors_sn)).sort_values(ascending=False)
PB_CELL_TYPES = list(_cov[_cov >= PB_MIN_COVERAGE].index)
print(f"coverage (fraction of donors with >= {PB_MIN_CELLS} cells):\n{_cov.round(2).to_string()}")
print(f"-> using {PB_CELL_TYPES}" + (" + ALL cells" if PB_INCLUDE_ALL_CELLS else ""))

pb_parts = []
if PB_INCLUDE_ALL_CELLS:
    pb_parts.append(_log_cpm_block(pb_sum.groupby(level="patient_id").sum(), "ALL"))
for ct in PB_CELL_TYPES:
    sub = pb_sum.xs(ct, level="CellType")[_ok.xs(ct, level="CellType").values]
    pb_parts += [_log_cpm_block(sub, ct), pd.Series(1.0, index=sub.index, name=f"pbhas|{ct}")]

X_pb_genes = pd.concat(pb_parts, axis=1).reindex(donors_sn)
_has = [c for c in X_pb_genes.columns if c.startswith("pbhas|")]
X_pb_genes[_has] = X_pb_genes[_has].fillna(0.0)          # absent cell type -> flag 0, genes imputed in the CV
BLOCKS_3M["pb_genes"] = X_pb_genes
print(f"\npb_genes block: {X_pb_genes.shape[0]} donors x {X_pb_genes.shape[1]} features")

In [ ]:
# --- PSEUDOBULK GENES, STEP 7: same donor-level CV as Step 2, with the gene block ---
MODELS_PB = {
    "0_base":              ["base"],
    "1_clinical":          ["base", "clinical"],
    "4_pb_genes":          ["base", "pb_genes"],
    "5_clinical+pb_genes": ["base", "clinical", "pb_genes"],
}
COMPARISONS_PB = [("4_pb_genes", "0_base"), ("5_clinical+pb_genes", "1_clinical")]


from sklearn.linear_model import RidgeCV


def _make_ridge_pb():
    # ridge instead of elastic net: with thousands of genes an elastic net takes ~45 min for 25 folds,
    # RidgeCV takes seconds (efficient leave-one-out over the alphas when features > donors)
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         RidgeCV(alphas=np.logspace(0, 6, 25)))


class ClinicalPlusGenes:
    """Step 2's clinical elastic net, then a ridge on the genes fit to what the clinical model misses.
    Fitting them jointly would put 15 clinical features under the same penalty as thousands of genes."""
    def __init__(self, clin_cols, gene_cols):
        self.clin_cols, self.gene_cols = clin_cols, gene_cols

    def fit(self, X, y):
        self.clin_ = _make_model_3m().fit(X[self.clin_cols], y)
        self.genes_ = _make_ridge_pb().fit(X[self.gene_cols], y - self.clin_.predict(X[self.clin_cols]))
        return self

    def predict(self, X):
        return self.clin_.predict(X[self.clin_cols]) + self.genes_.predict(X[self.gene_cols])


def _make_model_pb(m, X):
    if m in ("0_base", "1_clinical"):
        return _make_model_3m()                       # identical to Step 2
    if m == "4_pb_genes":
        return _make_ridge_pb()
    clin_cols = list(BLOCKS_3M["base"].columns) + list(BLOCKS_3M["clinical"].columns)
    return ClinicalPlusGenes(clin_cols, [c for c in X.columns if c not in clin_cols])


def _gene_coefs(m, pipe, X):
    # standardized coefficients of the gene part (ridge keeps every gene, so report the largest)
    if m == "4_pb_genes":
        return X.columns, pipe[-1].coef_
    return pipe.gene_cols, pipe.genes_[-1].coef_


pb_rows, pb_coef_rows = [], []
for target in TARGETS_3M:
    y_all = Y_3M[target].reindex(donors_sn)
    donors = y_all.index[y_all.notna()]
    y = y_all.loc[donors].values
    X_by_model = {m: pd.concat([BLOCKS_3M[b].reindex(donors) for b in blocks], axis=1)
                  for m, blocks in MODELS_PB.items()}
    print(f"\n===== {target}: {len(donors)} donors =====")
    rkf = RepeatedKFold(n_splits=N_FOLDS_3M, n_repeats=N_REPEATS_3M, random_state=0)   # same folds as Step 2
    for fold, (tr, te) in enumerate(rkf.split(donors)):
        lo, hi = np.quantile(y[tr], quantile_low), np.quantile(y[tr], quantile_high)
        ext = (y[te] <= lo) | (y[te] >= hi)
        for m, X in X_by_model.items():
            pipe = _make_model_pb(m, X).fit(X.iloc[tr], y[tr])
            pred = pipe.predict(X.iloc[te])
            pb_rows.append({
                "target": target, "model": m, "fold": fold, "n_train": len(tr), "n_test": len(te),
                "n_features": X.shape[1],
                "test_r2": r2_score(y[te], pred),
                "test_spearman": stats.spearmanr(y[te], pred)[0] if np.std(pred) > 0 else np.nan,
                "test_auc_high_vs_low": (roc_auc_score(y[te][ext] >= hi, pred[ext])
                                         if len(np.unique(y[te][ext] >= hi)) == 2 else np.nan),
                "train_r2": r2_score(y[tr], pipe.predict(X.iloc[tr])),
            })
            if m in ("4_pb_genes", "5_clinical+pb_genes"):
                pb_coef_rows += [{"target": target, "model": m, "fold": fold, "feature": f, "coef": c}
                                 for f, c in zip(*_gene_coefs(m, pipe, X))]
        print(f"   fold {fold + 1}/{N_FOLDS_3M * N_REPEATS_3M} done", end="\r")
    print()

df_folds_pb = pd.DataFrame(pb_rows)
_metrics = ["test_r2", "test_spearman", "test_auc_high_vs_low", "train_r2"]
df_summary_pb = df_folds_pb.groupby(["target", "model"])[_metrics].agg(["mean", "std"]).round(3)
df_summary_pb.columns = [f"{a}_{b}" for a, b in df_summary_pb.columns]
df_summary_pb = df_summary_pb.reset_index()

_comp = []
for target, sub in df_folds_pb.groupby("target"):
    for a, b in COMPARISONS_PB:
        for metric in ("test_r2", "test_spearman", "test_auc_high_vs_low"):
            d = (sub[sub["model"] == a].set_index("fold")[metric]
                 - sub[sub["model"] == b].set_index("fold")[metric]).dropna()
            t, p = corrected_ttest(d, sub["n_train"].mean(), sub["n_test"].mean())
            _comp.append({"target": target, "comparison": f"{a} - {b}", "metric": metric, "mean_diff": d.mean(),
                          "frac_folds_better": (d > 0).mean(), "corrected_t": t, "p_corrected": p})
df_comp_pb = pd.DataFrame(_comp)

# gene weights: mean standardized ridge coefficient over folds, and how often its sign agrees with that mean
df_coef_pb = (pd.DataFrame(pb_coef_rows).groupby(["target", "model", "feature"])["coef"]
              .agg(mean_coef="mean", frac_same_sign=lambda c: (np.sign(c) == np.sign(c.mean())).mean())
              .reset_index())
df_coef_pb["abs_mean_coef"] = df_coef_pb["mean_coef"].abs()
df_coef_pb = df_coef_pb.sort_values(["target", "model", "abs_mean_coef"], ascending=[True, True, False])

for name, d in [("pb_genes_per_fold", df_folds_pb), ("pb_genes_summary", df_summary_pb),
                ("pb_genes_comparisons", df_comp_pb), ("pb_genes_selected_features", df_coef_pb)]:
    d.to_csv(f"{OUTDIR_3M}/{name}.csv", index=False, sep=';', decimal=',')

print(df_summary_pb[["target", "model", "test_r2_mean", "test_r2_std", "test_spearman_mean",
                     "test_auc_high_vs_low_mean", "train_r2_mean"]].to_string(index=False))
print()
print(df_comp_pb.round(4).to_string(index=False))
for target in TARGETS_3M:
    print(f"\nLargest gene weights, 4_pb_genes, {target} (mean over {N_FOLDS_3M * N_REPEATS_3M} folds):")
    print(df_coef_pb[(df_coef_pb["target"] == target) & (df_coef_pb["model"] == "4_pb_genes")]
          .head(20)[["feature", "mean_coef", "frac_same_sign"]].round(4).to_string(index=False))

# plot: same layout as Step 3
_jit = np.random.default_rng(0)
_order = list(MODELS_PB)
_labels = ["base\n(sex, QC)", "clinical\n(age 26)", "pseudobulk\ngenes", "clinical +\ngenes"]
fig, axes = plt.subplots(1, len(TARGETS_3M), figsize=(5.5 * len(TARGETS_3M), 4.8), sharey=True, squeeze=False)
for ax, target in zip(axes[0], TARGETS_3M):
    wide = df_folds_pb[df_folds_pb["target"] == target].pivot(index="fold", columns="model", values="test_r2")[_order]
    for fold_vals in wide.values:
        ax.plot(range(len(_order)), fold_vals, color="#d0d0d0", linewidth=0.6, zorder=1)
    for k, m in enumerate(_order):
        v = wide[m].values
        ax.scatter(k + _jit.uniform(-0.08, 0.08, len(v)), v, s=14, color="#6f8fb3", alpha=0.8,
                   edgecolors="white", linewidths=0.4, zorder=2)
        ax.errorbar(k, v.mean(), yerr=v.std(), fmt="o", color="#1f3b63", markersize=8, capsize=4,
                    linewidth=2, zorder=3)
        ax.text(k + 0.18, v.mean(), f"{v.mean():.3f}", va="center", fontsize=9, color="#333333")
    ax.axhline(0, color="#999999", linestyle="--", linewidth=1, zorder=0)
    c = df_comp_pb[(df_comp_pb["target"] == target) & (df_comp_pb["metric"] == "test_r2")
                   & (df_comp_pb["comparison"] == "5_clinical+pb_genes - 1_clinical")].iloc[0]
    ax.set_title(f"{target}\nadding pseudobulk genes to clinical: ΔR² = {c['mean_diff']:+.3f} "
                 f"(p = {c['p_corrected']:.2g})", fontsize=10)
    ax.set_xticks(range(len(_order))); ax.set_xticklabels(_labels, fontsize=9)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0][0].set_ylabel("held-out test R² (one dot per CV fold)")
fig.suptitle(f"Pseudobulk gene expression ({X_pb_genes.shape[1]} features) vs age-26 clinical: "
             f"{N_REPEATS_3M}x{N_FOLDS_3M}-fold donor-level CV", fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{OUTDIR_3M}/pb_genes_r2.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{OUTDIR_3M}/pb_genes_r2.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUTDIR_3M}/pb_genes_r2.png (+ .pdf and CSVs)")

# Version 12: full-transcriptome, library-corrected pseudobulk + proxy features, same XGBoost AUC pipeline

This section is **self-contained**. It needs only the GLOBAL CONFIG cell (`BIOMARKER`, `TARGET_GROUP`,
`quantile_low`, `quantile_high`, `N_ITER_PDF`) and reads the data straight from the CellBender h5ad.
It does **not** use the scVI model or `df_pb_latent`.

**What changes vs. `run_discovery_pdf_report` (v2):** only the features given to XGBoost.

1. **Full-transcriptome pseudobulk:** raw counts of all 29,572 genes, summed per donor × library (× cell type),
   instead of scVI-normalized expression of the top 1000 HVGs.
2. **Library correction:** log-CPM per donor × library, minus that library's mean profile, then a cell-count-weighted
   average per donor. This removes sequencing-library batch effects gene by gene. It is label-free.
3. **Proxy features** (`feature_mode="proxy"`): inside each outer split, ridge regressions learn to predict 18 age-26
   traits (13 clinical + 5 blood counts) **and** the continuous biomarker from the pseudobulk. They are fit on the
   non-test donors only; the training donors get inner out-of-fold predictions. The resulting 19 "predicted trait"
   features go to XGBoost. The clinical traits are only training labels: a new donor needs only snRNA-seq.
   `feature_mode="genes"` gives XGBoost the top library-corrected genes directly (like v2, without scVI),
   and `"proxy+genes"` gives both.

**What stays identical:** High/Low labels (`quantile_low`/`quantile_high`, `label_cutoffs`), the stratified 70/30
outer split and its seeds, XGBoost stability selection on train donors, inner validation, the final XGBoost
(`n_estimators=500`, `max_depth=4`), test AUC, train AUC / overfit gap, the multi-split summary, and the PDF
(ROC, SHAP, table, tree). Output files get a `_{feature_mode}_libcorr` suffix so the v2 reports are not overwritten.

Overnight benchmark (`/home/nlapique/Blood/script/NIGHT_PaceOfAging_snRNA/REPORT.md`): the proxy model with a ridge
second level reached AUC ≈ 0.70 on held-out donors, snRNA-seq only.

In [ ]:
# --- V12 STEP 1: donor table + library-corrected full-transcriptome pseudobulk (cached to disk) ---
import os, time, h5py
import numpy as np
import pandas as pd
from scipy import sparse
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem

V12_H5AD = "/home/nlapique/Blood/cell_bender_processed/combined_annotated_final_with_dunedinpace_traits.h5ad"
V12_H5AD_ANNOT = "/home/nlapique/Blood/adata_reduced_HVG_2000_with_sex_qcmt.h5ad"   # CellType + library (same cell order)
V12_CACHE = "/home/nlapique/Blood/script/v12_libcorr_pseudobulk_cache/"
V12_MIN_CPM = 3.0            # keep genes with pooled CPM >= this (≈ 11.8k genes for all cells)
V12_MIN_GROUP_CELLS = 3      # a donor x library group needs >= this many cells
V12_MIN_DONOR_CELLS = 10     # for a single cell type: donors need >= this many cells of it in total
PROXY_TRAITS = ["bmi26", "choles26", "trigly26", "hdl26", "urea26", "VOmax26", "MeanArtPr26", "LnCRP26", "wbc26",
                "glyhaem26", "cortis26np", "CurrentSmk26", "PackYrsBy26",
                "neutro26np", "lymph26np", "mono26np", "eosino26np", "baso26np"]   # age-26 only: training labels
os.makedirs(V12_CACHE, exist_ok=True)

_BIOMARKER = globals().get("BIOMARKER", "PaceOfAgingP45")

with h5py.File(V12_H5AD) as f:
    _cols = ["patient_id", "sex", _BIOMARKER, *PROXY_TRAITS] + [c for c in f["obs"].attrs["column-order"]
                                                                  if str(c).startswith("POA45_no") and c != _BIOMARKER]
    _obs = pd.DataFrame({c: np.asarray(read_elem(f["obs"][c])) for c in dict.fromkeys(_cols)})
with h5py.File(V12_H5AD_ANNOT) as g:
    _obs["CellType"] = np.asarray(read_elem(g["obs"]["CellType"])).astype(str)
    _obs["library"] = np.asarray(read_elem(g["obs"]["batch_merging"])).astype(str)
_obs["patient_id"] = _obs["patient_id"].astype(str).str.strip()

df_v12_donors = _obs.drop(columns=["CellType", "library"]).groupby("patient_id").first()
df_v12_donors["n_cells"] = _obs.groupby("patient_id").size()
print(f"donor table: {df_v12_donors.shape[0]} donors, {df_v12_donors[_BIOMARKER].notna().sum()} with {_BIOMARKER}")


def _group_counts(cell_type):
    """Summed raw counts per (donor, library) for one cell type ('ALL' = all cells); cached on disk."""
    tag = cell_type.replace(" ", "_")
    f_sum, f_idx = V12_CACHE + f"sum_{tag}.npy", V12_CACHE + f"groups_{tag}.pkl"
    if os.path.exists(f_sum):
        return np.load(f_sum, mmap_mode="r"), pd.read_pickle(f_idx)
    t0 = time.time()
    use = np.ones(len(_obs), bool) if cell_type == "ALL" else (_obs["CellType"].values == cell_type)
    keys = pd.MultiIndex.from_arrays([_obs["patient_id"].values, _obs["library"].values])
    codes, groups = pd.factorize(keys[use])
    cells = np.flatnonzero(use)
    G = sparse.csc_matrix((np.ones(len(cells), np.float32), (codes, cells)), shape=(len(groups), len(_obs)))
    S = None
    with h5py.File(V12_H5AD) as f:
        L = f["layers"]["counts"]; ip = L["indptr"][:]; ng = int(L.attrs["shape"][1])
        for s in range(0, len(_obs), 40_000):
            e = min(s + 40_000, len(_obs)); a, b = ip[s], ip[e]
            X = sparse.csr_matrix((L["data"][a:b].astype(np.float32), L["indices"][a:b], ip[s:e + 1] - a),
                                  shape=(e - s, ng))
            part = G[:, s:e] @ X
            S = part if S is None else S + part
    S = S.toarray().astype(np.float32)
    idx = pd.DataFrame(list(groups), columns=["patient_id", "library"])
    idx["n_cells"] = np.bincount(codes)
    np.save(f_sum, S); idx.to_pickle(f_idx)
    print(f"   built pseudobulk cache for {cell_type}: {len(idx)} donor x library groups ({time.time() - t0:.0f}s)")
    return np.load(f_sum, mmap_mode="r"), idx


with h5py.File(V12_H5AD) as f:
    V12_GENES = np.asarray(read_elem(f["var"]).index).astype(str)


def libcorr_pseudobulk(cell_type="ALL", min_cpm=V12_MIN_CPM):
    """Donor x gene log-CPM, library-corrected (label-free), cell-count-weighted over each donor's libraries."""
    S, idx = _group_counts(cell_type)
    keep_g = idx["n_cells"].to_numpy() >= V12_MIN_GROUP_CELLS
    s = np.asarray(S[np.flatnonzero(keep_g)], dtype=np.float64)
    idx = idx[keep_g].reset_index(drop=True)
    genes = (s.sum(0) / s.sum() * 1e6) >= min_cpm
    lc = np.log((s[:, genes] + 1.0) / (s.sum(1, keepdims=True) + 1.0) * 1e6)
    w = idx["n_cells"].to_numpy(float)
    grand = np.average(lc, axis=0, weights=w)
    libs = idx["library"].to_numpy()
    for lib in np.unique(libs):
        m = libs == lib
        if m.sum() >= 3:
            lc[m] -= np.average(lc[m], axis=0, weights=w[m]) - grand
    W = sparse.csr_matrix((w, (pd.factorize(idx["patient_id"])[0], np.arange(len(idx)))))
    pids = pd.factorize(idx["patient_id"])[1]
    out = (W @ lc) / np.asarray(W.sum(1))
    df = pd.DataFrame(out, index=pids, columns=[f"{cell_type}|{g}" for g in V12_GENES[genes]])
    if cell_type != "ALL":
        n_tot = idx.groupby("patient_id")["n_cells"].sum().reindex(df.index)
        df = df[n_tot.to_numpy() >= V12_MIN_DONOR_CELLS]
    return df


V12_PB = {"ALL": libcorr_pseudobulk("ALL")}
print(f"library-corrected pseudobulk ALL: {V12_PB['ALL'].shape[0]} donors x {V12_PB['ALL'].shape[1]} genes")

In [ ]:
# --- V12 STEP 2: discovery function = v2 pipeline with proxy / library-corrected gene features ---
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_curve, auc
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from xgboost import XGBClassifier
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

XGB_DEVICE = "cuda"          # <-- set "cpu" if CUDA errors


def _proxy_ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-1, 6, 30), alpha_per_target=True))


def proxy_features(G_fit, labels_fit, G_apply, n_inner=5, seed=0):
    """Level 1 of the proxy model. G_fit: donors x genes used for fitting (non-test donors);
    labels_fit: donors x (traits + biomarker), standardized inside; G_apply: donors to predict with the full fit.
    Returns (inner out-of-fold predictions for G_fit, predictions for G_apply)."""
    L = labels_fit.apply(pd.to_numeric, errors="coerce")
    L = L.fillna(L.median())
    mu, sd = L.mean(), L.std() + 1e-9
    Y = ((L - mu) / sd).to_numpy()
    oof = np.zeros_like(Y)
    for tr, te in KFold(n_inner, shuffle=True, random_state=seed).split(G_fit):
        oof[te] = _proxy_ridge().fit(G_fit.iloc[tr], Y[tr]).predict(G_fit.iloc[te])
    full = _proxy_ridge().fit(G_fit, Y)
    cols = [f"pred_{c}" for c in labels_fit.columns]
    return (pd.DataFrame(oof, index=G_fit.index, columns=cols),
            pd.DataFrame(full.predict(G_apply), index=G_apply.index, columns=cols))


def run_discovery_proxy_report(
    df_donors, biomarker="PaceOfAgingP45", cell_type="ALL",
    feature_mode="proxy",         # "proxy" | "genes" | "proxy+genes"
    proxy_train="non_test",       # "non_test" = fit level 1 on ALL non-test donors of the group (incl. the middle,
                                   #              unlabeled ones: they are never in the test set)
                                   # "all_non_test" = fit level 1 on all non-test donors of EVERY group (e.g. all
                                   #              tertiles): more data; test donors are still never used
                                   # "train_only" = fit level 1 on the High/Low train donors only
    n_iterations=50, stability_thresh=60, n_top_genes=1000, target_group="ALL",
    make_pdf=True, outer_test_size=0.3, outer_random_state=42, min_train=15, min_test=5,
    inner_val_size=0.3, n_estimators=500, max_depth=4, group_col=None, label_cutoffs="group",
):
    tag = f"{feature_mode.replace('+', '_')}_libcorr"
    filename = f"Rapport_{biomarker}_{cell_type}_{target_group}_{tag}.pdf"
    filename_csv = f"Tableau_{biomarker}_{cell_type}_{target_group}_{tag}.csv"
    file_tree = f"Arbre_{biomarker}_{cell_type}_{tag}.pdf"
    if make_pdf: print(f"⚡️ ROBUST ANALYSIS & PDF REPORT: {filename}")

    # --- 1. LABELS (identical to v2) ---
    df_clin = df_donors.copy()
    if "patient_id" not in df_clin.columns: df_clin["patient_id"] = df_clin.index.astype(str)
    df_clin["patient_id"] = df_clin["patient_id"].astype(str).str.strip()
    df_clin["sex"] = df_clin["sex"].astype(str).str.replace(".0", "", regex=False).str.strip()
    df_clin["sex_mapped"] = df_clin["sex"].map({'1': 'F', '2': 'M', 'F': 'F', 'M': 'M'}).fillna('Unknown')
    filter_col = group_col if group_col is not None else 'sex_mapped'
    if str(target_group).upper() in ('ALL', 'BOTH'):
        df_target = df_clin.drop_duplicates(subset=['patient_id'])
    else:
        df_target = df_clin[df_clin[filter_col] == target_group].drop_duplicates(subset=['patient_id'])
    if isinstance(label_cutoffs, (tuple, list)):
        low, high = label_cutoffs
    elif label_cutoffs == "global":
        high, low = df_clin[biomarker].quantile(quantile_high), df_clin[biomarker].quantile(quantile_low)
    elif label_cutoffs == "group":
        high, low = df_target[biomarker].quantile(quantile_high), df_target[biomarker].quantile(quantile_low)
    else:
        raise ValueError(f"label_cutoffs must be 'group', 'global' or a (low, high) tuple, got {label_cutoffs!r}")
    df_target['Label'] = np.nan
    df_target.loc[df_target[biomarker] >= high, 'Label'] = 1
    df_target.loc[df_target[biomarker] <= low, 'Label'] = 0

    if cell_type not in V12_PB:
        V12_PB[cell_type] = libcorr_pseudobulk(cell_type)
    PB = V12_PB[cell_type]
    df_target = df_target[df_target["patient_id"].isin(PB.index)]
    df_ml = df_target.dropna(subset=['Label'])
    if df_ml['Label'].nunique() < 2 or len(df_ml) < (min_train + min_test): return None
    if make_pdf:
        print(f"   -> Task: {int(df_ml['Label'].sum())} High vs {int((df_ml['Label'] == 0).sum())} Low "
              f"(High >= {high:.3f}, Low <= {low:.3f}, label_cutoffs={label_cutoffs!r}, features={feature_mode})")

    # --- 2. OUTER SPLIT FIRST (identical to v2) ---
    train_pat, test_pat = train_test_split(df_ml['patient_id'], test_size=outer_test_size,
                                           random_state=outer_random_state, stratify=df_ml['Label'])
    train_pat, test_pat = list(train_pat), list(test_pat)
    labels_by_patient = df_ml.set_index('patient_id')['Label']

    # --- 3. FEATURES (the only change): built from non-test donors, applied to test donors ---
    parts_tr, parts_te = [], []
    if "proxy" in feature_mode:
        pool = df_clin[df_clin["patient_id"].isin(PB.index)] if proxy_train == "all_non_test" else df_target
        if proxy_train in ("non_test", "all_non_test"):
            fit_ids = [p for p in pool.loc[pool[biomarker].notna(), "patient_id"] if p not in set(test_pat)]
        else:
            fit_ids = train_pat
        lab = pool.drop_duplicates("patient_id").set_index("patient_id").loc[fit_ids, PROXY_TRAITS + [biomarker]]
        oof, pred_test = proxy_features(PB.loc[fit_ids], lab, PB.loc[test_pat], seed=outer_random_state)
        parts_tr.append(oof.loc[train_pat]); parts_te.append(pred_test)
    if "genes" in feature_mode:
        top = PB.loc[train_pat].mean().sort_values(ascending=False).head(n_top_genes).index   # train donors only
        parts_tr.append(PB.loc[train_pat, top]); parts_te.append(PB.loc[test_pat, top])
    X_train_full = pd.concat(parts_tr, axis=1)
    X_test_full = pd.concat(parts_te, axis=1)
    y_train_full = labels_by_patient.loc[X_train_full.index]
    y_test_full = labels_by_patient.loc[X_test_full.index]
    candidates = list(X_train_full.columns)
    if len(X_train_full) < min_train or len(X_test_full) < min_test: return None
    if y_train_full.nunique() < 2 or y_test_full.nunique() < 2: return None

    # --- 4. STABILITY SELECTION on TRAIN only (identical to v2) ---
    counts = pd.Series(0, index=candidates)
    for i in range(n_iterations):
        seed = 42 + i
        Xtr, _, ytr, _ = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=seed,
                                          stratify=y_train_full)
        clf = XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.6,
                            device=XGB_DEVICE, tree_method="hist", eval_metric='logloss', random_state=seed)
        clf.fit(Xtr, ytr)
        imps = pd.Series(clf.feature_importances_, index=X_train_full.columns)
        counts[imps[imps > 0].index] += 1
    stability_pct = counts / n_iterations * 100
    stable = stability_pct[stability_pct >= stability_thresh].index.tolist()
    if len(stable) < 2:
        stable = stability_pct.sort_values(ascending=False).head(10).index.tolist()
    X_train_stable, X_test_stable = X_train_full[stable], X_test_full[stable]

    # --- 5. INNER VALIDATION (identical to v2) ---
    inner_val_auc = np.nan
    n_subtrain = n_subval = 0
    try:
        sub_tr, sub_val = train_test_split(X_train_stable.index, test_size=inner_val_size,
                                           random_state=outer_random_state, stratify=y_train_full)
        if y_train_full.loc[sub_tr].nunique() == 2 and y_train_full.loc[sub_val].nunique() == 2:
            n_subtrain, n_subval = len(sub_tr), len(sub_val)
            iclf = XGBClassifier(n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth, subsample=0.8,
                                 colsample_bytree=0.8, device=XGB_DEVICE, tree_method="hist", eval_metric='logloss',
                                 random_state=42)
            iclf.fit(X_train_stable.loc[sub_tr], y_train_full.loc[sub_tr])
            f_iv, t_iv, _ = roc_curve(y_train_full.loc[sub_val], iclf.predict_proba(X_train_stable.loc[sub_val])[:, 1])
            inner_val_auc = auc(f_iv, t_iv)
    except ValueError:
        pass

    # --- 6. FINAL MODEL + held-out TEST AUC (identical to v2) ---
    final_clf = XGBClassifier(n_estimators=n_estimators, learning_rate=0.02, max_depth=max_depth, subsample=0.8,
                              colsample_bytree=0.8, device=XGB_DEVICE, tree_method="hist", eval_metric='logloss',
                              random_state=42)
    final_clf.fit(X_train_stable, y_train_full)
    y_proba = final_clf.predict_proba(X_test_stable)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_full, y_proba)
    roc_auc = auc(fpr, tpr)
    fpr_tr, tpr_tr, _ = roc_curve(y_train_full, final_clf.predict_proba(X_train_stable)[:, 1])
    train_auc = auc(fpr_tr, tpr_tr)
    overfit_gap = train_auc - roc_auc
    overfit_gap_innerval = (train_auc - inner_val_auc) if not np.isnan(inner_val_auc) else np.nan
    if make_pdf:
        print(f"🎯 TEST (held-out) AUC: {roc_auc:.2f} | TRAIN AUC: {train_auc:.2f} | OVERFIT GAP: {overfit_gap:+.2f}")
        print(f"   -> Inner-val AUC: {inner_val_auc:.2f} (sub-train {n_subtrain} / sub-val {n_subval})")

    all_X = pd.concat([X_train_stable, X_test_stable])
    scores_all = final_clf.predict_proba(all_X)[:, 1]
    lbl = labels_by_patient.loc[all_X.index].astype(int)
    df_scores = pd.DataFrame({"patient_id": all_X.index, "label": lbl.values,
                              "category": np.where(lbl.values == 1, "High", "Low"), "score_high": scores_all,
                              "predicted": np.where(scores_all >= 0.5, "High", "Low"),
                              "in_train": [p in set(train_pat) for p in all_X.index]})

    # --- 7. TABLES (same layout as v2; "Gene" = feature: a predicted trait or a gene) ---
    imp_map = dict(zip(stable, final_clf.feature_importances_))
    df_all = pd.DataFrame({"Gene": stability_pct.index, "Stability (%)": stability_pct.values})
    df_all = df_all[df_all["Stability (%)"] > 50].sort_values("Stability (%)", ascending=False)
    df_all["Final Importance"] = df_all["Gene"].map(imp_map)
    df_res = pd.DataFrame({"Gene": stable, "Stability (%)": stability_pct[stable].values,
                           "Final Importance": final_clf.feature_importances_}
                          ).sort_values("Stability (%)", ascending=False).head(25)

    # --- 8. PDF (same content as v2) ---
    if make_pdf:
        import shap
        df_all.to_csv(filename_csv, index=False, sep=';', decimal=',')
        with PdfPages(filename) as pdf:
            fig_roc = plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'Test ROC (AUC = {roc_auc:.2f}, held-out)')
            plt.plot(fpr_tr, tpr_tr, color='#1f77b4', lw=2, linestyle=':', label=f'Train ROC (AUC = {train_auc:.2f}, in-sample)')
            plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
            plt.title(f"Diagnostic Performance: {biomarker} ({cell_type}, {feature_mode})\nOverfit gap = {overfit_gap:+.2f}", fontsize=13)
            plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
            pdf.savefig(fig_roc); plt.show(); plt.close(fig_roc)
            shap_values = shap.TreeExplainer(final_clf).shap_values(X_train_stable)
            fig_shap = plt.figure()
            shap.summary_plot(shap_values, X_train_stable, show=False, max_display=20)
            plt.title(f"SHAP - {cell_type} ({feature_mode})", fontsize=13)
            plt.tight_layout(); pdf.savefig(fig_shap, bbox_inches='tight'); plt.show(); plt.close(fig_shap)
            if "draw_table" in globals():
                draw_table(df_res, f"Top robust features: {cell_type} ({feature_mode})", pdf)
            try:
                fig_tree, ax_tree = plt.subplots(figsize=(30, 40))
                xgb.plot_tree(final_clf, tree_idx=0, ax=ax_tree, rankdir='TB')
                plt.savefig(file_tree, bbox_inches='tight'); plt.close(fig_tree)
            except Exception as e:
                print(f"   ⚠️ Tree error: {e}")
        print(f"✅ Done! Files saved: {filename}, {filename_csv}")

    return {"roc_auc": roc_auc, "train_auc": train_auc, "overfit_gap": overfit_gap,
            "inner_val_auc": inner_val_auc, "overfit_gap_innerval": overfit_gap_innerval,
            "n_subtrain": n_subtrain, "n_subval": n_subval, "df_all": df_all, "df_res": df_res,
            "final_clf": final_clf, "donors": train_pat + test_pat, "df_scores": df_scores,
            "outer_random_state": outer_random_state, "n_train": len(X_train_full), "n_test": len(X_test_full),
            "label_cutoff_low": low, "label_cutoff_high": high}


def run_discovery_multi_split_proxy(df_donors, n_outer_splits=10, outer_split_seeds=None, make_pdf_for_first=False,
                                    **kwargs):
    """Same as run_discovery_multi_split (v2), calling run_discovery_proxy_report; same summary keys."""
    seeds = list(outer_split_seeds) if outer_split_seeds is not None else list(range(n_outer_splits))
    rows, hits = [], {}
    for k, seed in enumerate(seeds):
        r = run_discovery_proxy_report(df_donors, outer_random_state=seed, make_pdf=(make_pdf_for_first and k == 0), **kwargs)
        if r is None:
            rows.append({"split": k, "seed": seed, "test_auc": np.nan, "train_auc": np.nan, "inner_val_auc": np.nan,
                         "overfit_gap": np.nan, "overfit_gap_innerval": np.nan, "n_train": np.nan, "n_test": np.nan})
            continue
        rows.append({"split": k, "seed": seed, "test_auc": r["roc_auc"], "train_auc": r["train_auc"],
                     "inner_val_auc": r["inner_val_auc"], "overfit_gap": r["overfit_gap"],
                     "overfit_gap_innerval": r["overfit_gap_innerval"], "n_train": r["n_train"], "n_test": r["n_test"]})
        for g in r["df_res"]["Gene"]:
            hits[g] = hits.get(g, 0) + 1
    d = pd.DataFrame(rows)
    s = {"cell_type": kwargs.get("cell_type", "ALL"), "feature_mode": kwargs.get("feature_mode", "proxy"),
         "biomarker": kwargs.get("biomarker"), "target_group": kwargs.get("target_group", "ALL"),
         "label_cutoffs": kwargs.get("label_cutoffs", "group"),
         "n_splits_requested": len(seeds), "n_splits_valid": int(d["test_auc"].notna().sum()),
         "test_auc_mean": d["test_auc"].mean(), "test_auc_std": d["test_auc"].std(),
         "train_auc_mean": d["train_auc"].mean(),
         "inner_val_auc_mean": d["inner_val_auc"].mean(), "inner_val_auc_std": d["inner_val_auc"].std(),
         "overfit_gap_mean": d["overfit_gap"].mean(), "overfit_gap_std": d["overfit_gap"].std(),
         "overfit_gap_innerval_mean": d["overfit_gap_innerval"].mean(),
         "overfit_gap_innerval_std": d["overfit_gap_innerval"].std(),
         "n_test_mean": d["n_test"].mean()}
    print(f"📊 {s['cell_type']} | {s['feature_mode']} | {s['biomarker']} ({s['target_group']}) over "
          f"{s['n_splits_valid']}/{len(seeds)} splits: TEST AUC {s['test_auc_mean']:.3f} ± {s['test_auc_std']:.3f} | "
          f"inner-val {s['inner_val_auc_mean']:.3f} | train {s['train_auc_mean']:.3f}")
    feats = pd.Series(hits, dtype=float).sort_values(ascending=False).rename("times_in_top25").to_frame()
    if len(seeds): feats["pct_of_splits"] = feats["times_in_top25"] / len(seeds) * 100
    return {"df_splits": d, "summary": s, "genes_across_splits": feats}

In [ ]:
# --- V12 STEP 3: run the multi-split sweep (same AUC protocol as the v2 sweeps) ---
V12_CELL_TYPES = ["ALL", "CD4 Active", "Naive T cells", "CD8 Active", "B Naive", "B Memory", "NK cells",
                  "Classical Monocytes"]   # MAIT, Non-classical Monocytes, Plasma cells: too few donors with >= 10 cells
V12_FEATURE_MODES = ["proxy", "genes"]      # "proxy+genes" also possible
V12_N_SPLITS = 10
V12_TARGET_GROUP = "ALL"                    # or TARGET_GROUP ("M"/"F") for one sex
OUTDIR_V12 = f"MULTISPLIT_{_BIOMARKER}_v12_libcorr_proxy"
os.makedirs(OUTDIR_V12, exist_ok=True)

v12_results, v12_rows = {}, []
for ct in V12_CELL_TYPES:
    for fm in V12_FEATURE_MODES:
        print(f"\n===== {ct} | features = {fm} =====")
        res = run_discovery_multi_split_proxy(
            df_v12_donors, n_outer_splits=V12_N_SPLITS, make_pdf_for_first=True,
            biomarker=_BIOMARKER, cell_type=ct, feature_mode=fm, target_group=V12_TARGET_GROUP,
            n_iterations=globals().get("N_ITER_PDF", 50), label_cutoffs="group",
        )
        v12_results[(ct, fm)] = res
        v12_rows.append(res["summary"])
        res["df_splits"].to_csv(f"{OUTDIR_V12}/splits_{ct.replace(' ', '_')}_{fm.replace('+', '_')}.csv", index=False, sep=';', decimal=',')
        res["genes_across_splits"].to_csv(f"{OUTDIR_V12}/features_{ct.replace(' ', '_')}_{fm.replace('+', '_')}.csv", sep=';', decimal=',')

df_v12_summary = pd.DataFrame(v12_rows).sort_values("test_auc_mean", ascending=False)
df_v12_summary.to_csv(f"{OUTDIR_V12}/summary.csv", index=False, sep=';', decimal=',')
df_v12_summary[["cell_type", "feature_mode", "n_splits_valid", "n_test_mean", "test_auc_mean", "test_auc_std",
                "inner_val_auc_mean", "train_auc_mean", "overfit_gap_mean"]].round(3)

In [ ]:
# --- V12 STEP 4: Low/Medium/High tertile categories for age-26 traits (added to df_v12_donors) ---
TRAITS_FOR_TERTILE_V12 = ["bmi26", "choles26", "trigly26", "hdl26", "urea26",
                      "VOmax26", "MeanArtPr26", "crp26ns","cortis26np","PackYrsBy26"]

#TRAITS_FOR_TERTILE_V12 = ["crp26ns"]

_missing = [t for t in TRAITS_FOR_TERTILE_V12 if t not in df_v12_donors.columns]
if _missing:   # traits not loaded in STEP 1: read them from the h5ad obs
    with h5py.File(V12_H5AD) as f:
        _extra = pd.DataFrame({c: np.asarray(read_elem(f["obs"][c])) for c in ["patient_id", *_missing]})
    _extra["patient_id"] = _extra["patient_id"].astype(str).str.strip()
    df_v12_donors = df_v12_donors.join(_extra.groupby("patient_id").first())

for t in TRAITS_FOR_TERTILE_V12:
    try:
        df_v12_donors[f"{t}_cat"] = pd.qcut(pd.to_numeric(df_v12_donors[t], errors="coerce"), q=3,
                                            labels=["Low", "Medium", "High"], duplicates="drop").astype(object)
    except ValueError as e:
        print(f"⚠️ Could not tertile-split {t}: {e}")
        continue
    print(f"{t}: {df_v12_donors[t].notna().sum()} donors with a value -> "
          f"{dict(df_v12_donors[f'{t}_cat'].value_counts(dropna=False))}")

In [ ]:
import os, re, gc
# --- V12 STEP 5: tertile sweep, full-transcriptome proxy features -> same XGBoost AUC protocol ---
# Same loop as the v2 tertile sweep (group_col = the trait's *_cat column, both sexes pooled, High/Low labels
# defined inside each tertile with label_cutoffs="group"). Proxy level 1 is fit on all non-test donors of all
# tertiles (proxy_train="all_non_test"); XGBoost, labels and AUC stay within the tertile.
#
# PROGRESSIVE SAVE + RESUME (same trick as the v2 tertile sweep): after every (cell_type, trait, tertile) run,
# the summary CSV + Excel workbook are rewritten and that run's per-split AUCs and feature hits are written to
# OUTDIR_TERTILE_V12/per_combo/. If the kernel crashes, just re-run this cell: combos already in the summary CSV
# (with the same splits / feature_mode / proxy_train / label_cutoffs) are loaded from disk and skipped.
# Set RESUME_TERTILE_V12 = False to start from scratch. Crashed combos go to crashes_by_tertile.csv and are
# retried on the next run.
#
# Excel workbook (rebuilt after every combo): sheets "summary", "splits" (one row per split per combo),
# "features" (per-combo feature hits across splits), "crashes", "settings".
_missing_cat_v12 = [t for t in TRAITS_FOR_TERTILE_V12 if f"{t}_cat" not in df_v12_donors.columns]
assert len(_missing_cat_v12) < len(TRAITS_FOR_TERTILE_V12), (
    "No tertile category columns in df_v12_donors -- run V12 STEP 4 first."
)
if _missing_cat_v12:
    print(f"⚠️ No _cat column for {_missing_cat_v12} -- these traits are skipped in the sweep.")

CELL_TYPES_FOR_TERTILE_V12 = ["CD4 Active", "Naive T cells", "CD8 Active", "B Naive", "B Memory",
                              "NK cells", "Classical Monocytes"]
N_OUTER_SPLITS_TERTILE_V12 = 20
TERTILE_LEVELS = ["Low", "Medium", "High"]
FEATURE_MODE_TERTILE_V12 = "proxy"               # "proxy" | "genes" | "proxy+genes"
PROXY_TRAIN_TERTILE_V12 = "all_non_test"
LABEL_CUTOFFS_TERTILE_V12 = "group"
RESUME_TERTILE_V12 = True                        # <<< False = ignore previous results and redo everything

OUTDIR_TERTILE_V12 = f"MULTISPLIT_{_BIOMARKER}_by_tertile_v12_{FEATURE_MODE_TERTILE_V12.replace('+', '_')}"
COMBO_DIR_TERTILE_V12 = f"{OUTDIR_TERTILE_V12}/per_combo"
SUMMARY_TERTILE_V12_CSV = f"{OUTDIR_TERTILE_V12}/summary_all_celltypes_by_tertile_v12.csv"
SUMMARY_TERTILE_V12_XLSX = f"{OUTDIR_TERTILE_V12}/RESULTATS_TERTILE_V12_{_BIOMARKER}.xlsx"
CRASH_TERTILE_V12_CSV = f"{OUTDIR_TERTILE_V12}/crashes_by_tertile.csv"
os.makedirs(COMBO_DIR_TERTILE_V12, exist_ok=True)

_RUN_CONFIG_V12 = {"n_splits_requested": N_OUTER_SPLITS_TERTILE_V12, "feature_mode": FEATURE_MODE_TERTILE_V12,
                   "proxy_train": PROXY_TRAIN_TERTILE_V12, "label_cutoffs": LABEL_CUTOFFS_TERTILE_V12}

def _combo_tag_v12(cell_type, trait, level):
    # settings are part of the file name so a run with other settings never overwrites these files
    cfg = f"n{N_OUTER_SPLITS_TERTILE_V12}_{PROXY_TRAIN_TERTILE_V12}_{LABEL_CUTOFFS_TERTILE_V12}"
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{trait}__{level}__{cell_type}__{cfg}")

def _write_csv_atomic_v12(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False, sep=';', decimal=',')
    os.replace(tmp, path)   # never leaves a half-written file if the kernel dies mid-write

def _write_excel_v12(summary_rows, results, crash_rows, path):
    """Rebuild the whole workbook from what is in memory; written to a temp file, then swapped in."""
    splits, feats = [], []
    for (ct, trait, lv), r in results.items():
        if r.get("df_splits") is not None:
            splits.append(r["df_splits"].assign(cell_type=ct, trait=trait, tertile=lv))
        g = r.get("genes_across_splits")
        if g is not None and len(g):
            g = g.reset_index()
            g = g.rename(columns={g.columns[0]: "Gene"})
            feats.append(g.assign(cell_type=ct, trait=trait, tertile=lv))
    first = ["cell_type", "trait", "tertile"]
    df_sum = pd.DataFrame(summary_rows)
    if len(df_sum):
        df_sum = df_sum.sort_values("test_auc_mean", ascending=False)
    df_spl = pd.concat(splits, ignore_index=True) if splits else pd.DataFrame(columns=first)
    df_fea = pd.concat(feats, ignore_index=True) if feats else pd.DataFrame(columns=first)
    df_spl = df_spl[first + [c for c in df_spl.columns if c not in first]]
    df_fea = df_fea[first + [c for c in df_fea.columns if c not in first]]
    settings = pd.DataFrame({"setting": ["biomarker", *_RUN_CONFIG_V12, "cell_types", "traits"],
                             "value": [_BIOMARKER, *map(str, _RUN_CONFIG_V12.values()),
                                       ", ".join(CELL_TYPES_FOR_TERTILE_V12), ", ".join(TRAITS_FOR_TERTILE_V12)]})
    tmp = path[:-5] + ".tmp.xlsx"   # keep the .xlsx suffix so pandas picks the openpyxl engine
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        df_sum.to_excel(xw, sheet_name="summary", index=False)
        df_spl.to_excel(xw, sheet_name="splits", index=False)
        df_fea.to_excel(xw, sheet_name="features", index=False)
        pd.DataFrame(crash_rows, columns=["cell_type", "trait", "tertile", "error"]).to_excel(
            xw, sheet_name="crashes", index=False)
        settings.to_excel(xw, sheet_name="settings", index=False)
    os.replace(tmp, path)

# --- resume: load what previous runs already saved (only runs made with the same settings) ---
rows_tertile_v12 = []
_other_rows_v12 = pd.DataFrame()   # rows saved with OTHER settings: kept in the CSV, not resumed, not in Excel
results_tertile_v12 = {}   # (cell_type, trait, level) -> {"df_splits", "summary", "genes_across_splits"}
done_tertile_v12 = set()
if RESUME_TERTILE_V12 and os.path.exists(SUMMARY_TERTILE_V12_CSV):
    _prev = pd.read_csv(SUMMARY_TERTILE_V12_CSV, sep=';', decimal=',')
    _same = pd.Series(True, index=_prev.index)
    for _k, _v in _RUN_CONFIG_V12.items():
        _same &= (_prev[_k].astype(str) == str(_v)) if _k in _prev.columns else False
    _other_rows_v12, _prev = _prev[~_same], _prev[_same]
    rows_tertile_v12 = _prev.to_dict("records")
    for r in rows_tertile_v12:
        key = (r["cell_type"], r["trait"], r["tertile"])
        done_tertile_v12.add(key)
        tag = _combo_tag_v12(*key)
        _p_splits = f"{COMBO_DIR_TERTILE_V12}/{tag}__splits.csv"
        _p_genes = f"{COMBO_DIR_TERTILE_V12}/{tag}__genes.csv"
        results_tertile_v12[key] = {
            "summary": r,
            "df_splits": pd.read_csv(_p_splits, sep=';', decimal=',') if os.path.exists(_p_splits) else None,
            "genes_across_splits": (pd.read_csv(_p_genes, sep=';', decimal=',', index_col=0)
                                    if os.path.exists(_p_genes) else None),
        }
    print(f"↩️ Resuming: {len(done_tertile_v12)} combos already done (loaded from {SUMMARY_TERTILE_V12_CSV})")

crash_rows_tertile_v12 = []
_todo_v12 = [(t, ct, lv) for t in TRAITS_FOR_TERTILE_V12 if f"{t}_cat" in df_v12_donors.columns
             for ct in CELL_TYPES_FOR_TERTILE_V12 for lv in TERTILE_LEVELS
             if (ct, t, lv) not in done_tertile_v12]
print(f"▶️ {len(_todo_v12)} combos to run")

for i_combo, (trait, cell_type, level) in enumerate(_todo_v12, 1):
    cat_col = f"{trait}_cat"
    print(f"\n===== [{i_combo}/{len(_todo_v12)}] {cell_type} | {trait} = {level} =====")
    try:
        res = run_discovery_multi_split_proxy(
            df_v12_donors, n_outer_splits=N_OUTER_SPLITS_TERTILE_V12,
            biomarker=_BIOMARKER, cell_type=cell_type, feature_mode=FEATURE_MODE_TERTILE_V12,
            proxy_train=PROXY_TRAIN_TERTILE_V12,
            n_iterations=globals().get("N_ITER_PDF", 50), target_group=level, group_col=cat_col,
            label_cutoffs=LABEL_CUTOFFS_TERTILE_V12,
        )
    except Exception as e:
        print(f"⚠️ CRASH on {cell_type} ({trait}={level}): {e}")
        crash_rows_tertile_v12.append({"cell_type": cell_type, "trait": trait, "tertile": level,
                                       "error": repr(e)})
        _write_csv_atomic_v12(pd.DataFrame(crash_rows_tertile_v12), CRASH_TERTILE_V12_CSV)
        continue

    key = (cell_type, trait, level)
    results_tertile_v12[key] = res
    s = res["summary"]
    rows_tertile_v12.append({
        "cell_type": cell_type, "trait": trait, "tertile": level,
        "n_splits_valid": s["n_splits_valid"], "n_splits_requested": s["n_splits_requested"],
        "n_test_mean": s["n_test_mean"],
        "test_auc_mean": s["test_auc_mean"], "test_auc_std": s["test_auc_std"],
        "inner_val_auc_mean": s["inner_val_auc_mean"], "inner_val_auc_std": s["inner_val_auc_std"],
        "train_auc_mean": s["train_auc_mean"],
        "overfit_gap_mean": s["overfit_gap_mean"], "overfit_gap_std": s["overfit_gap_std"],
        "overfit_gap_innerval_mean": s["overfit_gap_innerval_mean"],
        "overfit_gap_innerval_std": s["overfit_gap_innerval_std"],
        "feature_mode": FEATURE_MODE_TERTILE_V12, "proxy_train": PROXY_TRAIN_TERTILE_V12,
        "label_cutoffs": LABEL_CUTOFFS_TERTILE_V12,
    })

    # --- progressive save: this combo's details + the full summary so far (CSV, then Excel) ---
    tag = _combo_tag_v12(*key)
    _write_csv_atomic_v12(res["df_splits"], f"{COMBO_DIR_TERTILE_V12}/{tag}__splits.csv")
    _g = res["genes_across_splits"].reset_index()
    _g = _g.rename(columns={_g.columns[0]: "Gene"})
    _write_csv_atomic_v12(_g, f"{COMBO_DIR_TERTILE_V12}/{tag}__genes.csv")
    _write_csv_atomic_v12(pd.concat([pd.DataFrame(rows_tertile_v12), _other_rows_v12], ignore_index=True)
                          .sort_values("test_auc_mean", ascending=False), SUMMARY_TERTILE_V12_CSV)
    try:
        _write_excel_v12(rows_tertile_v12, results_tertile_v12, crash_rows_tertile_v12, SUMMARY_TERTILE_V12_XLSX)
    except Exception as e:   # the CSVs are the resume source; an Excel problem must not stop the sweep
        print(f"⚠️ Excel write failed ({e!r}); CSVs are saved, the workbook will be rebuilt after the next combo.")
    print(f"💾 saved ({len(rows_tertile_v12)} combos in {SUMMARY_TERTILE_V12_CSV} + .xlsx)")
    gc.collect()

# final Excel rebuild (also covers a run where everything was resumed or only crashes happened)
_write_excel_v12(rows_tertile_v12, results_tertile_v12, crash_rows_tertile_v12, SUMMARY_TERTILE_V12_XLSX)
df_tertile_v12 = pd.DataFrame(rows_tertile_v12)
if len(df_tertile_v12):
    df_tertile_v12 = df_tertile_v12.sort_values("test_auc_mean", ascending=False)
print(f"📗 Excel: {SUMMARY_TERTILE_V12_XLSX}")
if crash_rows_tertile_v12:
    print(f"⚠️ {len(crash_rows_tertile_v12)} combos crashed -- see {CRASH_TERTILE_V12_CSV}; re-run this cell to retry them.")
df_tertile_v12